# S3 — 검증 게이트웨이 골격 종단 간 측정

## 왜 하나

S2-b까지 판정기(student_vark, k=8)는 **오프라인**으로만 쟀다. 즉 완성된 응답을 한꺼번에 문장으로 나눠 채점했다.
미들웨어로 쓰려면 실제 흐름에서 같은 성능이 나오는지 확인해야 한다. 실제 흐름은 LLM이 토큰을 조금씩 흘려보내고,
게이트웨이가 그걸 받아 문장을 끊고, 판정하고, 사용자에게 내보내는 과정이다.
또 사용자가 느끼는 지연은 판정 시간만으로 정해지지 않는다. **언제 내보내느냐(정책)**가 함께 결정한다.

## 무엇을 하나

OpenAI 호환 프록시(hallugate)를 띄우고, 가짜 LLM이 RAGTruth dev 응답 60개를 실제 API와 비슷한 속도로 재생한다
(첫 토큰 300ms, 토큰당 25ms). 각 응답을 세 모드로 게이트웨이에 통과시킨다.

| 모드 | 동작 |
|---|---|
| off | 검증 없이 전달 (기준선) |
| pass | 토큰 즉시 전달, 문장 완성 시 백그라운드 판정 → 결과를 뒤따라 전달 |
| hold | 문장 완성까지 보류 → 판정 → 전달 |

## 사전 정의 기준

- **본문 보존**: 세 모드 모두 받은 본문이 원래 응답과 글자 단위로 100% 같을 것
- **판별력 유지**: 게이트웨이 안에서 나온 문장 AUROC ≥ 0.80. 오프라인 채택 기준은 0.78이었고, 표본이 60개 응답이라 여유를 둔다.
- **판정 속도(GPU)**: 문장당 판정 p95 ≤ 30ms, pass 모드에서 문장 완성부터 판정 전달까지 p95 ≤ 50ms
- **정책 트레이드오프 기준선 (합격/불합격 없음, S4의 출발점)**:
  hold의 추가 지연(첫 글자, 글자당 평균)과 pass의 누출량(환각 문장을 잡기 전까지 이미 나간 글자 수)

> 런타임 **GPU(T4)**. S2-b에서 저장한 `student_vark/`가 필요하다(드라이브 또는 업로드). 전체 약 20~25분.


## 1. 환경 준비

In [ ]:
!pip -q install pysbd tiktoken pytest-asyncio openai anthropic fastapi uvicorn httpx sentence-transformers
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'
!mkdir -p gateway/hallugate gateway/scripts gateway/tests

In [ ]:
# student_vark 위치 — 드라이브에 저장했다면 경로를 맞추고, 아니면 zip을 업로드해 푼 경로
import os, shutil
USE_DRIVE = True
SRC = '/content/drive/MyDrive/selfproject/student_vark'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir('student_vark'):
    shutil.copytree(SRC, 'student_vark')
print('판정기:', os.listdir('student_vark'))

## 2. 게이트웨이 코드 (hallugate)

아래 셀들은 파일을 만들기만 한다. 로컬에서 16개 테스트를 통과한 코드와 같다.

In [ ]:
import os
os.makedirs('gateway/hallugate/static', exist_ok=True)

In [ ]:
%%writefile gateway/hallugate/__init__.py
"""hallugate — RAG 응답을 스트리밍 중에 문장 단위로 검증하는 OpenAI 호환 게이트웨이."""
from .app import create_app
from .config import Settings

__all__ = ['create_app', 'Settings']

In [ ]:
%%writefile gateway/hallugate/config.py
"""게이트웨이 설정. 환경 변수로 덮어쓸 수 있다.

판정기 관련 값(임베더·k·입력 길이·임계값·재판정 간격)은 비워 두면 판정기 폴더의 judge_config.json 값을 쓴다 (SPEC §4, §7).
"""
import os
from dataclasses import dataclass, field


def _env(name, default, cast=str):
    v = os.environ.get(name)
    return default if v is None or v == '' else cast(v)


@dataclass
class Settings:
    # 실제 LLM API (OpenAI 호환). 로컬 테스트 때는 mock_upstream 주소.
    upstream_base_url: str = field(default_factory=lambda: _env('UPSTREAM_BASE_URL', 'https://api.openai.com/v1'))
    upstream_api_key: str = field(default_factory=lambda: _env('UPSTREAM_API_KEY', ''))
    upstream_timeout: float = field(default_factory=lambda: _env('UPSTREAM_TIMEOUT', 120.0, float))

    # 판정기 폴더 (모델 v1.0 = F1 base). judge_config.json이 있으면 아래 None 값들을 거기서 채운다.
    judge_model_dir: str = field(default_factory=lambda: _env('JUDGE_MODEL_DIR', 'judge_base'))
    judge_embedder: str = field(default_factory=lambda: _env('JUDGE_EMBEDDER', None))
    judge_k: int = field(default_factory=lambda: _env('JUDGE_K', None, int))
    judge_max_len: int = field(default_factory=lambda: _env('JUDGE_MAX_LEN', None, int))
    judge_chunk_words: int = field(default_factory=lambda: _env('JUDGE_CHUNK_WORDS', None, int))
    # 판정기 출력(logit)이 이 값 이상이면 환각. 지정하면 설정 파일의 간격별 임계값 대신 이 값 하나를 쓴다.
    judge_threshold: float = field(default_factory=lambda: _env('JUDGE_THRESHOLD', None, float))
    # 문장 앞부분 재판정 간격(토큰). 0이면 문장 끝에서만 판정. None이면 설정 파일 기본값(F1 base는 8).
    judge_stride: int = field(default_factory=lambda: _env('JUDGE_STRIDE', None, int))
    # 재판정 간격을 세는 토큰 단위. 학습과 같은 cl100k_base. 'words'는 테스트용 대체 단위
    stride_tokenizer: str = field(default_factory=lambda: _env('STRIDE_TOKENIZER', 'cl100k_base'))
    judge_device: str = field(default_factory=lambda: _env('JUDGE_DEVICE', ''))
    # 판정 1회 시간 제한(ms). 넘거나 판정기가 오류를 내면 그 요청은 검증을 멈추고 본문을 그대로 전달한다(fail-open).
    judge_timeout_ms: float = field(default_factory=lambda: _env('JUDGE_TIMEOUT_MS', 2000.0, float))

    # 기본 정책 (요청마다 verification 필드로 바꿀 수 있음). SSOT §7 10/28 A6: hold + 8토큰 + flag
    #   pass : 토큰을 즉시 내보내고 판정 결과를 뒤따라 보냄 (지연 0, 틀린 문장이 먼저 나갈 수 있음)
    #   hold : 판정이 끝날 때까지 붙잡았다가 내보냄 (틀린 문장 차단 가능, 재판정 간격만큼 지연)
    #   off  : 검증 없이 그대로 전달
    default_mode: str = field(default_factory=lambda: _env('VERIFY_MODE', 'hold'))
    #   flag : 환각 판정 문장에 표시만 함
    #   cut  : 환각 판정 시 응답을 끊음 (finish_reason=content_filter)
    default_action: str = field(default_factory=lambda: _env('VERIFY_ACTION', 'flag'))

    # 컬렉션(RAG, SPEC §2.5·2.6). COLLECTION_DB를 빈 값으로 두면 컬렉션 기능을 끈다
    collection_db: str = field(default_factory=lambda: os.environ.get('COLLECTION_DB', 'collections.sqlite3'))
    # 검색 임베더. 비우면 판정기(retrieval 방식)의 임베더를 같이 쓰고, 판정기에 임베더가 없으면 all-MiniLM-L6-v2
    collection_embedder: str = field(default_factory=lambda: _env('COLLECTION_EMBEDDER', None))
    collection_top_k: int = field(default_factory=lambda: _env('COLLECTION_TOP_K', 4, int))
    collection_chunk_words: int = field(default_factory=lambda: _env('COLLECTION_CHUNK_WORDS', 120, int))
    collection_overlap: int = field(default_factory=lambda: _env('COLLECTION_OVERLAP', 20, int))
    # 문서 등록 API 접근 토큰 (Authorization: Bearer). 비우면 토큰 없이 열린다(로컬 개발용)
    admin_token: str = field(default_factory=lambda: _env('ADMIN_TOKEN', ''))

    # 응답 끝 2단 검증 (SPEC §5.3, 🧪 실험 기능). 기본 꺼짐. 검증기·임계값은 J2 결과로 바꿀 임시값
    post_check_default: bool = field(default_factory=lambda: _env('POST_CHECK', False, lambda v: v.lower() in ('1', 'true', 'yes', 'on')))
    post_check_model: str = field(default_factory=lambda: _env('POST_CHECK_MODEL', 'claude-sonnet-5-5'))
    post_check_votes: int = field(default_factory=lambda: _env('POST_CHECK_VOTES', 3, int))
    post_check_threshold: float = field(default_factory=lambda: _env('POST_CHECK_THRESHOLD', 0.5, float))
    post_check_timeout_ms: float = field(default_factory=lambda: _env('POST_CHECK_TIMEOUT_MS', 30000.0, float))
    anthropic_api_key: str = field(default_factory=lambda: _env('ANTHROPIC_API_KEY', ''))

    # 운영 모드 (관리형 베타, SPEC §10). 1이면 x-hallugate-key 인증·사용자별 컬렉션·한도 적용
    hosted: bool = field(default_factory=lambda: _env('HOSTED', False, lambda v: v.lower() in ('1', 'true', 'yes', 'on')))
    keys_db: str = field(default_factory=lambda: _env('KEYS_DB', '/data/hallugate.sqlite3'))
    max_docs: int = field(default_factory=lambda: _env('MAX_DOCS', 200, int))
    # 업스트림 허용 목록 (SPEC §10.3): x-upstream 값=주소, 쉼표로 구분. 운영 모드는 이 주소만 호출한다
    upstream_allowlist: str = field(default_factory=lambda: _env(
        'UPSTREAM_ALLOWLIST', 'openai=https://api.openai.com/v1,anthropic=https://api.anthropic.com/v1'))
    # 키별 기본 한도 (SPEC §10.4). 키 생성 때 limits로 바꿀 수 있다
    rate_rpm: int = field(default_factory=lambda: _env('RATE_RPM', 30, int))
    max_concurrent: int = field(default_factory=lambda: _env('MAX_CONCURRENT', 4, int))
    post_check_daily: int = field(default_factory=lambda: _env('POST_CHECK_DAILY', 50, int))
    max_ref_chars: int = field(default_factory=lambda: _env('MAX_REF_CHARS', 50000, int))
    metrics_retention_days: int = field(default_factory=lambda: _env('METRICS_RETENTION_DAYS', 30, int))
    max_doc_chars: int = field(default_factory=lambda: _env('MAX_DOC_CHARS', 200000, int))

    # 근거 다이어트 기본 조각 수 (SPEC §2.9). T1c 채택값 8 (top_k 4는 T1b에서 충실성 미달)
    trim_top_k: int = field(default_factory=lambda: _env('TRIM_TOP_K', 8, int))
    # 데모 화면 GET /demo (SPEC §2.8). 0이면 끈다
    demo: bool = field(default_factory=lambda: _env('DEMO', True, lambda v: v.lower() not in ('0', 'false', 'no', 'off')))

    # 빈 값이면 기록하지 않는다
    metrics_path: str = field(default_factory=lambda: os.environ.get('METRICS_PATH', 'gateway_metrics.jsonl'))

In [ ]:
%%writefile gateway/hallugate/context.py
"""요청에서 근거 문서(RAG 컨텍스트)와 검증 옵션을 꺼낸다.

우선순위
1. 최상위 필드 `verification.context` (문자열 또는 문자열 리스트)
   - OpenAI SDK에서는 `extra_body={"verification": {...}}` 로 보낸다.
2. 메시지 본문 안의 `<context>...</context>` 태그 (RAG 서비스가 흔히 쓰는 형식)
3. `verification.collection` — 게이트웨이에 등록된 컬렉션에서 검색 (app이 처리, SPEC §2.6)
4. 모두 없으면 검증하지 않는다 (mode=off).

`verification` 필드는 실제 LLM API가 모르는 필드이므로 전달 전에 제거한다.
"""
import copy
import re

CTX_TAG = re.compile(r'<context>(.*?)</context>', re.S | re.I)
MODES = {'pass', 'hold', 'off'}
ACTIONS = {'flag', 'cut'}


def _content_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):          # [{"type":"text","text":...}, ...]
        return '\n'.join(p.get('text', '') for p in content if isinstance(p, dict))
    return ''


RAG_INSTRUCTION = ('Answer using only the reference documents below. '
                   'If they do not contain the answer, say that you do not know.')


def last_user_text(body):
    users = [m for m in body.get('messages', []) if m.get('role') == 'user']
    return _content_text(users[-1].get('content')) if users else ''


def rag_messages(messages, evidence):
    """근거 조각을 담은 시스템 메시지를 맨 앞에 더한다. 서비스가 보낸 메시지는 바꾸지 않는다 (SPEC §2.6)."""
    block = '\n\n'.join(f'[{i + 1}] {e.text}' for i, e in enumerate(evidence))
    return [{'role': 'system', 'content': f'{RAG_INSTRUCTION}\n\n<documents>\n{block}\n</documents>'}] + list(messages)


def strip_context_tags(text):
    """질문 텍스트: <context> 태그 내용을 뺀 나머지 (근거 다이어트 검색어, SPEC §2.9)."""
    return CTX_TAG.sub(' ', text or '').strip()


def replace_context_tags(messages, new_ref):
    """첫 <context> 태그 내용을 new_ref로 바꾸고 나머지 태그는 지운다. 다른 내용은 그대로 (SPEC §2.9)."""
    out, done = [], False
    for m in messages:
        c = m.get('content')
        if isinstance(c, str) and CTX_TAG.search(c):
            def sub(mt):
                nonlocal done
                if done:
                    return ''
                done = True
                return f'<context>{new_ref}</context>'
            m = {**m, 'content': CTX_TAG.sub(sub, c)}
        out.append(m)
    return out


def rag_reference(evidence):
    """판정 근거 = LLM에 넣은 조각과 같은 텍스트."""
    return '\n\n'.join(e.text for e in evidence)


def extract(body, default_mode='pass', default_action='flag', default_stride=0, with_stride=False, with_opts=False):
    """반환: (upstream으로 보낼 body, reference 문자열 또는 None, mode, action[, stride][, opts])
    with_stride=True면 앞부분 재판정 간격(stride)도 함께 돌려준다.
    with_opts=True면 원래 verification 옵션도 돌려준다(collection 등). collection이 있고 근거가 없으면 mode를 off로 바꾸지 않는다."""
    body = copy.deepcopy(body)
    opts = body.pop('verification', None) or {}

    ref = opts.get('context')
    if isinstance(ref, list):
        ref = '\n\n'.join(str(x) for x in ref)
    if not ref:
        found = []
        for m in body.get('messages', []):
            found += CTX_TAG.findall(_content_text(m.get('content')))
        ref = '\n\n'.join(s.strip() for s in found) or None

    mode = opts.get('mode', default_mode)
    action = opts.get('action', default_action)
    if mode not in MODES:
        raise ValueError(f'verification.mode must be one of {sorted(MODES)}')
    if action not in ACTIONS:
        raise ValueError(f'verification.action must be one of {sorted(ACTIONS)}')
    stride = opts.get('stride', default_stride)
    if not isinstance(stride, int) or isinstance(stride, bool) or stride < 0:
        raise ValueError('verification.stride must be a non-negative integer')
    coll = opts.get('collection')
    if coll is not None and (not isinstance(coll, str) or not coll):
        raise ValueError('verification.collection must be a non-empty string')
    tr = opts.get('trim')
    if tr is not None and not (isinstance(tr, bool) or (isinstance(tr, dict) and set(tr) <= {'top_k'}
                                                         and isinstance(tr.get('top_k'), int) and not isinstance(tr.get('top_k'), bool)
                                                         and 1 <= tr['top_k'] <= 32)):
        raise ValueError('verification.trim must be true/false or {"top_k": 1..32}')
    pc = opts.get('post_check')
    if pc is not None and not isinstance(pc, bool):
        raise ValueError('verification.post_check must be a boolean')
    if not ref and not coll:
        mode = 'off'
    out = (body, ref, mode, action) + ((stride,) if with_stride else ()) + ((opts,) if with_opts else ())
    return out

In [ ]:
%%writefile gateway/hallugate/splitter.py
"""스트리밍 텍스트를 문장 단위로 끊는다.

토큰이 조금씩 들어오므로, 버퍼에 쌓다가 '다음 문장이 시작된 것이 확인되면'
앞 문장을 완성된 것으로 보고 내보낸다. 마지막 문장은 스트림이 끝날 때 flush로 내보낸다.

내보낸 문장들을 이어 붙이면 원문과 정확히 같다(공백 포함). hold 모드에서 문장을
그대로 사용자에게 내보내야 하므로 이 성질이 중요하다.
"""
from dataclasses import dataclass

import pysbd


@dataclass
class Sentence:
    idx: int
    text: str        # 원문 그대로 (뒤 공백 포함)
    start: int       # 응답 전체에서의 시작 위치 (문자)

    @property
    def clean(self):
        return self.text.strip()


class IncrementalSplitter:
    def __init__(self, language='en'):
        self._seg = pysbd.Segmenter(language=language, clean=False, char_span=True)
        self._buf = ''
        self._offset = 0      # 버퍼 시작이 응답 전체에서 몇 번째 문자인지
        self._n = 0

    def _emit(self, text):
        s = Sentence(self._n, text, self._offset)
        self._n += 1
        self._offset += len(text)
        return s

    def feed(self, delta):
        """새 조각을 넣고, 완성된 문장 리스트를 돌려준다."""
        self._buf += delta
        if not self._buf.strip():
            return []
        starts = [sp.start for sp in self._seg.segment(self._buf) if sp.sent.strip()]
        if len(starts) < 2:
            return []
        out = []
        cuts = starts[1:]                 # 각 문장의 끝 = 다음 문장의 시작
        prev = 0
        for c in cuts:
            out.append(self._emit(self._buf[prev:c]))
            prev = c
        self._buf = self._buf[prev:]
        return out

    def pending(self):
        """아직 끝나지 않은 현재 문장: (응답 전체에서의 시작 위치, 지금까지의 텍스트)."""
        return self._offset, self._buf

    def flush(self):
        """스트림 종료 시 남은 텍스트를 마지막 문장으로 내보낸다 (공백뿐이면 판정은 건너뛰게 됨)."""
        if not self._buf:
            return []
        s = self._emit(self._buf)
        self._buf = ''
        return [s]

In [ ]:
%%writefile gateway/hallugate/judge.py
"""문장 판정기.

입력 방식은 판정기 폴더의 `judge_config.json`이 정한다 (SPEC §4).
  retrieval (F1 base): 근거 → 40단어 조각 → 임베딩 (요청당 1회, prepare)
                       판정 텍스트 → 임베딩 → 유사한 조각 top-k (원문 순서) → 교차 인코더 [근거 조각, 문장 앞부분]
  full      (I2식)    : 교차 인코더 [근거 전체, 응답 처음부터 판정 지점까지]. 근거 검색 없음
점수는 logit(클수록 환각). 임계값은 재판정 간격(stride)별로 설정 파일의 값을 쓴다.
설정 파일이 "type": "combo"이면 하위 판정기 여러 개의 표준화 점수를 평균하는 ComboJudge를 만든다 (배포 판정기 I3).
"""
import contextlib
import json
import os
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import torch


@dataclass
class Verdict:
    score: float          # logit, 클수록 환각
    hallucinated: bool
    judge_ms: float       # 검색 + 판정 시간


@dataclass
class RefIndex:
    chunks: list
    emb: object           # torch.Tensor [n_chunks, d] (retrieval) / None (full)
    prepare_ms: float
    text: str = ''        # 근거 원문 (full 방식)


def chunk_text(text, n):
    w = text.split()
    return [' '.join(w[i:i + n]) for i in range(0, max(len(w), 1), n)] or [text]


def _stride_key(stride):
    return 'end' if not stride else str(stride)


def read_config(model_dir):
    p = os.path.join(model_dir, 'judge_config.json')
    return json.load(open(p)) if os.path.isfile(p) else {}


def input_mode(cfg):
    """설정 파일에서 입력 방식을 정한다. 명시(`input_mode`)가 없으면 I2 노트북 형식(retrieval: null)을 full로 본다."""
    if cfg.get('input_mode'):
        return cfg['input_mode']
    inp = cfg.get('input') or {}
    return 'full' if isinstance(inp, dict) and 'retrieval' in inp and inp['retrieval'] is None else 'retrieval'


class StudentJudge:
    def __init__(self, model_dir, embedder='sentence-transformers/all-MiniLM-L6-v2', k=8, max_len=512,
                 chunk_words=40, threshold=0.0, device='', mode='retrieval', thresholds=None, threshold_key='svc10_orig',
                 default_stride=0):
        from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.mode = mode
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        cfg = AutoConfig.from_pretrained(model_dir)
        kw = {}
        if getattr(cfg, 'model_type', '') == 'modernbert':
            # 새 transformers는 reference_compile을 모델 인자로 받지 않으므로 설정에 넣는다
            if hasattr(cfg, 'reference_compile'):
                cfg.reference_compile = False
            kw['attn_implementation'] = 'sdpa'
        # 가중치가 fp16으로 저장돼 있어도 fp32로 올린다 (CPU 판정, 노트북 점수와 일치). GPU에서는 autocast가 반정밀도로 돌린다.
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir, config=cfg, **kw).float().to(self.device).eval()
        if mode == 'retrieval' and isinstance(embedder, str):
            from sentence_transformers import SentenceTransformer
            embedder = SentenceTransformer(embedder, device=self.device)
        self.embedder = embedder
        self.k, self.max_len, self.chunk_words, self.threshold = k, max_len, chunk_words, threshold
        self.thresholds, self.threshold_key = thresholds or {}, threshold_key
        self.default_stride = default_stride
        self._lock = threading.Lock()     # 모델 1개를 여러 요청이 공유하므로 직렬화

    def threshold_for(self, stride, key=None):
        """재판정 간격별 임계값 (설정 파일 thresholds[간격][key]). 없으면 기본 임계값."""
        return (self.thresholds.get(_stride_key(stride)) or {}).get(key or self.threshold_key, self.threshold)

    def _embed(self, texts):
        return self.embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True,
                                    show_progress_bar=False)

    def prepare(self, reference):
        t0 = time.perf_counter()
        if self.mode == 'full':
            return RefIndex([], None, (time.perf_counter() - t0) * 1000, reference)
        chunks = chunk_text(reference, self.chunk_words)
        with self._lock:
            emb = self._embed(chunks)
        return RefIndex(chunks, emb, (time.perf_counter() - t0) * 1000, reference)

    def _pair(self, ref, text, prior):
        if self.mode == 'full':
            return ref.text, (prior or '') + text
        e = self._embed([text])[0]
        top = torch.topk(ref.emb @ e, min(self.k, ref.emb.shape[0])).indices.tolist()
        return ' ... '.join(ref.chunks[i] for i in sorted(top)), text

    @torch.no_grad()
    def score(self, ref, text, prior='', stream=None):
        """logit 하나. stream(CUDA 스트림)을 주면 그 스트림에서 계산하고 그 스트림만 기다린다(조합 판정기의 동시 실행용)."""
        ctx = torch.cuda.stream(stream) if (stream is not None and self.device == 'cuda') else contextlib.nullcontext()
        with self._lock, ctx:
            a, b = self._pair(ref, text, prior)
            enc = self.tok([a], [b], truncation='only_first', max_length=self.max_len,
                           return_tensors='pt').to(self.device)
            with torch.autocast(self.device, enabled=(self.device == 'cuda')):
                score = float(self.model(**enc).logits.squeeze().float().cpu())   # .cpu()가 현재 스트림을 기다림
            if self.device == 'cuda' and stream is None:
                torch.cuda.synchronize()
        return score

    def judge(self, ref, text, prior='', threshold=None):
        """text: 판정할 문장(또는 문장 앞부분, 앞뒤 공백 제거), prior: 응답에서 그 문장 앞까지의 원문."""
        t0 = time.perf_counter()
        score = self.score(ref, text, prior)
        thr = self.threshold if threshold is None else threshold
        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)


@dataclass
class ComboRef:
    refs: list            # 하위 판정기마다 RefIndex (members 순서)
    prepare_ms: float


class ComboJudge:
    """하위 판정기 점수를 (score − mean) / std로 표준화해 평균한다 (SPEC §4, 노트북 I3·L3의 combine과 같은 식).
    판정 1회에서 하위 판정기를 동시에 돌린다(스레드, GPU면 판정기마다 별도 CUDA 스트림)."""

    mode = 'combo'

    def __init__(self, members, thresholds=None, threshold=0.0, threshold_key='svcB', default_stride=0):
        # members: [(이름, 판정기, mean, std), ...]
        assert members and all(sd > 0 for _, _, _, sd in members), '하위 판정기와 양수 표준편차가 필요'
        self.members = members
        self.thresholds, self.threshold, self.threshold_key = thresholds or {}, threshold, threshold_key
        self.default_stride = default_stride
        self._pool = ThreadPoolExecutor(len(members), thread_name_prefix='combo')
        self._streams = [torch.cuda.Stream() if getattr(j, 'device', '') == 'cuda' else None for _, j, _, _ in members]

    @property
    def embedder(self):
        # 컬렉션 검색 임베더를 비웠을 때 같이 쓸 임베더: retrieval 방식 하위 판정기의 것
        return next((e for _, j, _, _ in self.members for e in [getattr(j, 'embedder', None)] if hasattr(e, 'encode')), None)

    def threshold_for(self, stride, key=None):
        return (self.thresholds.get(_stride_key(stride)) or {}).get(key or self.threshold_key, self.threshold)

    def prepare(self, reference):
        t0 = time.perf_counter()
        refs = [j.prepare(reference) for _, j, _, _ in self.members]
        return ComboRef(refs, (time.perf_counter() - t0) * 1000)

    @staticmethod
    def _member_score(j, ref, text, prior, stream):
        if hasattr(j, 'score'):
            return j.score(ref, text, prior, stream=stream)
        return j.judge(ref, text, prior).score                  # 대체 판정기(테스트)

    def judge(self, ref, text, prior='', threshold=None):
        t0 = time.perf_counter()
        futs = [self._pool.submit(self._member_score, j, r, text, prior, st)
                for (_, j, _, _), r, st in zip(self.members, ref.refs, self._streams)]
        raw = [f.result() for f in futs]                         # 하위 판정기 예외는 그대로 올라가 fail-open으로 처리됨
        score = sum((x - m) / sd for x, (_, _, m, sd) in zip(raw, self.members)) / len(self.members)
        thr = self.threshold if threshold is None else threshold
        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)


def _load_combo(settings, cfg):
    base = settings.judge_model_dir
    std, members = cfg['standardize'], []
    for name, sub in cfg['members'].items():
        d = sub if os.path.isabs(sub) else os.path.join(base, sub)
        if not os.path.isfile(os.path.join(d, 'config.json')):
            raise FileNotFoundError(f'조합 하위 판정기 {name} 폴더가 없음: {d}')
        mcfg = read_config(d)
        j = StudentJudge(d, embedder=mcfg.get('embedder', 'sentence-transformers/all-MiniLM-L6-v2'), k=mcfg.get('k', 8),
                         max_len=mcfg.get('max_len', 512), chunk_words=mcfg.get('chunk_words', 40),
                         device=settings.judge_device, mode=input_mode(mcfg))
        members.append((name, j, float(std[name]['mean']), float(std[name]['std'])))
    dflt = cfg.get('default') or {}
    kind = dflt.get('kind', 'end')
    fixed = settings.judge_threshold
    return ComboJudge(members, thresholds={} if fixed is not None else cfg.get('thresholds', {}),
                      threshold=fixed if fixed is not None else cfg.get('threshold', 0.0),
                      threshold_key=dflt.get('threshold', 'svcB'), default_stride=0 if kind == 'end' else int(kind))


def load_judge(settings):
    """판정기 폴더의 judge_config.json을 읽어 StudentJudge(또는 "type": "combo"면 ComboJudge)를 만든다.
    환경 변수로 지정한 값이 설정 파일보다 우선한다."""
    cfg = read_config(settings.judge_model_dir)
    if cfg.get('type') == 'combo':
        return _load_combo(settings, cfg)
    pick = lambda env, key, default: env if env is not None else cfg.get(key, default)
    dflt = (cfg.get('default') or {})
    kind = dflt.get('kind', 'end')
    return StudentJudge(settings.judge_model_dir,
                        embedder=pick(settings.judge_embedder, 'embedder', 'sentence-transformers/all-MiniLM-L6-v2'),
                        k=pick(settings.judge_k, 'k', 8), max_len=pick(settings.judge_max_len, 'max_len', 512),
                        chunk_words=pick(settings.judge_chunk_words, 'chunk_words', 40),
                        threshold=pick(settings.judge_threshold, 'threshold', 0.0), device=settings.judge_device,
                        mode=input_mode(cfg),
                        # 임계값을 환경 변수로 고정하면 간격별 임계값은 쓰지 않는다
                        thresholds={} if settings.judge_threshold is not None else cfg.get('thresholds', {}),
                        threshold_key=dflt.get('threshold', 'svc10_orig'),
                        default_stride=0 if kind == 'end' else int(kind))


class LexicalJudge:
    """모델 없이 파이프라인을 시험하기 위한 대체 판정기 (테스트·데모 전용).
    근거에 없는 단어 비율을 점수로 쓴다. 판별력은 없다고 봐야 한다."""

    default_stride = 0

    def __init__(self, threshold=0.5, delay_ms=0.0, fail=None):
        self.threshold, self.delay_ms = threshold, delay_ms
        self.fail = fail          # 테스트용: 이 함수가 True를 돌려주는 판정에서 예외를 낸다

    def threshold_for(self, stride, key=None):
        return self.threshold

    def prepare(self, reference):
        return RefIndex([reference], {w.lower().strip('.,') for w in reference.split()}, 0.0, reference)

    def judge(self, ref, text, prior='', threshold=None):
        t0 = time.perf_counter()
        if self.fail is not None and self.fail(text):
            raise RuntimeError('judge failure (test)')
        if self.delay_ms:
            time.sleep(self.delay_ms / 1000)
        words = [w.lower().strip('.,') for w in text.split() if w.strip('.,')]
        score = sum(w not in ref.emb for w in words) / max(len(words), 1)
        thr = self.threshold if threshold is None else threshold
        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)

In [ ]:
%%writefile gateway/hallugate/pipeline.py
"""스트리밍 검증 파이프라인 (게이트웨이의 핵심).

입력: 실제 LLM이 흘려보내는 텍스트 조각(str) 또는 판정 없이 통과시킬 청크(dict, tool_calls 등)
출력: 사용자에게 보낼 이벤트 — text(본문), verdict(판정 결과), cut(응답 중단), raw(그대로 통과)

정책 (SPEC §5)
  pass : 본문을 받는 즉시 내보내고, 판정은 백그라운드로 실행해 결과를 뒤따라 보낸다.
         추가 지연은 0이지만, 판정이 끝나기 전에 틀린 내용이 사용자에게 이미 나간다(누출).
  hold : 판정이 끝날 때까지 본문을 붙잡았다가 내보낸다. 누출은 0이지만 사용자는 기다려야 한다.
  action=cut 이면 환각 판정 시 그 자리에서 응답을 끊는다.

판정 시점 (SPEC §5.1)
  stride=0 : 문장이 완성되면 그 문장을 판정한다.
  stride=s : 위에 더해, 현재 문장의 토큰(cl100k)이 s개 확정될 때마다 '지금까지의 문장 앞부분'을 판정한다.
             마지막 토큰은 다음 글자가 와야 확정되므로, 토큰 s개 다음의 글자가 도착한 시점에 판정한다.
             pass는 문장 도중에도 경보를 낼 수 있고, hold는 안전 판정된 앞부분까지 먼저 내보낸다.

검증 중단 (SPEC §4, fail-open)
  판정기 오류·시간 초과·영어가 아닌 텍스트 → 판정 청크 {"kind": "unverified", "verified": false, "reason": ...}를
  한 번 보내고, 그 요청의 나머지 본문은 판정 없이 그대로 내보낸다.

측정 (RequestRecord)
  sentences : 완성 문장 판정 — 완성 시각, 판정 시각, 전달 시각, 점수, 판정 시간
  checks    : 앞부분 판정 — 판정 구간, 시각, 점수, 판정 시점까지 사용자에게 나간 글자 수
"""
import asyncio
import time
from dataclasses import asdict, dataclass, field

from .splitter import IncrementalSplitter
from .tokens import WordTokens


@dataclass
class Event:
    kind: str                 # 'text' | 'verdict' | 'cut' | 'raw'
    text: str = ''
    info: dict = None


@dataclass
class RequestRecord:
    mode: str
    action: str
    stride: int = 0
    tag: str = ''                      # 요청 헤더 x-request-tag (측정용 식별자)
    t0: float = field(default_factory=time.perf_counter)
    prepare_ms: float = 0.0
    t_upstream_first: float = None     # 실제 LLM의 첫 토큰 도착 (ms, 요청 기준)
    t_client_first: float = None       # 사용자에게 첫 본문이 나간 시각
    t_upstream_done: float = None
    t_client_done: float = None
    released_chars: int = 0
    cut: bool = False
    verified: bool = True              # 검증을 끝까지 했는지 (fail-open이면 False)
    unverified_reason: str = ''
    threshold: float = None
    collection: str = ''               # RAG 컬렉션 (SPEC §2.6)
    retrieve_ms: float = 0.0
    n_evidence: int = 0
    sentences: list = field(default_factory=list)
    checks: list = field(default_factory=list)
    post_check: dict = None            # 응답 끝 2단 검증 결과 (SPEC §5.3)
    tenant: str = ''                   # 운영 모드 키 ID (SPEC §10)
    upstream: str = ''                 # 운영 모드 업스트림 이름 (SPEC §10.3)
    ref_chars: int = 0                 # 근거 글자 수 (원문 대신 길이만 기록, SPEC §10.5)
    status: int = 200
    stream: bool = True
    cold: bool = False                 # 컨테이너 시작 직후 첫 요청 (SPEC §10.6)
    model: str = ''                    # 업스트림 응답 모델 (SPEC §10.10)
    usage: dict = None                 # 업스트림 토큰 사용량 (SPEC §10.10)
    trim_chars_before: int = 0         # 근거 다이어트 (SPEC §2.9)
    trim_chars_after: int = 0
    trim_chunks_kept: int = 0
    trim_chunks_total: int = 0

    def now(self):
        return (time.perf_counter() - self.t0) * 1000

    def to_dict(self):
        d = asdict(self)
        d.pop('t0')
        return d


def is_english(text, min_letters=3, min_ratio=0.8):
    """글자 중 라틴 문자 비율로 영어 여부를 본다. 판정기는 영어 RAGTruth로만 학습·평가됐다(SSOT §4)."""
    letters = [c for c in text if c.isalpha()]
    if len(letters) < min_letters:
        return True
    return sum(c.isascii() for c in letters) / len(letters) >= min_ratio


class VerifyPipeline:
    def __init__(self, judge, ref_index, mode, action, record, stride=0, tokens=None, threshold=None,
                 timeout_ms=None):
        assert mode in ('pass', 'hold')
        self.judge, self.ref, self.mode, self.action, self.rec = judge, ref_index, mode, action, record
        self.stride = stride or 0
        self.tokens = tokens or WordTokens()
        self.threshold = threshold
        self.timeout = timeout_ms / 1000 if timeout_ms else None
        self.rec.threshold = threshold
        self.splitter = IncrementalSplitter()
        self.text = ''            # 지금까지 받은 전체 본문
        self.released_upto = 0    # hold: 사용자에게 내보낸 위치
        self.cur_sidx = 0         # 앞부분 재판정: 현재 문장 번호
        self.last_mult = 0        #   현재 문장에서 마지막으로 판정한 s의 배수
        self.disabled = None      # 검증 중단 사유
        self.notified = False
        if ref_index is None:     # 근거 준비 실패
            self._disable(record.unverified_reason or 'judge_error')

    # ---- 내부 도우미 ----
    def _release_text(self, text):
        if text:
            if self.rec.t_client_first is None:
                self.rec.t_client_first = self.rec.now()
            self.rec.released_chars += len(text)
        return Event('text', text)

    def _release_to(self, end):
        """hold: 본문을 end 위치까지 내보낸다."""
        piece = self.text[self.released_upto:end]
        self.released_upto = max(self.released_upto, end)
        return self._release_text(piece)

    def _disable(self, reason):
        if self.disabled is None:
            self.disabled = reason
            self.rec.verified, self.rec.unverified_reason = False, reason

    async def _score(self, text, prior):
        """판정 1회. 검증을 멈춰야 하면 None."""
        if self.disabled:
            return None
        if not is_english(text):
            self._disable('non_english')
            return None
        try:
            call = asyncio.to_thread(self.judge.judge, self.ref, text, prior, self.threshold)
            return await (asyncio.wait_for(call, self.timeout) if self.timeout else call)
        except asyncio.TimeoutError:
            self._disable('judge_timeout')
        except Exception:                                # noqa: BLE001 — 판정기 장애는 본문 전달을 막지 않는다
            self._disable('judge_error')
        return None

    async def _judge_sentence(self, s, srec):
        if not s.clean:                                   # 공백뿐인 꼬리
            srec.update(skipped=True, t_verdict=self.rec.now(), hallucinated=False)
            return None
        v = await self._score(s.clean, self.text[:s.start])
        if v is None:
            srec.update(skipped=True, t_verdict=self.rec.now(), hallucinated=False)
            return None
        srec.update(score=v.score, hallucinated=v.hallucinated, judge_ms=v.judge_ms,
                    t_verdict=self.rec.now(),
                    leak_chars=max(0, self.rec.released_chars - s.start) if v.hallucinated else 0)
        return {'kind': 'sentence', 'sentence_idx': s.idx, 'start': s.start, 'upto': s.start + len(s.text),
                'text': s.clean, 'score': v.score, 'hallucinated': v.hallucinated, 'judge_ms': v.judge_ms}

    async def _judge_prefix(self, sidx, start, raw):
        clean = raw.strip()
        if not clean:
            return None
        t_check = self.rec.now()
        v = await self._score(clean, self.text[:start])
        if v is None:
            return None
        info = {'kind': 'prefix', 'sentence_idx': sidx, 'start': start, 'upto': start + len(raw), 'text': clean,
                'score': v.score, 'hallucinated': v.hallucinated, 'judge_ms': v.judge_ms}
        self.rec.checks.append({k: info[k] for k in ('sentence_idx', 'start', 'upto', 'score', 'hallucinated', 'judge_ms')}
                               | {'t_check': t_check, 't_verdict': self.rec.now()})
        return info

    def _verdict(self, info):
        info = dict(info, released_at_verdict=self.rec.released_chars)
        if info['kind'] == 'prefix':
            self.rec.checks[-1]['released_at_verdict'] = info['released_at_verdict']
        return Event('verdict', info=info)

    def _new_srec(self, s):
        srec = {'sentence_idx': s.idx, 'start': s.start, 'n_chars': len(s.text), 't_complete': self.rec.now()}
        self.rec.sentences.append(srec)
        return srec

    def _prefix_point(self, d):
        """이번 조각으로 앞부분 재판정 시점이 됐으면 (문장 시작 위치, 판정할 앞부분 원문), 아니면 None.
        문장 시작부터 확정된 토큰 수가 s의 새 배수에 이르면 그 배수 번째 토큰 끝까지를 판정한다
        (한 조각이 여러 배수를 넘으면 마지막 배수 하나만)."""
        if not self.stride or d is None or self.disabled:
            return None
        off, buf = self.splitter.pending()
        if self.splitter._n != self.cur_sidx:
            self.cur_sidx, self.last_mult = self.splitter._n, 0
        toks = self.tokens.offsets(buf)
        m = (len(toks) - 1) // self.stride               # 마지막 토큰은 아직 이어질 수 있어 확정 전
        if m <= self.last_mult:
            return None
        self.last_mult = m
        return off, buf[:toks[m * self.stride - 1][1]]

    def _cut(self, info):
        self.rec.cut = True
        return Event('cut', info={'sentence_idx': info['sentence_idx'], 'kind': info['kind']})

    def _unverified_notice(self):
        """검증이 멈췄으면 한 번만 알린다."""
        if self.disabled and not self.notified:
            self.notified = True
            return Event('verdict', info={'kind': 'unverified', 'verified': False, 'reason': self.disabled,
                                          'sentence_idx': self.splitter._n, 'released_at_verdict': self.rec.released_chars})
        return None

    # ---- 본체 ----
    async def run(self, deltas):
        q = asyncio.Queue()

        async def producer():
            try:
                async for d in deltas:
                    if self.rec.t_upstream_first is None:
                        self.rec.t_upstream_first = self.rec.now()
                    await q.put(d)
            finally:
                self.rec.t_upstream_done = self.rec.now()
                await q.put(None)

        prod = asyncio.create_task(producer())
        pending = []                                     # pass 모드: 판정 task들 (발생 순서)
        get = None
        try:
            while True:
                if get is None:
                    get = asyncio.create_task(q.get())
                # pass: 새 조각 도착 또는 가장 앞선 판정 완료 중 먼저 일어나는 쪽을 처리
                waits = {get} | ({pending[0]} if pending else set())
                done, _ = await asyncio.wait(waits, return_when=asyncio.FIRST_COMPLETED)

                while pending and pending[0].done():         # 끝난 판정은 순서대로 즉시 전달
                    info = pending.pop(0).result()
                    if info is None:
                        continue
                    yield self._verdict(info)
                    if info['hallucinated'] and self.action == 'cut':
                        yield self._cut(info)
                        return
                if get not in done:
                    if (n := self._unverified_notice()):
                        yield n
                    continue
                d, get = get.result(), None
                if isinstance(d, dict):                       # tool_calls 등: 판정 없이 그대로 통과
                    yield Event('raw', info=d)
                    continue
                if d:
                    self.text += d
                sents = self.splitter.flush() if d is None else self.splitter.feed(d)
                point = self._prefix_point(d)

                if self.mode == 'pass':
                    if d:
                        yield self._release_text(d)
                    for s in sents:
                        srec = self._new_srec(s)
                        srec['t_release'] = srec['t_complete']   # 이미 나갔음
                        pending.append(asyncio.create_task(self._judge_sentence(s, srec)))
                    if point:
                        pending.append(asyncio.create_task(self._judge_prefix(self.splitter._n, *point)))
                    if d is None:                                 # 스트림 종료: 남은 판정을 순서대로 기다림
                        for t in pending:
                            info = await t
                            if info is None:
                                continue
                            yield self._verdict(info)
                            if info['hallucinated'] and self.action == 'cut':
                                yield self._cut(info)
                                return
                        pending = []
                    if (n := self._unverified_notice()):
                        yield n
                else:  # hold
                    for s in sents:
                        srec = self._new_srec(s)
                        info = await self._judge_sentence(s, srec)
                        if info is not None:
                            yield self._verdict(info)
                            if info['hallucinated'] and self.action == 'cut':
                                yield self._cut(info)
                                return
                        srec['t_release'] = self.rec.now()
                        yield self._release_to(s.start + len(s.text))
                    if point:
                        info = await self._judge_prefix(self.splitter._n, *point)
                        if info is not None:
                            yield self._verdict(info)
                            if info['hallucinated'] and self.action == 'cut':
                                yield self._cut(info)
                                return
                            # 안전하거나(flag 모드에서는 표시 후) → 판정한 앞부분까지 내보낸다
                            yield self._release_to(point[0] + len(point[1]))
                    if self.disabled:                             # 검증 중단: 알리고 보류 중인 본문을 모두 내보낸다
                        if (n := self._unverified_notice()):
                            yield n
                        yield self._release_to(len(self.text))

                if d is None:
                    break
        finally:
            self.rec.t_client_done = self.rec.now()
            for t in pending:
                t.cancel()
            if get is not None:
                get.cancel()
            prod.cancel()
            try:
                await prod
            except (asyncio.CancelledError, Exception):
                pass

In [ ]:
%%writefile gateway/hallugate/tokens.py
"""재판정 간격을 세는 토큰 단위 (SPEC §5.1).

판정기는 tiktoken cl100k_base 토큰으로 '문장 시작부터 s토큰마다' 판정 지점을 잡아 학습·보정됐다(SSOT §4).
게이트웨이도 같은 단위로 세야 학습·임계값과 맞는다. 업스트림 조각 수로 세면 Claude(조각당 약 5토큰)에서 어긋난다.

  offsets(text) -> [(start, end), ...]  각 토큰의 글자 구간. 이어 붙이면 text 전체를 덮는다.

'words'는 네트워크 없이 테스트하기 위한 대체 단위다(앞 공백 + 단어 = 토큰 1개, cl100k와 비슷한 모양).
"""
import re


class WordTokens:
    name = 'words'
    _pat = re.compile(r'\s*\S+|\s+')

    def offsets(self, text):
        return [m.span() for m in self._pat.finditer(text)]


class TiktokenTokens:
    def __init__(self, encoding='cl100k_base'):
        import tiktoken
        self.name = encoding
        self.enc = tiktoken.get_encoding(encoding)

    def offsets(self, text):
        ids = self.enc.encode(text, disallowed_special=())
        dec, starts = self.enc.decode_with_offsets(ids)
        if dec != text:                       # 드문 경우: 멀티바이트 문자 경계 — 누적 디코드로 대체 (E1과 같은 처리)
            starts, pos = [], 0
            for i in range(len(ids)):
                starts.append(pos)
                pos = len(self.enc.decode(ids[:i + 1]))
        ends = starts[1:] + [len(text)]
        return [(a, b) for a, b in zip(starts, ends) if b > a]


def make_tokens(name):
    return WordTokens() if name == 'words' else TiktokenTokens(name)

In [ ]:
%%writefile gateway/hallugate/store.py
"""컬렉션 저장소 (SPEC §2.5, §2.6).

관리자·개발자가 등록한 문서를 조각으로 나눠 임베딩과 함께 저장하고, 질문으로 검색한다.
게이트웨이는 찾은 조각을 LLM 프롬프트에 넣고, **같은 조각을 판정 근거로** 쓴다(LLM이 본 근거 = 판정 근거).

  Store 인터페이스
    add(collection, text, title='', doc_id=None) -> dict(doc_id, title, n_chunks, n_chars)
    list(collection) -> [dict(doc_id, title, n_chunks, n_chars, created)]
    delete(collection, doc_id) -> bool
    collections() -> [dict(collection, n_docs)]
    search(collection, query, k) -> [Evidence]       # 점수 내림차순

처음 구현은 SQLite 파일 하나 + 임베딩 배열(코사인 유사도 전수 검색)이다. 규모가 커지면 같은 인터페이스로 벡터 DB로 바꾼다.
임베더는 encode(texts) -> 정규화된 벡터 배열을 돌려주는 객체(sentence-transformers 호환)면 된다.
"""
import sqlite3
import threading
import time
import uuid
from dataclasses import asdict, dataclass

import numpy as np

SCHEMA = '''
CREATE TABLE IF NOT EXISTS documents (
    collection TEXT NOT NULL, doc_id TEXT NOT NULL, title TEXT, text TEXT NOT NULL, created REAL NOT NULL,
    PRIMARY KEY (collection, doc_id));
CREATE TABLE IF NOT EXISTS chunks (
    collection TEXT NOT NULL, doc_id TEXT NOT NULL, idx INTEGER NOT NULL, text TEXT NOT NULL, emb BLOB NOT NULL,
    PRIMARY KEY (collection, doc_id, idx));
'''


@dataclass
class Evidence:
    doc_id: str
    title: str
    chunk_idx: int
    text: str
    score: float

    def to_dict(self):
        return asdict(self)


def split_words(text, n, overlap=0):
    """n단어 조각(겹침 overlap단어). 빈 문서는 조각 없음."""
    w = text.split()
    if not w:
        return []
    step = max(1, n - overlap)
    return [' '.join(w[i:i + n]) for i in range(0, max(len(w) - overlap, 1), step)]


class SQLiteStore:
    def __init__(self, path, embedder, chunk_words=120, overlap=20):
        self.path, self.embedder = path, embedder
        self.chunk_words, self.overlap = chunk_words, overlap
        self._db = sqlite3.connect(path, check_same_thread=False)
        self._db.executescript(SCHEMA)
        self._lock = threading.Lock()
        self._cache = {}          # collection -> (meta 리스트, 임베딩 행렬)

    def _embed(self, texts):
        v = self.embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)
        return np.asarray(v, dtype=np.float32).reshape(len(texts), -1)

    def add(self, collection, text, title='', doc_id=None):
        if not text or not text.strip():
            raise ValueError('document text is empty')
        doc_id = doc_id or uuid.uuid4().hex[:16]
        chunks = split_words(text, self.chunk_words, self.overlap)
        emb = self._embed(chunks)
        with self._lock, self._db:
            self._db.execute('DELETE FROM chunks WHERE collection=? AND doc_id=?', (collection, doc_id))   # 같은 id면 교체
            self._db.execute('INSERT OR REPLACE INTO documents VALUES (?,?,?,?,?)',
                             (collection, doc_id, title or '', text, time.time()))
            self._db.executemany('INSERT INTO chunks VALUES (?,?,?,?,?)',
                                 [(collection, doc_id, i, c, e.tobytes()) for i, (c, e) in enumerate(zip(chunks, emb))])
            self._cache.pop(collection, None)
        return {'doc_id': doc_id, 'title': title or '', 'n_chunks': len(chunks), 'n_chars': len(text)}

    def list(self, collection):
        with self._lock:
            rows = self._db.execute(
                'SELECT d.doc_id, d.title, length(d.text), d.created, '
                '(SELECT count(*) FROM chunks c WHERE c.collection=d.collection AND c.doc_id=d.doc_id) '
                'FROM documents d WHERE d.collection=? ORDER BY d.created', (collection,)).fetchall()
        return [{'doc_id': r[0], 'title': r[1], 'n_chars': r[2], 'created': r[3], 'n_chunks': r[4]} for r in rows]

    def delete(self, collection, doc_id):
        with self._lock, self._db:
            n = self._db.execute('DELETE FROM documents WHERE collection=? AND doc_id=?', (collection, doc_id)).rowcount
            self._db.execute('DELETE FROM chunks WHERE collection=? AND doc_id=?', (collection, doc_id))
            self._cache.pop(collection, None)
        return n > 0

    def collections(self):
        with self._lock:
            rows = self._db.execute('SELECT collection, count(*) FROM documents GROUP BY collection ORDER BY collection').fetchall()
        return [{'collection': c, 'n_docs': n} for c, n in rows]

    def exists(self, collection):
        with self._lock:
            return self._db.execute('SELECT 1 FROM documents WHERE collection=? LIMIT 1', (collection,)).fetchone() is not None

    def _matrix(self, collection):
        with self._lock:
            if collection not in self._cache:
                rows = self._db.execute(
                    'SELECT c.doc_id, d.title, c.idx, c.text, c.emb FROM chunks c JOIN documents d '
                    'ON c.collection=d.collection AND c.doc_id=d.doc_id WHERE c.collection=? ORDER BY d.created, c.idx',
                    (collection,)).fetchall()
                meta = [r[:4] for r in rows]
                mat = np.stack([np.frombuffer(r[4], dtype=np.float32) for r in rows]) if rows else np.zeros((0, 1), np.float32)
                self._cache[collection] = (meta, mat)
            return self._cache[collection]

    def search(self, collection, query, k):
        meta, mat = self._matrix(collection)
        if not meta or not query.strip():
            return []
        sims = mat @ self._embed([query])[0]
        top = np.argsort(-sims)[:k]
        return [Evidence(meta[i][0], meta[i][1], meta[i][2], meta[i][3], float(sims[i])) for i in top]

    def count_docs(self, prefix):
        """이름이 prefix로 시작하는 컬렉션들의 문서 수 (운영 모드 사용자별 한도, SPEC §10.2)."""
        with self._lock:
            return self._db.execute("SELECT count(*) FROM documents WHERE substr(collection, 1, ?)=?",
                                    (len(prefix), prefix)).fetchone()[0]

    def doc_exists(self, collection, doc_id):
        with self._lock:
            return self._db.execute('SELECT 1 FROM documents WHERE collection=? AND doc_id=?',
                                    (collection, doc_id)).fetchone() is not None

    def purge(self, prefix):
        """이름이 prefix로 시작하는 컬렉션을 모두 지운다 (키 폐기 --purge). 지운 문서 수."""
        with self._lock, self._db:
            n = self._db.execute('DELETE FROM documents WHERE substr(collection, 1, ?)=?', (len(prefix), prefix)).rowcount
            self._db.execute('DELETE FROM chunks WHERE substr(collection, 1, ?)=?', (len(prefix), prefix))
            self._cache.clear()
        return n

    def close(self):
        self._db.close()

In [ ]:
%%writefile gateway/hallugate/postcheck.py
"""응답 끝 2단 검증 (SPEC §5.3, 🧪 실험 기능, 기본 꺼짐).

응답이 끝난 뒤 강한 검증기(Claude)가 응답 전체를 근거와 다시 대조한다. 1단(경량 판정기)이 놓친 환각을 잡기 위한 것이다.
지시문·파싱은 J2 노트북의 `sonnet_H` 후보와 같다(experiments/builders/build_j2.py SYS_SENT, 테스트로 일치 확인).
검증기·임계값은 J2가 채택되면 바꾼다. 본문은 바꾸지 않는다. 오류·시간 초과는 verified=False로 알린다(fail-open).
"""
import asyncio
import json
import time
from dataclasses import asdict, dataclass, field

SYSTEM = '''You check whether a RESPONSE is fully supported by a REFERENCE document.
You receive the REFERENCE and the RESPONSE split into numbered sentences. Judge EVERY sentence against the REFERENCE only (not world knowledge).

A sentence contains a hallucination if any part of it is:
- Evident conflict: clearly contradicts the reference (wrong fact, number, name, date, negation).
- Subtle conflict: deviates from the reference in a way that changes meaning (altered term, scope, time, attribution, certainty).
- Evident baseless info: states information that is not in the reference and cannot be directly inferred from it, even if it may be true in the real world.
- Subtle baseless info: adds unsupported inference, opinion, or embellishment that the reference does not support.

Not hallucinations: faithful paraphrase; reasonable summarization or generalization of what the reference says; statements that the reference lacks the information.

Labels: "S" = fully supported. "H" = contains a hallucination. "A" = ambiguous, careful annotators could reasonably disagree.
For every H or A sentence also give "type" (evident_conflict, subtle_conflict, evident_baseless, subtle_baseless), "phrase" (shortest hallucinated span copied exactly) and "reason" (at most 20 words).

Return ONLY a JSON object, no other text:
{"sentences": [{"i": 1, "label": "S"}, {"i": 2, "label": "H", "type": "evident_baseless", "phrase": "...", "reason": "..."}]}
with exactly one entry per numbered sentence, in order.'''


def user_message(reference, sentences):
    lines = '\n'.join(f'[{i + 1}] {t}' for i, t in enumerate(sentences))
    return f"REFERENCE:\n<<<\n{reference}\n>>>\n\nRESPONSE ({len(sentences)} numbered sentences):\n{lines}"


def parse(text, n):
    """R4·J2와 같은 파싱: 문장마다 S/H/A. 모든 문장이 있어야 성공."""
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        labs = {}
        for e in d.get('sentences', []):
            i, lab = int(e.get('i', 0)), str(e.get('label', '')).strip().upper()[:1]
            if 1 <= i <= n and lab and lab in 'SHA':
                labs[i] = {'label': lab, 'phrase': e.get('phrase', '') or '', 'reason': e.get('reason', '') or ''}
        return labs, len(labs) == n
    except Exception:                                        # noqa: BLE001
        return {}, False


@dataclass
class PostCheckResult:
    verified: bool
    reason: str = ''
    model: str = ''
    votes: int = 0
    valid_votes: int = 0
    score: float = 0.0
    threshold: float = 0.0
    flagged: bool = False
    check_ms: float = 0.0
    in_tokens: int = 0
    out_tokens: int = 0
    sentences: list = field(default_factory=list)

    def to_dict(self):
        if not self.verified:                                # 실패·건너뜀은 이유만
            return {'kind': 'post_check', 'verified': False, 'reason': self.reason}
        d = {'kind': 'post_check', **asdict(self)}
        d.pop('reason')
        return d


def skipped(reason):
    return PostCheckResult(False, reason)


class ClaudePostChecker:
    """Claude에 문장별 S/H/A 판정을 votes번 동시에 요청하고 표를 모은다.
    응답 점수 = H 문장이 하나라도 있다고 본 판정의 비율(파싱 실패 판정은 H 없음으로 셈)."""

    def __init__(self, client=None, api_key='', model='claude-sonnet-5-5', votes=3, threshold=0.5, max_tokens=2048):
        if client is None:
            import anthropic
            client = anthropic.AsyncAnthropic(api_key=api_key, max_retries=2)
        self.client, self.model, self.votes, self.threshold, self.max_tokens = client, model, votes, threshold, max_tokens

    def params(self, reference, sentences):
        return {'model': self.model, 'max_tokens': self.max_tokens, 'system': SYSTEM,
                'messages': [{'role': 'user', 'content': user_message(reference, sentences)}]}

    async def _one(self, p):
        m = await self.client.messages.create(**p)
        u = getattr(m, 'usage', None)
        text = ''.join(getattr(b, 'text', '') for b in m.content if getattr(b, 'type', '') == 'text')
        return text, getattr(u, 'input_tokens', 0) or 0, getattr(u, 'output_tokens', 0) or 0

    async def check(self, reference, sents):
        """sents: [(sentence_idx, start, 문장 텍스트)] — 1단과 같은 문장 분할.
        모든 호출이 예외면 그 예외를 올린다(호출한 쪽이 fail-open으로 처리)."""
        t0 = time.perf_counter()
        p = self.params(reference, [t for _, _, t in sents])
        got = await asyncio.gather(*[self._one(p) for _ in range(self.votes)], return_exceptions=True)
        if all(isinstance(o, BaseException) for o in got):
            raise got[0]
        ok_calls = [o for o in got if not isinstance(o, BaseException)]
        tok_in, tok_out = sum(o[1] for o in ok_calls), sum(o[2] for o in ok_calls)   # 비용 추정용 (SPEC §10.6)
        outs = [o if isinstance(o, BaseException) else o[0] for o in got]
        n = len(sents)
        valid = [labs for labs, ok in (parse(o, n) if isinstance(o, str) else ({}, False) for o in outs) if ok]
        if not valid:
            return PostCheckResult(False, 'error')
        score = sum(any(v['label'] == 'H' for v in labs.values()) for labs in valid) / self.votes
        rows = []
        for k, (idx, start, _) in enumerate(sents, 1):
            hv = [labs[k] for labs in valid if labs[k]['label'] == 'H']
            row = {'sentence_idx': idx, 'start': start, 'h_votes': len(hv),
                   'flagged': len(hv) / len(valid) >= self.threshold}
            if hv:
                row.update(phrase=hv[0]['phrase'], reason=hv[0]['reason'])
            rows.append(row)
        return PostCheckResult(True, '', self.model, self.votes, len(valid), score, self.threshold,
                               score >= self.threshold, (time.perf_counter() - t0) * 1000, tok_in, tok_out, rows)

In [ ]:
%%writefile gateway/hallugate/keys.py
"""운영 모드 API 키 저장소 (SPEC §10.1).

키 형식: hg_<키 ID>_<비밀>. 저장소에는 비밀의 SHA-256 해시만 둔다(원문은 발급할 때 한 번만 보여 줌).
키 ID는 사용자(테넌트) 식별자로도 쓴다 — 컬렉션 이름 앞에 붙여 사용자별로 분리한다(SPEC §10.2).
"""
import hashlib
import hmac
import json
import secrets
import sqlite3
import threading
import time
from dataclasses import dataclass, field

SCHEMA = '''
CREATE TABLE IF NOT EXISTS api_keys (
    key_id TEXT PRIMARY KEY, hash TEXT NOT NULL, name TEXT, created REAL NOT NULL, revoked REAL, limits TEXT);
CREATE TABLE IF NOT EXISTS usage_daily (
    key_id TEXT NOT NULL, day TEXT NOT NULL, post_checks INTEGER NOT NULL DEFAULT 0, PRIMARY KEY (key_id, day));
CREATE TABLE IF NOT EXISTS usage_monthly (
    key_id TEXT NOT NULL, month TEXT NOT NULL, tokens INTEGER NOT NULL DEFAULT 0, cost_usd REAL NOT NULL DEFAULT 0,
    PRIMARY KEY (key_id, month));
'''


def _month():
    return time.strftime('%Y-%m', time.gmtime())


def _today():
    return time.strftime('%Y-%m-%d', time.gmtime())


@dataclass
class Tenant:
    key_id: str
    name: str = ''
    limits: dict = field(default_factory=dict)       # 키별 한도 (SPEC §10.4). 비어 있으면 서버 기본값


def _hash(secret):
    return hashlib.sha256(secret.encode()).hexdigest()


def parse_key(full):
    """'hg_<id>_<비밀>' → (id, 비밀). 형식이 아니면 None."""
    if not isinstance(full, str) or not full.startswith('hg_'):
        return None
    parts = full.split('_', 2)
    if len(parts) != 3 or not parts[1] or not parts[2]:
        return None
    return parts[1], parts[2]


class KeyStore:
    def __init__(self, path):
        self.path = path
        self._db = sqlite3.connect(path, check_same_thread=False)
        self._db.executescript(SCHEMA)
        self._lock = threading.Lock()

    def create(self, name='', limits=None):
        """새 키 발급. (키 ID, 전체 키 문자열) — 전체 키는 이때만 알 수 있다."""
        kid, secret = secrets.token_hex(4), secrets.token_urlsafe(24).replace('_', 'x')
        with self._lock, self._db:
            while self._db.execute('SELECT 1 FROM api_keys WHERE key_id=?', (kid,)).fetchone():
                kid = secrets.token_hex(4)
            self._db.execute('INSERT INTO api_keys VALUES (?,?,?,?,?,?)',
                             (kid, _hash(secret), name, time.time(), None, json.dumps(limits or {})))
        return kid, f'hg_{kid}_{secret}'

    def verify(self, full):
        """유효한 키면 Tenant, 아니면 None (없음·형식 오류·비밀 불일치·폐기)."""
        p = parse_key(full)
        if p is None:
            return None
        kid, secret = p
        with self._lock:
            row = self._db.execute('SELECT hash, name, revoked, limits FROM api_keys WHERE key_id=?', (kid,)).fetchone()
        if row is None or row[2] is not None or not hmac.compare_digest(row[0], _hash(secret)):
            return None
        return Tenant(kid, row[1] or '', json.loads(row[3] or '{}'))

    def list(self):
        with self._lock:
            rows = self._db.execute('SELECT key_id, name, created, revoked, limits FROM api_keys ORDER BY created').fetchall()
        return [{'key_id': r[0], 'name': r[1], 'created': r[2], 'revoked': r[3], 'limits': json.loads(r[4] or '{}')}
                for r in rows]

    def revoke(self, key_id):
        with self._lock, self._db:
            n = self._db.execute('UPDATE api_keys SET revoked=? WHERE key_id=? AND revoked IS NULL',
                                 (time.time(), key_id)).rowcount
        return n > 0

    def post_checks_today(self, key_id):
        with self._lock:
            r = self._db.execute('SELECT post_checks FROM usage_daily WHERE key_id=? AND day=?', (key_id, _today())).fetchone()
        return r[0] if r else 0

    def take_post_check(self, key_id, limit):
        """오늘(UTC) 2단 검증 수가 limit 미만이면 하나 늘리고 True (SPEC §10.4)."""
        day = _today()
        with self._lock, self._db:
            r = self._db.execute('SELECT post_checks FROM usage_daily WHERE key_id=? AND day=?', (key_id, day)).fetchone()
            if (r[0] if r else 0) >= limit:
                return False
            self._db.execute('INSERT INTO usage_daily (key_id, day, post_checks) VALUES (?,?,1) '
                             'ON CONFLICT(key_id, day) DO UPDATE SET post_checks = post_checks + 1', (key_id, day))
        return True

    def add_month_usage(self, key_id, tokens, cost):
        """이번 달(UTC) 사용자 LLM 사용량 누적 (예산, SPEC §10.10)."""
        with self._lock, self._db:
            self._db.execute('INSERT INTO usage_monthly VALUES (?,?,?,?) ON CONFLICT(key_id, month) DO UPDATE SET '
                             'tokens = tokens + excluded.tokens, cost_usd = cost_usd + excluded.cost_usd',
                             (key_id, _month(), int(tokens or 0), float(cost or 0.0)))

    def month_usage(self, key_id):
        with self._lock:
            r = self._db.execute('SELECT tokens, cost_usd FROM usage_monthly WHERE key_id=? AND month=?',
                                 (key_id, _month())).fetchone()
        return {'month': _month(), 'tokens': r[0] if r else 0, 'cost_usd': r[1] if r else 0.0}

    def close(self):
        self._db.close()

In [ ]:
%%writefile gateway/hallugate/limits.py
"""운영 모드 사용량 제한 (SPEC §10.4). 컨테이너 1대 전제라 메모리에서 센다.

  hit(key, rpm)        최근 60초 요청 수가 rpm 미만이면 기록하고 None, 아니면 다시 시도까지 남은 초(거절은 세지 않음)
  acquire(key, n)      처리 중인 요청이 n개 미만이면 하나 늘리고 True
  release(key)         처리 끝
"""
import math
import threading
import time
from collections import defaultdict, deque

WINDOW = 60.0


class Limiter:
    def __init__(self, now=time.monotonic):
        self.now = now
        self._hits = defaultdict(deque)
        self._active = defaultdict(int)
        self._lock = threading.Lock()

    def hit(self, key, rpm):
        t = self.now()
        with self._lock:
            q = self._hits[key]
            while q and q[0] <= t - WINDOW:
                q.popleft()
            if len(q) >= rpm:
                return max(1, math.ceil(q[0] + WINDOW - t))
            q.append(t)
            return None

    def acquire(self, key, limit):
        with self._lock:
            if self._active[key] >= limit:
                return False
            self._active[key] += 1
            return True

    def release(self, key):
        with self._lock:
            self._active[key] = max(0, self._active[key] - 1)

In [ ]:
%%writefile gateway/hallugate/metrics.py
"""운영 모드 측정 기록·통계 (SPEC §10.5·10.6).

요청마다 **원문 없는** 요약 한 줄을 SQLite `requests` 표(KEYS_DB와 같은 파일)에 남기고 표준 출력으로도 낸다.
남기는 항목은 허용 목록(summarize)으로만 고른다 — 기록 객체에 새 필드가 생겨도 원문이 새지 않게.
컨테이너 시작(예열 시간)은 `starts` 표에 남겨 콜드 스타트를 본다.
"""
import json
import re
import sqlite3
import threading
import time

import numpy as np

from .prices import cost_usd

SCHEMA = '''
CREATE TABLE IF NOT EXISTS requests (ts REAL NOT NULL, key_id TEXT, status INTEGER, data TEXT NOT NULL);
CREATE INDEX IF NOT EXISTS requests_ts ON requests (ts);
CREATE TABLE IF NOT EXISTS starts (ts REAL NOT NULL, warmup_ms REAL);
'''

# 2단 검증 실시간 단가 추정($/백만 토큰): 배치 단가(SSOT, M1)의 2배
PRICE_RT = {'claude-haiku-5-5': (0.10, 0.50), 'claude-sonnet-5-5': (2.0, 10.0), 'claude-opus-5-5': (4.0, 20.0)}

PC_KEYS = ('verified', 'reason', 'model', 'votes', 'valid_votes', 'score', 'threshold', 'flagged', 'check_ms',
           'in_tokens', 'out_tokens')


def _pct(xs, q):
    xs = [x for x in xs if x is not None]
    return float(np.percentile(xs, q)) if xs else None


def summarize(rec, status=200, stream=True, cold=False):
    """RequestRecord.to_dict() → 원문 없는 요약 (허용 목록)."""
    sents, checks = rec.get('sentences') or [], rec.get('checks') or []
    judged = [x for x in sents if 'score' in x] + list(checks)
    jms = [x.get('judge_ms') for x in judged if x.get('judge_ms') is not None]
    pc = rec.get('post_check')
    return {
        'key_id': rec.get('tenant', ''), 'upstream': rec.get('upstream', ''), 'status': status, 'stream': bool(stream),
        'mode': rec.get('mode'), 'action': rec.get('action'), 'stride': rec.get('stride'),
        'ttft_ms': rec.get('t_client_first'), 'total_ms': rec.get('t_client_done'),
        'upstream_first_ms': rec.get('t_upstream_first'), 'upstream_done_ms': rec.get('t_upstream_done'),
        'prepare_ms': rec.get('prepare_ms'), 'retrieve_ms': rec.get('retrieve_ms'), 'ref_chars': rec.get('ref_chars', 0),
        'released_chars': rec.get('released_chars'), 'n_sentences': len(sents), 'n_judgments': len(judged),
        'n_flagged': sum(1 for x in sents if x.get('hallucinated')), 'judge_ms_max': max(jms) if jms else None,
        'judge_ms_p95': _pct(jms, 95), 'verified': rec.get('verified'), 'unverified_reason': rec.get('unverified_reason') or '',
        'cut': rec.get('cut'), 'collection_used': bool(rec.get('collection')), 'n_evidence': rec.get('n_evidence', 0),
        'cold': bool(cold),
        'trim_chars_before': rec.get('trim_chars_before', 0), 'trim_chars_after': rec.get('trim_chars_after', 0),
        'trim_chunks_kept': rec.get('trim_chunks_kept', 0), 'trim_chunks_total': rec.get('trim_chunks_total', 0),
        'post_check': {k: pc[k] for k in PC_KEYS if k in pc} if isinstance(pc, dict) else None,
        **llm_usage(rec.get('model'), rec.get('usage')),
    }


def llm_usage(model, usage):
    """업스트림 usage → 토큰 수·추정 비용 (SPEC §10.10). 사용량이 없으면 None."""
    u = usage if isinstance(usage, dict) else {}
    tin, tout = u.get('prompt_tokens'), u.get('completion_tokens')
    cached = (u.get('prompt_tokens_details') or {}).get('cached_tokens')
    return {'model': model or '', 'tokens_in': tin, 'tokens_out': tout, 'tokens_cached': cached,
            'llm_cost_usd': cost_usd(model, tin, tout)}


def _verify_totals(rows):
    """검증·환각 집계 (SPEC §10.11). rows는 200 응답, 검증을 켠(mode≠off) 것만 센다."""
    vr = [r for r in rows if r.get('mode') and r.get('mode') != 'off' and r.get('verified') is not None]
    judged = sum(r.get('n_sentences') or 0 for r in vr)
    flagged = sum(r.get('n_flagged') or 0 for r in vr)
    reasons = {}
    for r in vr:
        if not r.get('verified'):
            k = r.get('unverified_reason') or 'unknown'
            reasons[k] = reasons.get(k, 0) + 1
    return {'responses': len(vr), 'fully_verified': sum(1 for r in vr if r.get('verified')),
            'responses_flagged': sum(1 for r in vr if (r.get('n_flagged') or 0) > 0),
            'sentences_judged': judged, 'sentences_flagged': flagged,
            'flag_rate': (flagged / judged) if judged else None, 'unverified_by_reason': reasons}


def _trim_totals(rows):
    """근거 다이어트 절감 집계 (SPEC §10.11)."""
    tr = [r for r in rows if (r.get('trim_chars_before') or 0) > 0]
    before = sum(r['trim_chars_before'] for r in tr)
    after = sum(r.get('trim_chars_after') or 0 for r in tr)
    return {'requests': len(tr), 'chars_before': before, 'chars_after': after,
            'saved_ratio': (1 - after / before) if before else None}


def _llm_totals(rows):
    out = {'tokens_in': 0, 'tokens_out': 0, 'tokens_cached': 0, 'est_cost_usd': 0.0, 'by_model': {}}
    for r in rows:
        if r.get('tokens_in') is None:
            continue
        m = out['by_model'].setdefault(r.get('model') or '?', {'requests': 0, 'tokens_in': 0, 'tokens_out': 0, 'est_cost_usd': 0.0})
        for d in (out, m):
            d['tokens_in'] += r.get('tokens_in') or 0
            d['tokens_out'] += r.get('tokens_out') or 0
            d['est_cost_usd'] += r.get('llm_cost_usd') or 0.0
        out['tokens_cached'] += r.get('tokens_cached') or 0
        m['requests'] += 1
    out['est_cost_usd'] = round(out['est_cost_usd'], 6)
    for m in out['by_model'].values():
        m['est_cost_usd'] = round(m['est_cost_usd'], 6)
    return out


def rejected(key_id, status, upstream='', stream=False, cold=False):
    """기록 객체가 없는 요청(인증 실패·한도·잘못된 요청·업스트림 오류)의 요약."""
    return {'key_id': key_id or '', 'upstream': upstream, 'status': status, 'stream': bool(stream), 'cold': bool(cold)}


def parse_window(w):
    m = re.fullmatch(r'(\d+)([mhd])', w or '')
    if not m:
        raise ValueError('window must look like 30m, 24h or 7d')
    return int(m.group(1)) * {'m': 60, 'h': 3600, 'd': 86400}[m.group(2)]


class MetricsStore:
    def __init__(self, path):
        self._db = sqlite3.connect(path, check_same_thread=False)
        self._db.executescript(SCHEMA)
        self._lock = threading.Lock()

    def add(self, row, ts=None):
        ts = time.time() if ts is None else ts
        with self._lock, self._db:
            self._db.execute('INSERT INTO requests VALUES (?,?,?,?)',
                             (ts, row.get('key_id', ''), row.get('status'), json.dumps({**row, 'ts': ts})))

    def add_start(self, warmup_ms, ts=None):
        with self._lock, self._db:
            self._db.execute('INSERT INTO starts VALUES (?,?)', (time.time() if ts is None else ts, warmup_ms))

    def rows(self, since):
        with self._lock:
            return [json.loads(r[0]) for r in self._db.execute('SELECT data FROM requests WHERE ts>=? ORDER BY ts', (since,))]

    def starts(self, since):
        with self._lock:
            return [{'ts': r[0], 'warmup_ms': r[1]} for r in
                    self._db.execute('SELECT ts, warmup_ms FROM starts WHERE ts>=? ORDER BY ts', (since,))]

    def prune(self, days):
        """보관 기간이 지난 기록을 지운다 (SPEC §10.5). 지운 요청 수."""
        cut = time.time() - days * 86400
        with self._lock, self._db:
            n = self._db.execute('DELETE FROM requests WHERE ts<?', (cut,)).rowcount
            self._db.execute('DELETE FROM starts WHERE ts<?', (cut,))
        return n

    def usage(self, key_id, window_s):
        """사용자 본인 사용량 (GET /v1/usage, SPEC §10.10)."""
        rows = [r for r in self.rows(time.time() - window_s) if r.get('key_id') == key_id and r.get('status') == 200]
        return {'requests': len(rows), **_llm_totals(rows), 'verification': _verify_totals(rows), 'trim': _trim_totals(rows)}

    def purge(self, key_id):
        with self._lock, self._db:
            return self._db.execute('DELETE FROM requests WHERE key_id=?', (key_id,)).rowcount

    def stats(self, window_s, label=''):
        """SPEC §10.6 관리자 통계."""
        since = time.time() - window_s
        rows = self.rows(since)
        ok = [r for r in rows if r.get('status') == 200]
        verified_rows = [r for r in ok if r.get('verified') is not None and r.get('mode') != 'off']
        count = lambda key: {str(k): sum(1 for r in rows if str(r.get(key)) == str(k))   # noqa: E731
                             for k in sorted({r.get(key) for r in rows if r.get(key) is not None}, key=str)}
        pcs = [r['post_check'] for r in ok if r.get('post_check')]
        ran = [p for p in pcs if p.get('verified')]
        cost = sum(p.get('in_tokens', 0) / 1e6 * PRICE_RT.get(p.get('model'), (2.0, 10.0))[0] +
                   p.get('out_tokens', 0) / 1e6 * PRICE_RT.get(p.get('model'), (2.0, 10.0))[1] for p in ran)
        reasons = {}
        for r in verified_rows:
            if not r.get('verified'):
                reasons[r.get('unverified_reason') or 'unknown'] = reasons.get(r.get('unverified_reason') or 'unknown', 0) + 1
        starts = self.starts(since)
        return {
            'window': label, 'since': since,
            'requests': {'total': len(rows), 'by_status': count('status'),
                         'by_key': {k: v for k, v in count('key_id').items() if k},
                         'by_upstream': {k: v for k, v in count('upstream').items() if k}},
            'latency': {'ttft_ms': {'p50': _pct([r.get('ttft_ms') for r in ok], 50), 'p95': _pct([r.get('ttft_ms') for r in ok], 95)},
                        'total_ms': {'p50': _pct([r.get('total_ms') for r in ok], 50), 'p95': _pct([r.get('total_ms') for r in ok], 95)},
                        'judge_ms_p95': _pct([r.get('judge_ms_p95') for r in ok], 95)},
            'unverified': {'rate': (sum(reasons.values()) / len(verified_rows)) if verified_rows else None, 'by_reason': reasons},
            'post_check': {'requested': len(pcs), 'runs': len(ran), 'flagged': sum(1 for p in ran if p.get('flagged')),
                           'quota': sum(1 for p in pcs if p.get('reason') == 'quota'), 'est_cost_usd': round(cost, 6)},
            'rate_limited': sum(1 for r in rows if r.get('status') == 429),
            'llm': _llm_totals(ok),
            'verification': _verify_totals(ok),
            'trim': _trim_totals(ok),
            'cold_starts': [{**st, 'first_ttft_ms': next((r.get('ttft_ms') for r in ok if r.get('cold') and r['ts'] >= st['ts']), None)}
                            for st in starts],
        }

In [ ]:
%%writefile gateway/hallugate/prices.py
"""사용자 LLM 추정 비용 (SPEC §10.10). 단가표는 prices.json — 운영자가 공개 가격표를 보고 갱신한다."""
import json
import os

_PATH = os.path.join(os.path.dirname(__file__), 'prices.json')
PRICES = json.load(open(_PATH, encoding='utf-8'))['models']


def price_for(model):
    """모델 이름의 가장 긴 접두어에 맞는 [입력, 출력] $/백만 토큰. 없으면 None."""
    if not model:
        return None
    keys = [k for k in PRICES if model.startswith(k)]
    return tuple(PRICES[max(keys, key=len)]) if keys else None


def cost_usd(model, tokens_in, tokens_out):
    p = price_for(model)
    if p is None or tokens_in is None or tokens_out is None:
        return None
    return tokens_in / 1e6 * p[0] + tokens_out / 1e6 * p[1]

In [ ]:
%%writefile gateway/hallugate/prices.json
{
 "_note": "사용자 LLM 추정 비용용 단가표 (SPEC §10.10). 값 = [입력, 출력] 달러/백만 토큰, 키 = 모델 이름 접두어(가장 긴 것이 우선). 운영자가 업체 공개 가격표를 보고 갱신한다. 캐시 할인은 반영하지 않는다. Claude 값은 SSOT M1의 배치 단가를 2배로 잡은 추정치.",
 "checked": "2026-11-23 (OpenAI 값은 공개 가격표 기준, 배포 전 다시 확인 필요)",
 "models": {
  "gpt-4o-mini": [0.15, 0.60],
  "gpt-4o": [2.50, 10.00],
  "gpt-4.1-nano": [0.10, 0.40],
  "gpt-4.1-mini": [0.40, 1.60],
  "gpt-4.1": [2.00, 8.00],
  "claude-haiku-5-5": [0.10, 0.50],
  "claude-sonnet-5-5": [2.00, 10.00],
  "claude-opus-5-5": [4.00, 20.00]
 }
}

In [ ]:
%%writefile gateway/hallugate/trim.py
"""근거 다이어트 (SPEC §2.9): 긴 근거에서 질문과 관련된 조각만 골라 LLM에 보내고 같은 조각으로 판정한다.

조각 나누기는 컬렉션(store.split_words)과 같고, 임베더도 컬렉션 검색과 같은 것을 쓴다.
고른 조각은 원문 순서대로 돌려준다(문서 흐름 유지).
"""
import numpy as np

from .store import split_words


class Trimmer:
    def __init__(self, embedder, chunk_words=120, overlap=20):
        self.embedder, self.chunk_words, self.overlap = embedder, chunk_words, overlap

    def _embed(self, texts):
        v = self.embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)
        return np.asarray(v, dtype=np.float32).reshape(len(texts), -1)

    def select(self, reference, query, k):
        """(고른 조각 리스트(원문 순서), 전체 조각 수). 조각이 k개 이하이거나 질문이 비면 전체를 그대로."""
        chunks = split_words(reference, self.chunk_words, self.overlap) or [reference]
        if len(chunks) <= k or not (query or '').strip():
            return chunks if len(chunks) > 1 else [reference], len(chunks)
        sims = self._embed(chunks) @ self._embed([query])[0]
        top = sorted(np.argsort(-sims)[:k].tolist())
        return [chunks[i] for i in top], len(chunks)

In [ ]:
%%writefile gateway/hallugate/app.py
"""OpenAI 호환 검증 게이트웨이.

서비스는 base_url만 이 게이트웨이로 바꾸면 된다.
    client = OpenAI(base_url="http://localhost:8000/v1", api_key=...)
    client.chat.completions.create(..., stream=True,
        extra_body={"verification": {"context": 근거문서, "mode": "hold", "action": "cut"}})

스트리밍 응답에는 일반 OpenAI 청크 사이에 판정 결과 청크가 섞여 나간다.
    {"object": "chat.completion.chunk", "choices": [], "verification": {...}}
choices가 비어 있으므로 일반 클라이언트는 본문에 영향 없이 무시하고, 필요한 클라이언트만
`chunk.verification` 으로 읽는다. 끊기면 finish_reason="content_filter".
"""
import asyncio
import json
import os
import time
import uuid
from types import SimpleNamespace
from contextlib import asynccontextmanager

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse, JSONResponse, Response, StreamingResponse

from . import context as ctxmod
from .config import Settings
from .pipeline import RequestRecord, VerifyPipeline, is_english
from .limits import Limiter
from .metrics import MetricsStore, parse_window, rejected, summarize
from .trim import Trimmer
from .postcheck import skipped
from .splitter import IncrementalSplitter
from .tokens import make_tokens


def _sse(obj):
    return f'data: {json.dumps(obj, ensure_ascii=False)}\n\n'


PASS_KEYS = ('tool_calls', 'function_call', 'refusal')


async def _upstream_deltas(resp, state):
    """업스트림 SSE → 본문 텍스트 조각(str) 또는 그대로 통과시킬 청크(dict).
    마지막 finish_reason, usage, 청크 메타데이터를 state에 남긴다 (SPEC §2.7, A10)."""
    async for line in resp.aiter_lines():
        if not line.startswith('data:'):
            continue
        data = line[5:].strip()
        if data == '[DONE]':
            break
        chunk = json.loads(data)
        state.setdefault('id', chunk.get('id'))
        state.setdefault('model', chunk.get('model'))
        if chunk.get('usage'):
            state['usage'] = chunk['usage']                 # 종료 청크 뒤에 그대로 보낸다
        for ch in chunk.get('choices') or []:
            if ch.get('finish_reason'):
                state['finish_reason'] = ch['finish_reason']
            delta = ch.get('delta') or {}
            other = {k: delta[k] for k in PASS_KEYS if delta.get(k)}
            if other:                                       # 본문이 아닌 부분(tool_calls 등)은 판정 없이 통과
                yield {**chunk, 'choices': [{'index': ch.get('index', 0), 'delta': other, 'finish_reason': None}],
                       **({'usage': None} if 'usage' in chunk else {})}
            text = delta.get('content')
            if text:
                yield text


async def _prepare(judge, ref, rec):
    """근거 준비. 실패하면 None (그 요청은 검증 없이 전달, fail-open)."""
    try:
        ref_idx = await asyncio.to_thread(judge.prepare, ref)
    except Exception:                                    # noqa: BLE001
        rec.verified, rec.unverified_reason = False, 'judge_error'
        return None
    rec.prepare_ms = ref_idx.prepare_ms
    return ref_idx


async def _post_check(checker, ref, text, rec, cut, timeout_ms, allow=None):
    """응답 끝 2단 검증 (SPEC §5.3). 본문은 바꾸지 않고 결과 dict만 돌려준다. 실패는 verified=False(fail-open)."""
    if cut:
        res = skipped('cut')
    elif checker is None:
        res = skipped('unavailable')
    else:
        sp = IncrementalSplitter()
        sents = [(s.idx, s.start, s.clean) for s in sp.feed(text) + sp.flush() if s.clean]
        if not sents:
            res = skipped('empty')
        elif allow is not None and not allow():          # 운영 모드 하루 한도 (SPEC §10.4)
            res = skipped('quota')
        else:
            try:
                res = await asyncio.wait_for(checker.check(ref, sents), timeout_ms / 1000)
            except asyncio.TimeoutError:
                res = skipped('timeout')
            except Exception:                                   # noqa: BLE001 — fail-open
                res = skipped('error')
    rec.post_check = res.to_dict()
    return rec.post_check


def _err(msg, status=400, kind='invalid_request_error'):
    return JSONResponse({'error': {'message': msg, 'type': kind}}, status_code=status)


def create_app(settings=None, judge=None, http_client=None, store=None, post_checker=None, keys=None, limiter=None):
    settings = settings or Settings()
    if settings.hosted and keys is None:                  # 운영 모드 API 키 저장소 (SPEC §10.1)
        from .keys import KeyStore
        keys = KeyStore(settings.keys_db)
    if post_checker is None and settings.anthropic_api_key:
        from .postcheck import ClaudePostChecker
        post_checker = ClaudePostChecker(api_key=settings.anthropic_api_key, model=settings.post_check_model,
                                         votes=settings.post_check_votes, threshold=settings.post_check_threshold)
    state = {'judge': judge, 'client': http_client, 'store': store, 'post_checker': post_checker, 'keys': keys,
             'limiter': limiter or Limiter(), 'cold': True,
             'metrics': MetricsStore(settings.keys_db) if settings.hosted else None}   # 운영 모드 기록 (SPEC §10.5)
    allow_up = dict(x.split('=', 1) for x in settings.upstream_allowlist.split(',') if '=' in x)   # SPEC §10.3
    tokens = make_tokens(settings.stride_tokenizer)

    def default_stride():
        return settings.judge_stride if settings.judge_stride is not None else getattr(state['judge'], 'default_stride', 0)

    @asynccontextmanager
    async def lifespan(app):
        if state['judge'] is None:
            from .judge import load_judge
            state['judge'] = load_judge(settings)
        # 예열: 첫 요청에서 GPU 커널 준비로 판정이 수백 ms 걸리는 것을 막는다 (S3 측정에서 첫 요청 468ms 관측)
        t_w = time.perf_counter()
        warm = state['judge'].prepare('warm up reference text ' * 40)
        for _ in range(3):
            state['judge'].judge(warm, 'This is a warm up sentence.')
        if state['metrics'] is not None:                 # 콜드 스타트 관측·보관 기간 정리 (SPEC §10.5·10.6)
            state['metrics'].add_start((time.perf_counter() - t_w) * 1000)
            state['metrics'].prune(settings.metrics_retention_days)
        if state['store'] is None and settings.collection_db:
            from .store import SQLiteStore
            emb = None if settings.collection_embedder else getattr(state['judge'], 'embedder', None)
            if emb is None or not hasattr(emb, 'encode'):
                from sentence_transformers import SentenceTransformer
                emb = SentenceTransformer(settings.collection_embedder or 'sentence-transformers/all-MiniLM-L6-v2')
            state['store'] = SQLiteStore(settings.collection_db, emb, settings.collection_chunk_words,
                                         settings.collection_overlap)
        own_client = state['client'] is None
        if own_client:
            state['client'] = httpx.AsyncClient(timeout=settings.upstream_timeout)
        yield
        if own_client:
            await state['client'].aclose()

    app = FastAPI(title='hallugate', lifespan=lifespan)

    def _emit(row):
        # 운영 모드: 원문 없는 요약을 SQLite와 표준 출력에 (SPEC §10.5)
        state['metrics'].add(row)
        print(json.dumps({'event': 'request', **row}, ensure_ascii=False), flush=True)

    def _log(rec):
        if settings.hosted:
            row = summarize(rec.to_dict(), rec.status, rec.stream, rec.cold)
            _emit(row)
            if rec.tenant and row.get('tokens_in') is not None:     # 예산용 월 누적 (SPEC §10.10)
                state['keys'].add_month_usage(rec.tenant, (row['tokens_in'] or 0) + (row['tokens_out'] or 0),
                                              row.get('llm_cost_usd') or 0.0)
            return
        if settings.metrics_path:
            with open(settings.metrics_path, 'a') as f:
                f.write(json.dumps(rec.to_dict(), ensure_ascii=False) + '\n')

    def _headers(request):
        h = {'Content-Type': 'application/json'}
        if settings.hosted:                              # 운영 모드: 사용자 LLM 키만, 운영자 키는 절대 안 씀 (SPEC §10.3)
            auth = request.headers.get('authorization')
        else:
            auth = f'Bearer {settings.upstream_api_key}' if settings.upstream_api_key else request.headers.get('authorization')
        if auth:
            h['Authorization'] = auth
        return h

    @app.get('/demo', include_in_schema=False)
    async def demo():
        # 발표·시연용 화면 (SPEC §2.8). 외부 리소스 없는 HTML 한 장
        path = os.path.join(os.path.dirname(__file__), 'static', 'demo.html')
        if not settings.demo or not os.path.isfile(path):
            return _err('demo page is disabled', 404, 'not_found_error')
        return HTMLResponse(open(path, encoding='utf-8').read())

    def _page(name):
        # 운영자·사용자 화면 (SPEC §10.11): 운영 모드 전용, 외부 리소스 없는 HTML 한 장
        path = os.path.join(os.path.dirname(__file__), 'static', name)
        if not settings.hosted or not os.path.isfile(path):
            return _err('page is available only in hosted mode', 404, 'not_found_error')
        return HTMLResponse(open(path, encoding='utf-8').read())

    @app.get('/admin', include_in_schema=False)
    async def admin_page():
        return _page('admin.html')

    @app.get('/console', include_in_schema=False)
    async def console_page():
        return _page('console.html')

    @app.get('/start', include_in_schema=False)
    async def start_page():
        return _page('start.html')

    @app.get('/health')
    async def health():
        if settings.hosted:                              # 운영 모드: 판정기 내부 값은 숨긴다 (SPEC §10.1)
            return {'ok': True, 'hosted': True}
        j, st, pc = state['judge'], default_stride(), state['post_checker']
        return {'ok': True, 'judge': type(j).__name__, 'input_mode': getattr(j, 'mode', ''),
                'mode': settings.default_mode, 'action': settings.default_action, 'stride': st,
                'stride_unit': tokens.name, 'threshold': j.threshold_for(st),
                'collections': state['store'] is not None, 'collections_auth': bool(settings.admin_token),
                'post_check': {'available': pc is not None, 'default': settings.post_check_default,
                               'model': getattr(pc, 'model', settings.post_check_model),
                               'votes': getattr(pc, 'votes', settings.post_check_votes),
                               'threshold': getattr(pc, 'threshold', settings.post_check_threshold)}}

    # ---------- 컬렉션 문서 등록 API (SPEC §2.5) ----------
    def _tenant(request):
        """운영 모드 인증 (SPEC §10.1): (Tenant 또는 None, 오류 응답 또는 None). 셀프호스팅이면 (None, None)."""
        if not settings.hosted:
            return None, None
        t = state['keys'].verify(request.headers.get('x-hallugate-key'))
        if t is None:
            return None, _err('invalid or missing x-hallugate-key', 401, 'authentication_error')
        return t, None

    def _cname(t, cid):
        # 운영 모드에서는 컬렉션 이름 앞에 키 ID를 붙여 사용자별로 분리한다 (SPEC §10.2)
        return f'{t.key_id}/{cid}' if t is not None else cid

    def _docs_access(request):
        """문서 API 접근 확인: (Tenant 또는 None, 오류 응답 또는 None)."""
        if settings.hosted:
            t, e = _tenant(request)
            if e:
                return None, e
        else:
            t = None
            if settings.admin_token and request.headers.get('authorization') != f'Bearer {settings.admin_token}':
                return None, _err('invalid or missing admin token', 401, 'authentication_error')
        if state['store'] is None:
            return None, _err('collections are disabled (COLLECTION_DB is empty)', 503, 'unavailable')
        return t, None

    @app.get('/v1/collections')
    async def list_collections(request: Request):
        t, e = _docs_access(request)
        if e:
            return e
        rows = await asyncio.to_thread(state['store'].collections)
        if t is not None:
            pre = _cname(t, '')
            rows = [{**r, 'collection': r['collection'][len(pre):]} for r in rows if r['collection'].startswith(pre)]
        return {'object': 'list', 'data': rows}

    @app.post('/v1/collections/{cid}/documents')
    async def add_documents(cid: str, request: Request):
        """{"text": ..., "title"?: ..., "doc_id"?: ...} 또는 {"documents": [그런 객체, ...]}. 같은 doc_id면 교체."""
        t, e = _docs_access(request)
        if e:
            return e
        b = await request.json()
        docs = b.get('documents') if isinstance(b.get('documents'), list) else [b]
        if not docs or not all(isinstance(d, dict) and isinstance(d.get('text'), str) and d['text'].strip() for d in docs):
            return _err('each document needs a non-empty "text"')
        store, name = state['store'], _cname(t, cid)
        if t is not None:                                # 운영 모드 크기 한도 (SPEC §10.2)
            if any(len(d['text']) > settings.max_doc_chars for d in docs):
                return _err(f'document longer than {settings.max_doc_chars} characters', 413, 'request_too_large')
            new = sum(1 for d in docs if not d.get('doc_id') or not store.doc_exists(name, d['doc_id']))
            if await asyncio.to_thread(store.count_docs, _cname(t, '')) + new > settings.max_docs:
                return _err(f'document limit ({settings.max_docs}) reached', 413, 'request_too_large')
        out = []
        for d in docs:
            out.append(await asyncio.to_thread(store.add, name, d['text'], d.get('title', ''), d.get('doc_id')))
        return {'object': 'list', 'collection': cid, 'data': out}

    @app.get('/v1/collections/{cid}/documents')
    async def list_documents(cid: str, request: Request):
        t, e = _docs_access(request)
        if e:
            return e
        name = _cname(t, cid)
        if t is not None and not await asyncio.to_thread(state['store'].exists, name):
            return _err(f'collection {cid} not found', 404, 'not_found_error')
        return {'object': 'list', 'collection': cid, 'data': await asyncio.to_thread(state['store'].list, name)}

    @app.delete('/v1/collections/{cid}/documents/{doc_id}')
    async def delete_document(cid: str, doc_id: str, request: Request):
        t, e = _docs_access(request)
        if e:
            return e
        if not await asyncio.to_thread(state['store'].delete, _cname(t, cid), doc_id):
            return _err(f'document {doc_id} not found in {cid}', 404, 'not_found_error')
        return {'deleted': True, 'collection': cid, 'doc_id': doc_id}

    def _admin_only(request):
        # 관리자 API (SPEC §10.1·10.6): 운영 모드에서만, ADMIN_TOKEN 필요
        if not settings.hosted:
            return _err('admin API is available only in hosted mode', 404, 'not_found_error')
        if not settings.admin_token or request.headers.get('authorization') != f'Bearer {settings.admin_token}':
            return _err('invalid or missing admin token', 401, 'authentication_error')
        return None

    @app.post('/admin/keys', include_in_schema=False)
    async def admin_create_key(request: Request):
        if (e := _admin_only(request)):
            return e
        b = await request.json() if await request.body() else {}
        limits = b.get('limits') or {}
        if not _valid_limits(limits):
            return _err('limits: integer rpm / concurrent / post_check_daily / monthly_tokens, number monthly_usd, '
                        'budget_action warn | block')
        kid, key = await asyncio.to_thread(state['keys'].create, str(b.get('name', '')), limits)
        return {'key_id': kid, 'key': key}

    @app.get('/admin/keys', include_in_schema=False)
    async def admin_list_keys(request: Request):
        if (e := _admin_only(request)):
            return e
        return {'object': 'list', 'data': await asyncio.to_thread(state['keys'].list)}

    @app.post('/admin/keys/{key_id}/revoke', include_in_schema=False)
    async def admin_revoke_key(key_id: str, request: Request):
        if (e := _admin_only(request)):
            return e
        b = await request.json() if await request.body() else {}
        out = {'key_id': key_id, 'revoked': await asyncio.to_thread(state['keys'].revoke, key_id)}
        if b.get('purge'):                               # 그 사용자의 컬렉션·기록 삭제 (SPEC §10.5)
            out['purged_docs'] = (await asyncio.to_thread(state['store'].purge, key_id + '/')) if state['store'] else 0
            out['purged_records'] = await asyncio.to_thread(state['metrics'].purge, key_id)
        return out

    @app.get('/admin/stats', include_in_schema=False)
    async def admin_stats(request: Request, window: str = '24h'):
        # 관리자 통계 (SPEC §10.6)
        if (e := _admin_only(request)):
            return e
        try:
            w = parse_window(window)
        except ValueError as ex:
            return _err(str(ex))
        m = state['metrics']
        await asyncio.to_thread(m.prune, settings.metrics_retention_days)
        return await asyncio.to_thread(m.stats, w, window)

    def _valid_limits(lm):
        if not isinstance(lm, dict):
            return False
        for k, v in lm.items():
            if k in ('rpm', 'concurrent', 'post_check_daily', 'monthly_tokens'):
                if not isinstance(v, int) or isinstance(v, bool) or v < 0:
                    return False
            elif k == 'monthly_usd':
                if not isinstance(v, (int, float)) or isinstance(v, bool) or v < 0:
                    return False
            elif k == 'budget_action':
                if v not in ('warn', 'block'):
                    return False
            else:
                return False
        return True

    @app.get('/v1/usage')
    async def usage(request: Request, window: str = '30d'):
        # 사용자 본인 LLM 사용량·예산 (SPEC §10.10)
        if not settings.hosted:
            return _err('usage is available only in hosted mode', 404, 'not_found_error')
        t, e = _tenant(request)
        if e:
            return e
        try:
            w = parse_window(window)
        except ValueError as ex:
            return _err(str(ex))
        out = await asyncio.to_thread(state['metrics'].usage, t.key_id, w)
        used, u = _budget(t)
        lm = t.limits or {}
        out['budget'] = None if used is None else {
            'month': u['month'], 'used': round(used, 4), 'tokens_used': u['tokens'], 'cost_used_usd': round(u['cost_usd'], 6),
            'monthly_tokens': lm.get('monthly_tokens'), 'monthly_usd': lm.get('monthly_usd'),
            'action': lm.get('budget_action', 'warn')}
        return {'window': window, **out}

    def _trim_embedder():
        # 근거 다이어트 임베더 = 컬렉션 검색과 같은 것 (SPEC §2.9)
        if state['store'] is not None:
            return state['store'].embedder
        emb = getattr(state['judge'], 'embedder', None)
        if hasattr(emb, 'encode') and not settings.collection_embedder:
            return emb
        if state.get('trim_emb') is None:
            from sentence_transformers import SentenceTransformer
            state['trim_emb'] = SentenceTransformer(settings.collection_embedder or 'sentence-transformers/all-MiniLM-L6-v2')
        return state['trim_emb']

    def _lim(t, name, default):
        return int((t.limits or {}).get(name, default))

    @app.post('/v1/chat/completions')
    async def chat(request: Request):
        tenant, e = _tenant(request)
        if e:
            if settings.hosted:
                _emit(rejected('', e.status_code))
            return e
        if tenant is None:
            return await _chat(request, None)
        resp = await _limited_chat(request, tenant)
        if not isinstance(resp, StreamingResponse) and not getattr(request.state, 'hg_logged', False):
            _emit(rejected(tenant.key_id, resp.status_code, request.headers.get('x-upstream', 'openai')))
        return resp

    def _budget(tenant):
        """이번 달 예산 사용 비율 (SPEC §10.10). 예산이 없으면 None."""
        lm = tenant.limits or {}
        if 'monthly_tokens' not in lm and 'monthly_usd' not in lm:
            return None, None
        u = state['keys'].month_usage(tenant.key_id)
        fr = [u['tokens'] / lm['monthly_tokens'] if lm.get('monthly_tokens') else 0.0,
              u['cost_usd'] / lm['monthly_usd'] if lm.get('monthly_usd') else 0.0]
        return max(fr), u

    async def _limited_chat(request, tenant):
        used, _ = _budget(tenant)
        resp = await _limited_chat_inner(request, tenant, used)
        if used is not None:
            resp.headers['x-budget-used'] = f'{used:.3f}'
            if used >= 1.0:
                resp.headers['x-budget-warning'] = 'exceeded'
            elif used >= 0.8:
                resp.headers['x-budget-warning'] = 'near'
        return resp

    async def _limited_chat_inner(request, tenant, used):
        if used is not None and used >= 1.0 and (tenant.limits or {}).get('budget_action') == 'block':
            return _err('monthly budget exceeded', 429, 'insufficient_quota')
        # 운영 모드 한도 (SPEC §10.4): 분당 → 동시. 거절된 요청은 업스트림을 부르지 않는다
        lim = state['limiter']
        ra = lim.hit(tenant.key_id, _lim(tenant, 'rpm', settings.rate_rpm))
        if ra is not None:
            r = _err('rate limit exceeded (requests per minute)', 429, 'rate_limit_error')
            r.headers['retry-after'] = str(ra)
            return r
        if not lim.acquire(tenant.key_id, _lim(tenant, 'concurrent', settings.max_concurrent)):
            r = _err('too many concurrent requests', 429, 'rate_limit_error')
            r.headers['retry-after'] = '1'
            return r
        try:
            resp = await _chat(request, tenant)
        except BaseException:
            lim.release(tenant.key_id)
            raise
        if isinstance(resp, StreamingResponse):          # 스트림이 끝나거나 끊길 때 풀어 준다
            inner = resp.body_iterator

            async def released():
                try:
                    async for x in inner:
                        yield x
                finally:
                    lim.release(tenant.key_id)
            resp.body_iterator = released()
        else:
            lim.release(tenant.key_id)
        return resp

    async def _chat(request, tenant):
        if tenant is not None:                           # 업스트림 허용 목록 (SPEC §10.3)
            up_name = request.headers.get('x-upstream', 'openai')
            if up_name not in allow_up:
                return _err(f'x-upstream must be one of {sorted(allow_up)}')
            up_base = allow_up[up_name]
        else:
            up_name, up_base = '', settings.upstream_base_url
        raw = await request.json()
        try:
            body, ref, mode, action, stride, opts = ctxmod.extract(raw, settings.default_mode, settings.default_action,
                                                                   default_stride(), with_stride=True, with_opts=True)
        except ValueError as e:
            return _err(str(e))
        if tenant is not None and ref and len(ref) > settings.max_ref_chars:
            return _err(f'reference longer than {settings.max_ref_chars} characters', 413, 'request_too_large')

        # 컬렉션 RAG (SPEC §2.6): 요청에 근거가 없고 collection이 있으면 검색 → 프롬프트 삽입 → 같은 근거로 판정
        evidence, coll = None, opts.get('collection') if not ref else None
        t_r = time.perf_counter()
        if coll:
            store = state['store']
            if store is None:
                return _err('collections are disabled (COLLECTION_DB is empty)', 503, 'unavailable')
            cname = _cname(tenant, coll)
            if not await asyncio.to_thread(store.exists, cname):
                return _err(f'collection {coll} not found', 404, 'not_found_error')
            evidence = await asyncio.to_thread(store.search, cname, ctxmod.last_user_text(body), settings.collection_top_k)
            if evidence:
                body['messages'] = ctxmod.rag_messages(body.get('messages', []), evidence)
                ref = ctxmod.rag_reference(evidence)
            else:
                mode = 'off'                               # 질문이 비어 근거를 못 찾음 → 검증 없이 전달
        retrieve_ms = (time.perf_counter() - t_r) * 1000

        # 근거 다이어트 (SPEC §2.9): 관련 조각만 LLM에 보내고 같은 조각으로 판정
        trim_info, tr = None, opts.get('trim')
        if tr and ref and not coll and mode != 'off':
            k = tr['top_k'] if isinstance(tr, dict) else settings.trim_top_k
            from_tag = not opts.get('context')
            query = ctxmod.strip_context_tags(ctxmod.last_user_text(body))
            trimmer = Trimmer(_trim_embedder(), settings.collection_chunk_words, settings.collection_overlap)
            kept, total = await asyncio.to_thread(trimmer.select, ref, query, k)
            new_ref = '\n\n'.join(kept) if total > k else ref
            if total > k:
                if from_tag:
                    body['messages'] = ctxmod.replace_context_tags(body.get('messages', []), new_ref)
                else:
                    body['messages'] = ctxmod.rag_messages(body.get('messages', []),
                                                           [SimpleNamespace(text=c) for c in kept])
            trim_info = {'kind': 'trim', 'chunks_total': total, 'chunks_kept': min(total, k) if total > k else total,
                         'chars_before': len(ref), 'chars_after': len(new_ref),
                         'saved_ratio': round(1 - len(new_ref) / max(len(ref), 1), 4), 'top_k': k}
            ref = new_ref

        client, judge = state['client'], state['judge']
        url = up_base.rstrip('/') + '/chat/completions'
        headers = _headers(request)
        rec = RequestRecord(mode=mode, action=action, stride=stride, tag=request.headers.get('x-request-tag', ''),
                            collection=coll or '', retrieve_ms=retrieve_ms if coll else 0.0,
                            n_evidence=len(evidence or []), tenant=tenant.key_id if tenant else '', upstream=up_name,
                            ref_chars=len(ref or ''), stream=bool(body.get('stream')), cold=state.pop('cold', False))
        thr = judge.threshold_for(stride)
        pc_on = bool(opts.get('post_check', settings.post_check_default)) and mode != 'off'
        vh = {'x-verification-mode': mode, 'x-verification-action': action, 'x-verification-stride': str(stride),
              'x-verification-post-check': 'on' if pc_on else 'off'}
        if coll:
            vh['x-verification-evidence'] = str(len(evidence or []))
        if tenant is not None:
            vh['x-upstream'] = up_name
        pc_allow = ((lambda: state['keys'].take_post_check(tenant.key_id, _lim(tenant, 'post_check_daily', settings.post_check_daily)))
                    if tenant is not None else None)
        ev_info = ({'kind': 'evidence', 'collection': coll, 'retrieve_ms': retrieve_ms,
                    'evidence': [e.to_dict() for e in evidence]} if evidence else None)
        if trim_info:
            vh['x-verification-trim'] = f"{trim_info['chunks_kept']}/{trim_info['chunks_total']}"
            rec.trim_chars_before, rec.trim_chars_after = trim_info['chars_before'], trim_info['chars_after']
            rec.trim_chunks_kept, rec.trim_chunks_total = trim_info['chunks_kept'], trim_info['chunks_total']

        # ---------- 비스트리밍 ----------
        if not body.get('stream'):
            r = await client.post(url, json=body, headers=headers)
            if r.status_code != 200 or mode == 'off':
                if r.status_code == 200:                 # 검증을 끈 요청도 기록 (오류는 바깥에서 상태 코드로 기록)
                    try:
                        _d = r.json(); rec.usage, rec.model = _d.get('usage'), _d.get('model', '')
                    except ValueError:
                        pass
                    rec.t_client_done = rec.now()
                    _log(rec)
                    request.state.hg_logged = True
                return Response(r.content, status_code=r.status_code, media_type='application/json', headers=vh)
            data = r.json()
            rec.usage, rec.model = data.get('usage'), data.get('model', '')
            content = data['choices'][0]['message'].get('content') or ''
            thr = judge.threshold_for(0)                  # 비스트리밍은 문장 끝에서만 판정
            ref_idx = await _prepare(judge, ref, rec)
            rec.threshold = thr
            sp = IncrementalSplitter()
            sents = sp.feed(content) + sp.flush()
            results, kept = [], ''
            for s in sents:
                if not s.clean or not rec.verified:
                    kept += s.text
                    continue
                reason = None
                if not is_english(s.clean):
                    reason = 'non_english'
                else:
                    try:
                        v = await asyncio.wait_for(asyncio.to_thread(judge.judge, ref_idx, s.clean, content[:s.start], thr),
                                                   settings.judge_timeout_ms / 1000)
                    except asyncio.TimeoutError:
                        reason = 'judge_timeout'
                    except Exception:                                   # noqa: BLE001 — fail-open
                        reason = 'judge_error'
                if reason:
                    rec.verified, rec.unverified_reason = False, reason
                    kept += s.text
                    continue
                results.append({'sentence_idx': s.idx, 'start': s.start, 'text': s.clean, 'score': v.score,
                                'hallucinated': v.hallucinated, 'judge_ms': v.judge_ms})
                rec.sentences.append(dict(results[-1]))
                if v.hallucinated and action == 'cut':
                    rec.cut = True
                    break
                kept += s.text
            if rec.cut and mode == 'hold':            # hold+cut: 환각 문장부터 잘라서 반환
                data['choices'][0]['message']['content'] = kept
                data['choices'][0]['finish_reason'] = 'content_filter'
            data['verification'] = {'mode': mode, 'action': action, 'stride': 0, 'threshold': thr,
                                    'prepare_ms': rec.prepare_ms, 'sentences': results, 'cut': rec.cut,
                                    'verified': rec.verified,
                                    **({'collection': coll, 'evidence': ev_info['evidence']} if ev_info else {}),
                                    **({'trim': trim_info} if trim_info else {}),
                                    **({'reason': rec.unverified_reason} if not rec.verified else {})}
            if pc_on:
                data['verification']['post_check'] = await _post_check(state['post_checker'], ref, content, rec, rec.cut,
                                                                       settings.post_check_timeout_ms, pc_allow)
            rec.t_client_done = rec.now()
            _log(rec)
            request.state.hg_logged = True
            return JSONResponse(data, headers=vh)

        # ---------- 스트리밍 ----------
        wants_usage = bool((body.get('stream_options') or {}).get('include_usage'))
        if tenant is not None:                           # 사용량 수집 (SPEC §10.10): 업스트림에는 항상 요청
            body['stream_options'] = {**(body.get('stream_options') or {}), 'include_usage': True}
        else:
            wants_usage = True                           # 셀프호스팅: 업스트림이 보낸 그대로
        req = client.build_request('POST', url, json=body, headers=headers)
        resp = await client.send(req, stream=True)
        if resp.status_code != 200:
            err = await resp.aread()
            await resp.aclose()
            return Response(err, status_code=resp.status_code, media_type='application/json', headers=vh)

        if mode == 'off':
            async def passthrough():
                try:
                    if tenant is None:
                        async for chunk in resp.aiter_raw():
                            yield chunk
                    else:                                # 운영 모드: usage만 가로채고 나머지는 그대로 (SPEC §10.10)
                        async for line in resp.aiter_lines():
                            if line.startswith('data:') and line[5:].strip() != '[DONE]':
                                try:
                                    c = json.loads(line[5:])
                                except ValueError:
                                    c = {}
                                rec.model = c.get('model') or rec.model
                                if c.get('usage'):
                                    rec.usage = c['usage']
                                    if not c.get('choices') and not wants_usage:
                                        continue
                            if line:
                                yield line + '\n\n'
                finally:
                    await resp.aclose()
                    rec.t_client_done = rec.now()
                    _log(rec)
            return StreamingResponse(passthrough(), media_type='text/event-stream', headers=vh)

        async def gen():
            meta = {}
            cid = 'chatcmpl-' + uuid.uuid4().hex[:24]
            created = int(time.time())

            def chunk(delta=None, finish=None, verification=None):
                c = {'id': meta.get('id') or cid, 'object': 'chat.completion.chunk', 'created': created,
                     'model': meta.get('model') or body.get('model', ''),
                     'choices': [] if delta is None and finish is None else
                     [{'index': 0, 'delta': delta or {}, 'finish_reason': finish}]}
                if verification is not None:
                    c['verification'] = verification
                return _sse(c)

            delivered = ''                                        # 사용자에게 나간 본문 (2단 검증 대상)
            try:
                ref_idx = await _prepare(judge, ref, rec)
                yield chunk(delta={'role': 'assistant', 'content': ''})
                if ev_info:                                       # 사용한 근거 조각을 본문보다 먼저 한 번 보낸다
                    yield chunk(verification=ev_info)
                if trim_info:                                     # 근거 다이어트 결과 (SPEC §2.9)
                    yield chunk(verification=trim_info)
                pipe = VerifyPipeline(judge, ref_idx, mode, action, rec, stride=stride, tokens=tokens,
                                      threshold=thr, timeout_ms=settings.judge_timeout_ms)
                async for ev in pipe.run(_upstream_deltas(resp, meta)):
                    if ev.kind == 'raw':
                        yield _sse(ev.info)
                    elif ev.kind == 'text':
                        if ev.text:
                            delivered += ev.text
                            yield chunk(delta={'content': ev.text})
                    elif ev.kind == 'verdict':
                        yield chunk(verification=ev.info)
                    elif ev.kind == 'cut':
                        yield chunk(finish='content_filter', verification={'cut': True, **ev.info})
                if not rec.cut:
                    yield chunk(finish=meta.get('finish_reason', 'stop'))
                if pc_on:                                         # 종료 청크 뒤, usage·[DONE] 앞에 1회 (SPEC §5.3)
                    yield chunk(verification=await _post_check(state['post_checker'], ref, delivered, rec, rec.cut,
                                                               settings.post_check_timeout_ms, pc_allow))
                rec.usage, rec.model = meta.get('usage'), meta.get('model') or ''
                if meta.get('usage') and wants_usage:
                    yield _sse({**json.loads(chunk()[6:]), 'usage': meta['usage']})
                yield 'data: [DONE]\n\n'
            finally:
                await resp.aclose()
                _log(rec)

        return StreamingResponse(gen(), media_type='text/event-stream', headers=vh)

    return app

In [ ]:
%%writefile gateway/serve.py
"""게이트웨이 실행.

  JUDGE_MODEL_DIR=judge_base UPSTREAM_BASE_URL=http://localhost:9000/v1 python serve.py --port 8000
  (판정기 없이 파이프라인만 시험: --lexical)
"""
import argparse

import uvicorn

from hallugate import Settings, create_app
from hallugate.judge import LexicalJudge

p = argparse.ArgumentParser()
p.add_argument('--port', type=int, default=8000)
p.add_argument('--host', default='127.0.0.1')
p.add_argument('--lexical', action='store_true', help='모델 없이 어휘 기반 대체 판정기 사용 (시험용)')
a = p.parse_args()

uvicorn.run(create_app(Settings(), judge=LexicalJudge() if a.lexical else None),
            host=a.host, port=a.port, log_level='warning')

In [ ]:
%%writefile gateway/mock_upstream.py
"""API 키 없이 게이트웨이를 시험하기 위한 가짜 LLM (OpenAI 호환).

요청 body에 다음 필드를 넣으면 그대로 재생한다 (실제 API로는 보내지 말 것):
  mock_response : 흘려보낼 답변 텍스트 (없으면 마지막 user 메시지를 되돌려줌)
  mock_ttft_ms  : 첫 토큰까지 지연 (기본 300ms)
  mock_token_ms : 토큰 간 간격 (기본 25ms ≈ 초당 40토큰, 상용 API와 비슷한 속도)
  mock_tool_call: 함수 이름. 본문 뒤에 tool_calls 청크 2개를 보내고 finish_reason=tool_calls
  stream_options.include_usage=true 이면 마지막에 usage 청크(choices: [])를 보낸다 (OpenAI와 같은 형식)

토큰 단위: 기본은 단어. 환경 변수 MOCK_TOKENIZER=tiktoken 이면 GPT 토큰(cl100k_base) 단위.

실행: python mock_upstream.py --port 9000
"""
import argparse
import asyncio
import json
import os
import re
import time
import uuid

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse

app = FastAPI(title='mock-llm')


_ENC = None
if os.environ.get('MOCK_TOKENIZER', 'words') == 'tiktoken':
    # 상용 API처럼 GPT 토큰 단위로 흘려보낸다 (E1·E2의 토큰 단위와 일치). 이어 붙이면 원문과 같다.
    import tiktoken
    _ENC = tiktoken.get_encoding('cl100k_base')


def _tokens(text):
    if _ENC is not None:
        ids = _ENC.encode(text)
        dec, starts = _ENC.decode_with_offsets(ids)
        if dec == text:
            ends = starts[1:] + [len(text)]
            return [text[a:b] for a, b in zip(starts, ends) if b > a]
    return re.findall(r'\S+\s*|\s+', text)


@app.post('/v1/chat/completions')
async def chat(request: Request):
    b = await request.json()
    text = b.get('mock_response')
    if text is None:
        users = [m for m in b.get('messages', []) if m.get('role') == 'user']
        text = users[-1]['content'] if users else 'Hello.'
    ttft, gap = b.get('mock_ttft_ms', 300) / 1000, b.get('mock_token_ms', 25) / 1000
    cid, created, model = 'chatcmpl-mock' + uuid.uuid4().hex[:12], int(time.time()), b.get('model', 'mock')

    if not b.get('stream'):
        await asyncio.sleep(ttft + gap * len(_tokens(text)))
        return JSONResponse({'id': cid, 'object': 'chat.completion', 'created': created, 'model': model,
                             'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': text},
                                          'finish_reason': 'stop'}],
                             'usage': {'prompt_tokens': 3, 'completion_tokens': 7, 'total_tokens': 10}})

    def c(delta, finish=None):
        return 'data: ' + json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': created,
                                      'model': model, 'choices': [{'index': 0, 'delta': delta,
                                                                   'finish_reason': finish}]}) + '\n\n'

    async def gen():
        yield c({'role': 'assistant', 'content': ''})
        await asyncio.sleep(ttft)
        for tok in _tokens(text):
            yield c({'content': tok})
            await asyncio.sleep(gap)
        tool = b.get('mock_tool_call')                    # 시험용: 본문 뒤에 도구 호출 청크
        if tool:
            yield c({'tool_calls': [{'index': 0, 'id': 'call_mock', 'type': 'function',
                                     'function': {'name': tool, 'arguments': ''}}]})
            yield c({'tool_calls': [{'index': 0, 'function': {'arguments': '{"q": 1}'}}]})
        yield c({}, 'tool_calls' if tool else 'stop')
        if (b.get('stream_options') or {}).get('include_usage'):
            yield 'data: ' + json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': created, 'model': model,
                                         'choices': [], 'usage': {'prompt_tokens': 3, 'completion_tokens': 7,
                                                                  'total_tokens': 10}}) + '\n\n'
        yield 'data: [DONE]\n\n'

    return StreamingResponse(gen(), media_type='text/event-stream')


if __name__ == '__main__':
    import uvicorn
    p = argparse.ArgumentParser()
    p.add_argument('--port', type=int, default=9000)
    a = p.parse_args()
    uvicorn.run(app, host='127.0.0.1', port=a.port, log_level='warning')

In [ ]:
%%writefile gateway/scripts/make_replay_set.py
"""RAGTruth dev 응답을 게이트웨이 재생용 JSONL로 만든다.

S1~S2-b와 같은 dev 표본(시드 42, 과제별 120개)에서 과제별 N개를 뽑는다.
각 줄: {id, task_type, reference, response, labels:[{start,end}]}

  python scripts/make_replay_set.py --ragtruth RAGTruth/dataset --per-task 20 --out replay_dev.jsonl
"""
import argparse
import json
import random

p = argparse.ArgumentParser()
p.add_argument('--ragtruth', default='RAGTruth/dataset')
p.add_argument('--per-task', type=int, default=20)
p.add_argument('--out', default='replay_dev.jsonl')
a = p.parse_args()

responses = [json.loads(l) for l in open(f'{a.ragtruth}/response.jsonl')]
sources = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{a.ragtruth}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']


def build_reference(src):          # S2와 동일
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))


TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']
random.seed(42)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)

rows = []
for tt in TASKS:
    for r in [x for x in dev if x['task_type'] == tt][:a.per_task]:
        rows.append({'id': str(r['id']), 'task_type': tt, 'reference': build_reference(sources[r['source_id']]),
                     'response': r['response'],
                     'labels': [{'start': l['start'], 'end': l['end']} for l in (r.get('labels') or [])]})
with open(a.out, 'w') as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print(f'{len(rows)}개 응답 → {a.out}')

In [ ]:
%%writefile gateway/scripts/bench.py
"""게이트웨이 종단 간(end-to-end) 측정.

가짜 LLM(mock_upstream)이 RAGTruth dev 응답을 실제 API와 비슷한 속도로 흘려보내고,
게이트웨이를 거쳐 받은 결과를 모드별로 비교한다.

  off  : 검증 없음 (기준선)
  pass : 즉시 전달 + 사후 판정
  hold : 문장 단위 보류 후 판정·전달

측정 항목
  [정확성] 받은 본문이 원래 응답과 글자 단위로 같은지 (게이트웨이가 텍스트를 망가뜨리지 않는지)
  [판별력] 게이트웨이 안에서 나온 문장 점수의 AUROC — 오프라인 실험(S2-b)과 같은 수준인지
  [지연]   첫 글자까지 시간(TTFT), 글자당 평균 지연(off 대비), 판정 소요 시간, pass 모드 판정 도착 지연
  [누출]   pass 모드에서 실제 환각 문장을 판정하기 전까지 사용자에게 이미 나간 글자 수

  python scripts/bench.py --replay replay_dev.jsonl --gateway http://127.0.0.1:8000 \
      --metrics gateway_metrics.jsonl --out bench_out
"""
import argparse
import asyncio
import json
import os
import statistics as st
import time

import httpx


def auroc(y, s):
    pos = [v for v, t in zip(s, y) if t]
    neg = [v for v, t in zip(s, y) if not t]
    if not pos or not neg:
        return float('nan')
    ranked = sorted([(v, i) for i, v in enumerate(s)])
    ranks, i = [0.0] * len(s), 0
    while i < len(ranked):                      # 동점 평균 순위
        j = i
        while j + 1 < len(ranked) and ranked[j + 1][0] == ranked[i][0]:
            j += 1
        for k in range(i, j + 1):
            ranks[ranked[k][1]] = (i + j) / 2 + 1
        i = j + 1
    rp = sum(r for r, t in zip(ranks, y) if t)
    return (rp - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))


def pct(xs, q):
    xs = sorted(x for x in xs if x is not None)
    if not xs:
        return float('nan')
    return xs[min(len(xs) - 1, int(round(q / 100 * (len(xs) - 1))))]


def parse_cfg(cfg):
    """'pass' → ('pass', 0), 'hold:4' → ('hold', 4). 콜론 뒤 숫자는 앞부분 재판정 간격(stride)."""
    m, _, s = cfg.partition(':')
    return m, int(s or 0)


def item_timing(item, a):
    """재생 속도. --timing 파일이 있으면 그 응답의 실측 값(L1: 실제 LLM의 첫 토큰 시간, 글자당 생성 시간)을 쓴다.
    글자당 시간 × (이 응답의 평균 글자/토큰)으로 토큰 간격을 만든다 — 가짜 LLM은 GPT 토큰 단위로 흘려보낸다."""
    t = TIMING.get(item['id'])
    if not t:
        return a.ttft_ms, a.token_ms
    n_tok = len(_ENC.encode(item['response'])) if _ENC else max(1, len(item['response']) / 4.0)
    return t['ttft_ms'], t['ms_per_char'] * len(item['response']) / max(n_tok, 1)


async def one(client, gw, item, mode, a):
    vmode, stride = parse_cfg(mode)
    ttft_ms, token_ms = item_timing(item, a)
    body = {'model': 'mock', 'stream': True,
            'messages': [{'role': 'system', 'content': 'Answer using only the provided context.'},
                         {'role': 'user', 'content': 'Respond based on the context.'}],
            'verification': {'context': item['reference'], 'mode': vmode, 'action': 'flag', 'stride': stride},
            'mock_response': item['response'], 'mock_ttft_ms': ttft_ms, 'mock_token_ms': token_ms}
    tag = f"{item['id']}|{mode}"
    t0 = time.perf_counter()
    timeline, text, verdicts = [], '', []
    async with client.stream('POST', f'{gw}/v1/chat/completions', json=body,
                             headers={'x-request-tag': tag}) as r:
        r.raise_for_status()
        async for line in r.aiter_lines():
            if not line.startswith('data:') or line.endswith('[DONE]'):
                continue
            c = json.loads(line[5:])
            now = (time.perf_counter() - t0) * 1000
            if c.get('verification'):
                verdicts.append({**c['verification'], 't_client': now})
            for ch in c.get('choices') or []:
                d = (ch.get('delta') or {}).get('content')
                if d:
                    text += d
                    timeline.append((now, len(text)))
    return {'id': item['id'], 'task_type': item['task_type'], 'mode': mode, 'tag': tag,
            'text_ok': text == item['response'], 'timeline': timeline, 'verdicts': verdicts,
            't_total': (time.perf_counter() - t0) * 1000}


def onset_metrics(rs, items):
    """환각 발생 시점 기준 지표 (SSOT 결정: 누출은 '환각 발생부터' 센다).

    발생 위치 θ = 응답에서 첫 환각 구간의 시작 글자. 클라이언트가 받은 판정 이벤트를 받은 순서대로 본다.
    - 탐지(엄격): 응답의 첫 경보(hallucinated=true)가 θ를 포함하는 텍스트를 판정한 것(upto > θ)일 때만 탐지.
      첫 경보가 θ 이전 텍스트에 대한 것이면 발생 전 오경보로 끝난 것(미탐지).
    - 탐지 지연(글자): 경보 시점에 판정한 텍스트의 끝 − θ (생성이 θ를 지나 얼마나 진행됐을 때 잡았나)
    - 발생 후 누출(글자): 경보가 나온 순간까지 사용자에게 나간 글자 중 θ 이후 글자 수
      미탐지면 θ 이후 전부가 누출된 것으로 보고 '기대 누출'에 포함
    - 멀쩡한 응답 표시율: 환각 없는 응답 중 경보가 한 번이라도 나간 비율
    """
    info = {it['id']: it for it in items}
    det, delay, leak_det, leak_all, clean_flag = [], [], [], [], []
    for r in rs:
        it = info[r['id']]
        alarms = [v for v in r['verdicts'] if v.get('hallucinated') and 'upto' in v]
        if not it['labels']:
            clean_flag.append(bool(alarms)); continue
        theta = min(l['start'] for l in it['labels'])
        after = len(it['response']) - theta
        first = alarms[0] if alarms else None
        if first is not None and first['upto'] > theta:
            det.append(True)
            delay.append(first['upto'] - theta)
            lk = max(0, first.get('released_at_verdict', 0) - theta)
            leak_det.append(lk); leak_all.append(lk)
        else:
            det.append(False); leak_all.append(after)
    n = max(len(det), 1)
    return {'onset_n_hallucinated': len(det), 'onset_recall_strict': sum(det) / n,
            'onset_detect_delay_chars_p50': pct(delay, 50),
            'onset_leak_chars_p50_detected': pct(leak_det, 50),
            'onset_leak_chars_mean_detected': st.mean(leak_det) if leak_det else float('nan'),
            'onset_expected_leak_chars_mean': st.mean(leak_all) if leak_all else float('nan'),
            'clean_resp_flagged': sum(clean_flag) / max(len(clean_flag), 1)}


def char_times(timeline, n):
    """글자 i가 사용자 화면에 나타난 시각 리스트."""
    out, j = [], 0
    for i in range(n):
        while j < len(timeline) and timeline[j][1] < i + 1:
            j += 1
        out.append(timeline[j][0] if j < len(timeline) else None)
    return out


TIMING = {}
try:
    import tiktoken
    _ENC = tiktoken.get_encoding('cl100k_base')
except Exception:          # 오프라인 등
    _ENC = None


async def main(a):
    items = [json.loads(l) for l in open(a.replay)][:a.n or None]
    if a.timing:
        for l in open(a.timing):
            t = json.loads(l)
            TIMING[str(t['id'])] = t
        print(f'실측 재생 속도 사용: {a.timing} ({len(TIMING)}개 응답)')
    if os.path.exists(a.metrics):
        os.remove(a.metrics)                      # 이번 측정분만 남기기 (게이트웨이와 같은 경로여야 함)
    runs = []
    async with httpx.AsyncClient(timeout=300) as client:
        for i, it in enumerate(items):
            for mode in a.modes:
                runs.append(await one(client, a.gateway, it, mode, a))
            if (i + 1) % 10 == 0:
                print(f'  {i + 1}/{len(items)} 응답 완료')
    await asyncio.sleep(0.5)
    recs = {}
    if os.path.exists(a.metrics):
        for l in open(a.metrics):
            r = json.loads(l)
            recs[r['tag']] = r

    by = {(r['id'], r['mode']): r for r in runs}
    labels = {it['id']: it['labels'] for it in items}
    lens = {it['id']: len(it['response']) for it in items}
    os.makedirs(a.out, exist_ok=True)
    summary, sent_rows = {}, []

    for mode in a.modes:
        rs = [r for r in runs if r['mode'] == mode]
        s = {'n_responses': len(rs), 'text_ok_rate': sum(r['text_ok'] for r in rs) / len(rs),
             'ttft_ms_p50': pct([r['timeline'][0][0] if r['timeline'] else None for r in rs], 50),
             'total_ms_p50': pct([r['t_total'] for r in rs], 50)}
        if mode != 'off' and 'off' in a.modes:
            delays = []
            for r in rs:
                off = by[(r['id'], 'off')]
                n = lens[r['id']]
                ct, co = char_times(r['timeline'], n), char_times(off['timeline'], n)
                delays += [x - y for x, y in zip(ct, co) if x is not None and y is not None]
            s['char_delay_ms_mean'] = st.mean(delays) if delays else float('nan')
            s['char_delay_ms_p95'] = pct(delays, 95)
            s['ttft_added_ms_p50'] = pct([(r['timeline'][0][0] - by[(r['id'], 'off')]['timeline'][0][0])
                                          for r in rs if r['timeline']], 50)
        if mode != 'off':
            rows = []
            prep = []
            for r in rs:
                rec = recs.get(r['tag'])
                if not rec:
                    continue
                prep.append(rec['prepare_ms'])
                for sr in rec['sentences']:
                    if sr.get('skipped') or 'score' not in sr:
                        continue
                    a0, a1 = sr['start'], sr['start'] + sr['n_chars']
                    y = int(any(min(a1, l['end']) > max(a0, l['start']) for l in labels[r['id']]))
                    rows.append({'id': r['id'], 'task_type': r['task_type'], 'mode': mode,
                                 'sentence_idx': sr['sentence_idx'], 'y': y, 'score': sr['score'],
                                 'hallucinated': sr['hallucinated'], 'judge_ms': sr['judge_ms'],
                                 'verdict_lag_ms': sr['t_verdict'] - sr['t_complete'],
                                 'release_delay_ms': (sr['t_release'] - sr['t_complete'])
                                 if sr.get('t_release') is not None else None,
                                 'leak_chars': sr.get('leak_chars')})
            sent_rows += rows
            ys = [x['y'] for x in rows]
            tp = sum(1 for x in rows if x['y'] and x['hallucinated'])
            s.update({'n_sentences': len(rows), 'pos_rate': sum(ys) / max(len(ys), 1),
                      'AUROC': auroc(ys, [x['score'] for x in rows]),
                      'recall_at_threshold': tp / max(sum(ys), 1),
                      'precision_at_threshold': tp / max(sum(x['hallucinated'] for x in rows), 1),
                      'judge_ms_p50': pct([x['judge_ms'] for x in rows], 50),
                      'judge_ms_p95': pct([x['judge_ms'] for x in rows], 95),
                      'verdict_lag_ms_p50': pct([x['verdict_lag_ms'] for x in rows], 50),
                      'verdict_lag_ms_p95': pct([x['verdict_lag_ms'] for x in rows], 95),
                      'prepare_ms_p50': pct(prep, 50)})
            if parse_cfg(mode)[0] == 'pass':
                # (구 지표) 실제 환각이면서 잡아낸 문장: 판정이 나올 때까지 그 문장 시작부터 사용자에게 나간 글자 수
                caught = [x['leak_chars'] for x in rows if x['y'] and x['hallucinated']]
                s['leak_chars_p50_on_caught'] = pct(caught, 50)
                s['leak_chars_p95_on_caught'] = pct(caught, 95)
            s.update(onset_metrics(rs, items))
        summary[mode] = s

    json.dump(summary, open(f'{a.out}/bench_summary.json', 'w'), indent=1, ensure_ascii=False)
    with open(f'{a.out}/bench_sentences.jsonl', 'w') as f:
        for row in sent_rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

    speed = f'실측 속도 {a.timing}' if a.timing else f'토큰 간격 {a.token_ms}ms, 첫 토큰 {a.ttft_ms}ms'
    print(f'\n=== 결과 ({len(items)}개 응답, {speed}) ===')
    for mode, s in summary.items():
        print(f'\n[{mode}]')
        for k, v in s.items():
            print(f'  {k:26} {v:.3f}' if isinstance(v, float) else f'  {k:26} {v}')
    print(f'\n저장: {a.out}/bench_summary.json, {a.out}/bench_sentences.jsonl')


if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--replay', default='replay_dev.jsonl')
    p.add_argument('--gateway', default='http://127.0.0.1:8000')
    p.add_argument('--metrics', default='gateway_metrics.jsonl', help='게이트웨이의 METRICS_PATH와 같은 파일')
    p.add_argument('--modes', nargs='+', default=['off', 'pass', 'hold'],
                   help="검증 설정 목록. 'pass:4'처럼 콜론 뒤에 앞부분 재판정 간격(stride)을 붙일 수 있다")
    p.add_argument('--n', type=int, default=0, help='앞에서 n개만 (0=전체)')
    p.add_argument('--ttft-ms', type=float, default=300)
    p.add_argument('--token-ms', type=float, default=25)
    p.add_argument('--timing', default='', help='L1 실측 재생 속도 파일 (id, ttft_ms, ms_per_char). 없으면 고정값')
    p.add_argument('--out', default='bench_out')
    asyncio.run(main(p.parse_args()))

In [ ]:
%%writefile gateway/scripts/keys.py
"""운영 모드 API 키 관리 (SPEC §10.1). 셀프 가입이 없으므로 관리자만 발급·폐기한다.

배포된 게이트웨이(관리자 API, 관리자 토큰은 ADMIN_TOKEN 환경 변수):
  ADMIN_TOKEN=... python scripts/keys.py --url https://<게이트웨이> create --name alice [--limits '{"rpm": 60}']
  ADMIN_TOKEN=... python scripts/keys.py --url https://<게이트웨이> list
  ADMIN_TOKEN=... python scripts/keys.py --url https://<게이트웨이> revoke <키 ID> [--purge]

로컬 저장소 직접(셀프호스팅·테스트):
  python scripts/keys.py --db hallugate.sqlite3 [--collections collections.sqlite3] create|list|revoke ...

--purge: 그 사용자의 컬렉션(문서·조각)과 측정 기록을 모두 지운다 (SPEC §10.5).
"""
import argparse
import datetime
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

TRANSPORT = None          # 테스트에서 바꿔 끼우는 httpx 전송 계층


def _ts(x):
    return datetime.datetime.fromtimestamp(x).strftime('%Y-%m-%d %H:%M') if x else '-'


def _print_list(rows):
    for row in rows:
        lim = json.dumps(row.get('limits') or {}) if row.get('limits') else ''
        print(f"{row['key_id']}  {row['name'] or '-':20}  만든 시각 {_ts(row['created'])}  폐기 {_ts(row['revoked'])}  {lim}")


def _created(kid, name, key):
    print(f'키 ID {kid} ({name})\n{key}\n이 키는 다시 볼 수 없습니다. 사용자에게 안전하게 전달하세요.')


def remote(a, limits):
    import httpx
    tok = os.environ.get('ADMIN_TOKEN', '')
    if not tok:
        sys.exit('ADMIN_TOKEN 환경 변수가 필요합니다')
    with httpx.Client(base_url=a.url.rstrip('/'), headers={'Authorization': f'Bearer {tok}'}, timeout=60,
                      **({'transport': TRANSPORT} if TRANSPORT else {})) as c:
        if a.cmd == 'create':
            r = c.post('/admin/keys', json={'name': a.name, 'limits': limits}); r.raise_for_status()
            _created(r.json()['key_id'], a.name, r.json()['key'])
        elif a.cmd == 'list':
            r = c.get('/admin/keys'); r.raise_for_status(); _print_list(r.json()['data'])
        else:
            r = c.post(f'/admin/keys/{a.key_id}/revoke', json={'purge': a.purge}); r.raise_for_status()
            d = r.json()
            print('폐기함' if d['revoked'] else '이미 폐기됐거나 없는 키')
            if a.purge:
                print(f"컬렉션 문서 {d.get('purged_docs', 0)}개, 측정 기록 {d.get('purged_records', 0)}줄 삭제")


def local(a, limits):
    from hallugate.keys import KeyStore
    ks = KeyStore(a.db)
    if a.cmd == 'create':
        kid, key = ks.create(a.name, limits)
        _created(kid, a.name, key)
    elif a.cmd == 'list':
        _print_list(ks.list())
    else:
        print('폐기함' if ks.revoke(a.key_id) else '이미 폐기됐거나 없는 키')
        if a.purge:
            from hallugate.metrics import MetricsStore
            from hallugate.store import SQLiteStore
            st = SQLiteStore(a.collections, None)
            print(f'컬렉션 문서 {st.purge(a.key_id + "/")}개 삭제')
            st.close()
            print(f'측정 기록 {MetricsStore(a.db).purge(a.key_id)}줄 삭제')
    ks.close()


def main(argv=None):
    p = argparse.ArgumentParser(description='hallugate API 키 관리')
    g = p.add_mutually_exclusive_group()
    g.add_argument('--url', help='배포된 게이트웨이 주소 (관리자 API 사용)')
    g.add_argument('--db', help='로컬 키 저장소 파일 (직접 수정)')
    p.add_argument('--collections', default=os.environ.get('COLLECTION_DB', 'collections.sqlite3'))
    sub = p.add_subparsers(dest='cmd', required=True)
    c = sub.add_parser('create'); c.add_argument('--name', default=''); c.add_argument('--limits', default='{}')
    sub.add_parser('list')
    r = sub.add_parser('revoke'); r.add_argument('key_id'); r.add_argument('--purge', action='store_true')
    a = p.parse_args(argv)
    limits = json.loads(a.limits) if a.cmd == 'create' else None
    if a.url:
        remote(a, limits)
    else:
        a.db = a.db or os.environ.get('KEYS_DB', 'hallugate.sqlite3')
        local(a, limits)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile gateway/scripts/warm.py
"""관리형 베타: 쓸 때만 컨테이너 켜기 (SPEC §10.7).

평소에는 최소 0대(요청이 없으면 꺼짐, A10G 콜드 스타트 약 20초). 시연처럼 첫 요청도 바로 응답해야 할 때 1대를 켜 두고, 끝나면 끈다.
재배포 없이 Modal 자동 확장 설정만 바꾸므로 몇 초면 된다. 끄는 것을 잊어도 배포의 auto_cooldown이
끌 시각이 지나면 10분 안에 끈다.

  python scripts/warm.py up [--hours 2] [--wait --url https://<게이트웨이>]
  python scripts/warm.py status
  python scripts/warm.py down
"""
import argparse
import datetime
import time

APP, FUNCTION, DICT = 'hallugate', 'web', 'hallugate-warm'
MAX_HOURS = 12

BACKEND = None            # 테스트에서 바꿔 끼우는 Modal 연결
NOW = time.time


class ModalBackend:
    """web 함수의 최소 컨테이너 수와 끌 시각(Modal Dict)을 다룬다."""
    def __init__(self):
        import modal
        self.fn = modal.Function.from_name(APP, FUNCTION)
        self.d = modal.Dict.from_name(DICT, create_if_missing=True)

    def set_min(self, n):
        self.fn.autoscaler.update(min_containers=n)

    def get(self, k):
        return self.d.get(k)

    def put(self, k, v):
        self.d.put(k, v)

    def pop(self, k):
        try:
            self.d.pop(k)
        except KeyError:
            pass

    def ping(self, url, limit=1200):
        import httpx
        t0 = time.time()
        while time.time() - t0 < limit:
            try:
                r = httpx.get(url.rstrip('/') + '/health', timeout=200, follow_redirects=True)
                if r.status_code == 200:
                    print(f'준비됨: {time.time() - t0:.0f}초 걸림')
                    return True
            except httpx.HTTPError:
                pass
            time.sleep(5)
        print(f'{limit}초 안에 응답이 없습니다. modal app logs hallugate로 확인하세요')
        return False


def cooldown(b, now):
    """끌 시각이 지났으면 최소 0으로 되돌린다. 바꿨으면 True (배포의 auto_cooldown이 10분마다 부름)."""
    until = b.get('until')
    if until is None or now < until:
        return False
    b.set_min(0)
    b.pop('until')
    return True


def _hhmm(ts):
    return datetime.datetime.fromtimestamp(ts).strftime('%m-%d %H:%M')


def _hours(x):
    h = float(x)
    if not 0 < h <= MAX_HOURS:
        raise argparse.ArgumentTypeError(f'0보다 크고 {MAX_HOURS} 이하')
    return h


def main(argv=None):
    ap = argparse.ArgumentParser(description='관리형 베타 컨테이너 켜기·끄기 (SPEC §10.7)')
    sub = ap.add_subparsers(dest='cmd', required=True)
    up = sub.add_parser('up', help='최소 1대로 켜고 끌 시각을 적는다')
    up.add_argument('--hours', type=_hours, default=2.0, help=f'몇 시간 뒤 자동으로 끌지 (기본 2, 최대 {MAX_HOURS})')
    up.add_argument('--wait', action='store_true', help='/health가 응답할 때까지 기다림')
    up.add_argument('--url', default='', help='게이트웨이 주소 (--wait에 필요)')
    sub.add_parser('down', help='최소 0대로 되돌린다')
    sub.add_parser('status', help='끌 시각')
    a = ap.parse_args(argv)
    if a.cmd == 'up' and a.wait and not a.url:
        ap.error('--wait에는 --url이 필요합니다')

    b = BACKEND or ModalBackend()
    now = NOW()
    if a.cmd == 'up':
        until = now + a.hours * 3600
        b.set_min(1)
        b.put('until', until)
        print(f'켬: 최소 1대, {_hhmm(until)}에 자동으로 끕니다 (끝나면 python scripts/warm.py down)')
        if a.wait:
            b.ping(a.url)
    elif a.cmd == 'down':
        b.set_min(0)
        b.pop('until')
        print('끔: 최소 0대 (남은 컨테이너는 유휴 5분 뒤 내려갑니다)')
    else:
        until = b.get('until')
        if until is None:
            print('꺼져 있음 (최소 0대)')
        else:
            print(f'켜져 있음: {_hhmm(until)}에 자동으로 끔 (남은 {max(0, (until - now) / 60):.0f}분)')


if __name__ == '__main__':
    main()

In [ ]:
%%writefile gateway/tests/test_gateway.py
"""게이트웨이 단위·통합 테스트 (모델 다운로드 없이 실행 가능).

  cd gateway && python -m pytest -q
"""
import json
import os
import random
import sys

import httpx
import pytest

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from hallugate import Settings, create_app                      # noqa: E402
from hallugate.context import extract                            # noqa: E402
from hallugate.judge import LexicalJudge                         # noqa: E402
from hallugate.splitter import IncrementalSplitter               # noqa: E402
import mock_upstream                                             # noqa: E402

REF = 'The company was founded in 1998 in Seoul. It reported revenue of 2 billion dollars in 2023.'
GOOD = 'The company was founded in 1998 in Seoul. It reported revenue of 2 billion dollars in 2023.'
BAD_2ND = 'The company was founded in 1998 in Seoul. Its chief executive flew rockets across Mars yesterday. It reported revenue of 2 billion dollars in 2023.'


# ---------------- 문장 분할 ----------------
@pytest.mark.parametrize('text', [
    GOOD, BAD_2ND,
    'Dr. Kim arrived at 3.5 p.m. on Jan. 5. He left early!  Then what?\n\nNew paragraph here.',
    'No terminal punctuation at all',
])
def test_splitter_streaming_matches_whole(text):
    rng = random.Random(0)
    sp, got, i = IncrementalSplitter(), [], 0
    while i < len(text):
        j = i + rng.randint(1, 6)
        got += sp.feed(text[i:j])
        i = j
    got += sp.flush()
    assert ''.join(s.text for s in got) == text                    # 글자 하나도 잃지 않음
    assert [s.start for s in got] == [sum(len(x.text) for x in got[:k]) for k in range(len(got))]
    whole = IncrementalSplitter()
    ref = whole.feed(text) + whole.flush()
    assert [s.clean for s in got] == [s.clean for s in ref]        # 조각 단위로 받아도 한 번에 받은 것과 같음


# ---------------- 컨텍스트 추출 ----------------
def test_extract_field_and_strip():
    body = {'model': 'x', 'messages': [], 'verification': {'context': ['a', 'b'], 'mode': 'hold', 'action': 'cut'}}
    up, ref, mode, action = extract(body)
    assert 'verification' not in up and ref == 'a\n\nb' and (mode, action) == ('hold', 'cut')


def test_extract_tag_and_off():
    body = {'messages': [{'role': 'user', 'content': 'Q?\n<context>doc one</context> <context>doc two</context>'}]}
    assert extract(body)[1:3] == ('doc one\n\ndoc two', 'pass')
    assert extract({'messages': [{'role': 'user', 'content': 'hi'}]})[2] == 'off'
    with pytest.raises(ValueError):
        extract({'messages': [], 'verification': {'context': 'x', 'mode': 'bogus'}})


# ---------------- 게이트웨이 통합 (가짜 LLM 연결) ----------------
@pytest.fixture
def gw(tmp_path):
    up = httpx.AsyncClient(transport=httpx.ASGITransport(app=mock_upstream.app), base_url='http://mock')
    s = Settings(upstream_base_url='http://mock/v1', metrics_path=str(tmp_path / 'm.jsonl'), stride_tokenizer='words')
    app = create_app(s, judge=LexicalJudge(threshold=0.5), http_client=up)
    return httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url='http://gw'), s


def body(resp, mode, action='flag', stream=True):
    return {'model': 'mock', 'stream': stream, 'messages': [{'role': 'user', 'content': 'q'}],
            'verification': {'context': REF, 'mode': mode, 'action': action},
            'mock_response': resp, 'mock_ttft_ms': 0, 'mock_token_ms': 0}


async def run(client, b):
    r = await client.post('/v1/chat/completions', json=b)
    assert r.status_code == 200
    text, events, finish = '', [], None
    for line in r.text.splitlines():
        if not line.startswith('data:') or line.endswith('[DONE]'):
            continue
        c = json.loads(line[5:])
        if c.get('verification'):
            events.append(('v', c['verification']))
        for ch in c['choices']:
            d = ch['delta'].get('content')
            if d:
                text += d
                events.append(('t', d))
            finish = ch.get('finish_reason') or finish
    return text, events, finish


@pytest.mark.asyncio
@pytest.mark.parametrize('mode', ['pass', 'hold'])
async def test_stream_text_intact_and_verdicts(gw, mode):
    client, s = gw
    text, ev, finish = await run(client, body(BAD_2ND, mode))
    assert text == BAD_2ND and finish == 'stop'
    vs = [e for k, e in ev if k == 'v']
    assert [v['sentence_idx'] for v in vs] == [0, 1, 2]
    assert [v['hallucinated'] for v in vs] == [False, True, False]
    rec = json.loads(open(s.metrics_path).read().splitlines()[-1])
    assert rec['mode'] == mode and len(rec['sentences']) == 3
    if mode == 'hold':
        # hold: 각 문장의 판정이 그 문장 본문보다 먼저 나가야 함 → 누출 0
        order = [k for k, _ in ev]
        assert order == ['v', 't', 'v', 't', 'v', 't']
        assert all(x.get('leak_chars', 0) == 0 for x in rec['sentences'])


@pytest.mark.asyncio
async def test_hold_cut_blocks_bad_sentence(gw):
    client, _ = gw
    text, ev, finish = await run(client, body(BAD_2ND, 'hold', 'cut'))
    assert finish == 'content_filter'
    assert text == 'The company was founded in 1998 in Seoul. '     # 틀린 문장은 한 글자도 안 나감
    assert 'rockets' not in text


# ---------------- 실제 서버 (시간 간격이 있는 스트리밍) ----------------
# ASGITransport는 응답을 한 번에 모아서 주므로, 토큰 간격이 중요한 테스트는 실제 서버로 한다.
def _free_port():
    import socket
    s = socket.socket(); s.bind(('127.0.0.1', 0)); p = s.getsockname()[1]; s.close(); return p


@pytest.fixture
def live(tmp_path):
    import threading, time as _t, uvicorn
    servers = []

    def start(app):
        port = _free_port()
        srv = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=port, log_level='error'))
        th = threading.Thread(target=srv.run, daemon=True); th.start()
        while not srv.started:
            _t.sleep(0.02)
        servers.append(srv)
        return f'http://127.0.0.1:{port}'

    up = start(mock_upstream.app)
    s = Settings(upstream_base_url=up + '/v1', metrics_path=str(tmp_path / 'm.jsonl'), stride_tokenizer='words')
    gw_url = start(create_app(s, judge=LexicalJudge(threshold=0.5, delay_ms=5)))
    yield gw_url, s
    for srv in servers:
        srv.should_exit = True


async def run_live(url, b):
    """(시각ms, 종류, 값) 이벤트 목록"""
    import time as _t
    out, t0 = [], _t.perf_counter()
    async with httpx.AsyncClient(timeout=30) as c:
        async with c.stream('POST', url + '/v1/chat/completions', json=b) as r:
            async for line in r.aiter_lines():
                if not line.startswith('data:') or line.endswith('[DONE]'):
                    continue
                ch = json.loads(line[5:]); now = (_t.perf_counter() - t0) * 1000
                if ch.get('verification'):
                    out.append((now, 'v', ch['verification']))
                for x in ch['choices']:
                    if x['delta'].get('content'):
                        out.append((now, 't', x['delta']['content']))
                    if x.get('finish_reason'):
                        out.append((now, 'f', x['finish_reason']))
    return out


@pytest.mark.asyncio
async def test_live_pass_cut_stops_after_verdict(live):
    url, _ = live
    b = body(BAD_2ND, 'pass', 'cut'); b['mock_token_ms'] = 20
    ev = await run_live(url, b)
    text = ''.join(v for _, k, v in ev if k == 't')
    assert ('f', 'content_filter') in [(k, v) for _, k, v in ev]
    assert 'rockets' in text and '2023' not in text        # 이미 나간 건 못 막지만 이후는 끊김


@pytest.mark.asyncio
async def test_live_pass_verdict_arrives_without_waiting_next_token(live):
    url, s = live
    b = body(BAD_2ND, 'pass'); b['mock_token_ms'] = 50
    ev = await run_live(url, b)
    rec = json.loads(open(s.metrics_path).read().splitlines()[-1])
    lags = [x['t_verdict'] - x['t_complete'] for x in rec['sentences']]
    assert max(lags) < 40, lags                              # 다음 토큰(50ms)을 기다리지 않고 바로 판정 전달


@pytest.mark.asyncio
async def test_live_hold_delays_but_no_leak(live):
    url, s = live
    for mode in ['off', 'hold']:
        b = body(BAD_2ND, mode); b['mock_token_ms'] = 20
        if mode == 'off':
            b.pop('verification')
        ev = await run_live(url, b)
        text = ''.join(v for _, k, v in ev if k == 't')
        assert text == BAD_2ND
        first = next(t for t, k, _ in ev if k == 't')
        if mode == 'off':
            off_first = first
        else:
            assert first > off_first + 100                   # 첫 문장이 다 만들어질 때까지 보류됨


@pytest.mark.asyncio
async def test_off_is_passthrough(gw):
    client, _ = gw
    b = body(GOOD, 'pass')
    b.pop('verification')
    r = await client.post('/v1/chat/completions', json=b)
    assert r.headers['x-verification-mode'] == 'off'
    assert 'verification' not in r.text


@pytest.mark.asyncio
async def test_non_stream(gw):
    client, _ = gw
    r = await client.post('/v1/chat/completions', json=body(BAD_2ND, 'hold', 'cut', stream=False))
    d = r.json()
    assert d['choices'][0]['finish_reason'] == 'content_filter'
    assert d['choices'][0]['message']['content'] == 'The company was founded in 1998 in Seoul. '
    assert [x['hallucinated'] for x in d['verification']['sentences']] == [False, True]


@pytest.mark.asyncio
async def test_openai_sdk_compatible(gw):
    from openai import AsyncOpenAI
    client, _ = gw
    oai = AsyncOpenAI(base_url='http://gw/v1', api_key='x', http_client=client)
    stream = await oai.chat.completions.create(
        model='mock', messages=[{'role': 'user', 'content': 'q'}], stream=True,
        extra_body={'verification': {'context': REF, 'mode': 'hold'}, 'mock_response': BAD_2ND,
                    'mock_ttft_ms': 0, 'mock_token_ms': 0})
    text, flags = '', []
    async for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            text += chunk.choices[0].delta.content
        v = getattr(chunk, 'verification', None)
        if v:
            flags.append(v['hallucinated'])
    assert text == BAD_2ND and flags == [False, True, False]


# ---------------- 학습된 판정기 로딩 경로 (작은 임의 모델) ----------------
def test_student_judge_loads(tmp_path):
    import torch
    from transformers import BertConfig, BertForSequenceClassification, BertTokenizerFast
    from hallugate.judge import StudentJudge
    words = 'the company was founded in seoul revenue billion dollars reported it'.split()
    (tmp_path / 'vocab.txt').write_text('\n'.join(['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '...', '.'] + words))
    BertTokenizerFast(str(tmp_path / 'vocab.txt')).save_pretrained(tmp_path)
    BertForSequenceClassification(BertConfig(vocab_size=len(words) + 7, hidden_size=16, num_hidden_layers=1,
                                             num_attention_heads=2, intermediate_size=32, num_labels=1)
                                  ).save_pretrained(tmp_path)

    class Emb:
        def encode(self, xs, **k):
            return torch.nn.functional.normalize(torch.randn(len(xs), 8), dim=-1)

    j = StudentJudge(str(tmp_path), embedder=Emb(), k=8, device='cpu')
    ref = j.prepare(REF * 5)
    v = j.judge(ref, 'The company was founded in 1998.')
    assert isinstance(v.score, float) and v.judge_ms > 0


# ---------------- 앞부분 재판정 (stride, SPEC §5.1) ----------------
def test_splitter_pending_tracks_current_sentence():
    sp = IncrementalSplitter()
    sp.feed('First sentence here. Second sen')
    off, buf = sp.pending()
    assert off == len('First sentence here. ') and buf == 'Second sen'


def test_extract_stride():
    body = {'messages': [], 'verification': {'context': 'x', 'stride': 4}}
    assert extract(body, with_stride=True)[4] == 4
    assert extract({'messages': [], 'verification': {'context': 'x'}}, default_stride=2, with_stride=True)[4] == 2
    with pytest.raises(ValueError):
        extract({'messages': [], 'verification': {'context': 'x', 'stride': -1}}, with_stride=True)


@pytest.mark.asyncio
@pytest.mark.parametrize('mode', ['pass', 'hold'])
async def test_stride_text_intact_and_prefix_verdicts(gw, mode):
    client, s = gw
    b = body(BAD_2ND, mode); b['verification']['stride'] = 2
    text, ev, finish = await run(client, b)
    assert text == BAD_2ND and finish == 'stop'                    # 본문 보존 (G2)
    vs = [e for k, e in ev if k == 'v']
    assert {'prefix', 'sentence'} <= {v['kind'] for v in vs}
    assert all(v['start'] < v['upto'] and 'released_at_verdict' in v for v in vs)
    # 두 번째 문장은 앞부분 판정에서 이미 환각으로 잡힘 (문장 판정보다 먼저)
    first_alarm = next(v for v in vs if v['hallucinated'])
    assert first_alarm['kind'] == 'prefix' and first_alarm['sentence_idx'] == 1
    rec = json.loads(open(s.metrics_path).read().splitlines()[-1])
    assert rec['stride'] == 2 and len(rec['checks']) > 0


@pytest.mark.asyncio
async def test_stride_hold_releases_safe_prefix_before_sentence_end(gw):
    client, _ = gw
    b = body(GOOD, 'hold'); b['verification']['stride'] = 2
    text, ev, _ = await run(client, b)
    assert text == GOOD
    # 첫 문장의 문장 판정('sentence') 이전에, 안전한 앞부분이 이미 전달돼야 한다
    first_sent_verdict = next(i for i, (k, e) in enumerate(ev) if k == 'v' and e['kind'] == 'sentence')
    texts_before = [e for k, e in ev[:first_sent_verdict] if k == 't']
    assert texts_before, '문장 판정 전에 아무것도 전달되지 않음'
    # 첫 전달은 문장 전체가 아니라 앞부분이어야 한다 (문장 끝까지 기다리지 않음)
    assert len(texts_before[0]) < len('The company was founded in 1998 in Seoul. ')


@pytest.mark.asyncio
async def test_stride_hold_cut_stops_inside_bad_sentence(gw):
    client, _ = gw
    b = body(BAD_2ND, 'hold', 'cut'); b['verification']['stride'] = 2
    text, ev, finish = await run(client, b)
    assert finish == 'content_filter'
    assert text == 'The company was founded in 1998 in Seoul. '     # 환각 문장은 한 글자도 나가지 않음
    cut = [e for k, e in ev if k == 'v' and e.get('cut')]
    assert cut and cut[0]['kind'] == 'prefix'


@pytest.mark.asyncio
async def test_live_stride_hold_reduces_wait(live):
    # 실제 서버: 앞부분 재판정을 켜면 hold의 첫 글자 지연이 문장 길이보다 짧아진다
    url, _ = live
    firsts = {}
    for stride in [0, 2]:
        b = body(BAD_2ND, 'hold'); b['mock_token_ms'] = 20; b['verification']['stride'] = stride
        ev = await run_live(url, b)
        assert ''.join(v for _, k, v in ev if k == 't') == BAD_2ND
        firsts[stride] = next(t for t, k, _ in ev if k == 't')
    assert firsts[2] < firsts[0] - 60, firsts


# ---------------- S4: 판정기 설정 파일, 토큰 단위 재판정, 통과 청크, fail-open ----------------
def _tiny_bert(path, words):
    from transformers import BertConfig, BertForSequenceClassification, BertTokenizerFast
    (path / 'vocab.txt').write_text('\n'.join(['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '...', '.'] + words))
    BertTokenizerFast(str(path / 'vocab.txt')).save_pretrained(path)
    BertForSequenceClassification(BertConfig(vocab_size=len(words) + 7, hidden_size=16, num_hidden_layers=1,
                                             num_attention_heads=2, intermediate_size=32, num_labels=1)
                                  ).save_pretrained(path)


def test_load_judge_reads_config_full_mode(tmp_path):
    # I2 노트북 형식(input.retrieval = null) → 근거 전체 + 응답 앞부분 입력, 간격별 임계값, 기본 간격 8
    from hallugate.judge import load_judge
    _tiny_bert(tmp_path, 'the company was founded in seoul revenue billion dollars reported it'.split())
    cfg = {'input': {'text_a': 'full reference', 'text_b': 'response prefix', 'retrieval': None}, 'max_len': 64,
           'default': {'kind': '8', 'threshold': 'svc10_orig'}, 'threshold': 1.5,
           'thresholds': {'end': {'svc10_orig': 0.5}, '8': {'svc10_orig': 1.5}, '4': {'svc10_orig': 2.5}}}
    (tmp_path / 'judge_config.json').write_text(json.dumps(cfg))
    j = load_judge(Settings(judge_model_dir=str(tmp_path), judge_device='cpu'))
    assert j.mode == 'full' and j.default_stride == 8 and j.max_len == 64
    assert (j.threshold_for(8), j.threshold_for(4), j.threshold_for(0), j.threshold_for(16)) == (1.5, 2.5, 0.5, 1.5)
    ref = j.prepare(REF)
    assert ref.emb is None and ref.text == REF                       # 근거 검색 없음
    a = j.judge(ref, 'It reported revenue.', prior='The company was founded in 1998 in Seoul. ')
    b = j.judge(ref, 'It reported revenue.')
    assert a.score != b.score                                        # 응답 앞부분(prior)이 입력에 들어감
    # 환경 변수로 임계값을 고정하면 간격별 임계값은 쓰지 않는다
    j2 = load_judge(Settings(judge_model_dir=str(tmp_path), judge_device='cpu', judge_threshold=9.0))
    assert j2.threshold_for(4) == 9.0


def test_word_tokens_cover_text():
    from hallugate.tokens import WordTokens
    t = 'The company  was\nfounded. '
    offs = WordTokens().offsets(t)
    assert ''.join(t[a:b] for a, b in offs) == t and offs[0] == (0, 3)


class RecordingJudge(LexicalJudge):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.calls = []

    def judge(self, ref, text, prior='', threshold=None):
        self.calls.append((text, prior, threshold))
        return super().judge(ref, text, prior, threshold)


def _gw_with(tmp_path, judge, **kw):
    up = httpx.AsyncClient(transport=httpx.ASGITransport(app=mock_upstream.app), base_url='http://mock')
    s = Settings(upstream_base_url='http://mock/v1', metrics_path=str(tmp_path / 'm.jsonl'), stride_tokenizer='words', **kw)
    return httpx.AsyncClient(transport=httpx.ASGITransport(app=create_app(s, judge=judge, http_client=up)),
                             base_url='http://gw'), s


@pytest.mark.asyncio
async def test_token_stride_points_and_prior(tmp_path):
    # 문장 시작부터 토큰 2개마다 앞부분 판정, 판정 텍스트는 토큰 경계에서 끝나고, prior는 그 문장 앞까지의 응답
    j = RecordingJudge(threshold=2.0)                                   # 환각 판정 없음
    client, _ = _gw_with(tmp_path, j)
    b = body(GOOD, 'hold'); b['verification']['stride'] = 2
    text, ev, _ = await run(client, b)
    assert text == GOOD
    prefixes = [(t, p) for t, p, _ in j.calls]
    assert ('The company', '') in prefixes and ('The company was founded', '') in prefixes
    assert ('It reported', 'The company was founded in 1998 in Seoul. ') in prefixes
    assert ('The company was', '') not in prefixes                   # 홀수 토큰 지점은 판정하지 않음


@pytest.mark.asyncio
async def test_default_stride_and_threshold_from_judge(tmp_path):
    class J(RecordingJudge):
        default_stride = 4
        def threshold_for(self, stride, key=None):
            return {0: 0.7, 4: 0.6}[stride]
    j = J()
    client, s = _gw_with(tmp_path, j)
    b = body(GOOD, 'hold'); b['verification'].pop('mode')
    r = await client.post('/v1/chat/completions', json=b)
    assert r.headers['x-verification-mode'] == 'hold' and r.headers['x-verification-stride'] == '4'
    assert {th for _, _, th in j.calls} == {0.6}
    rec = json.loads(open(s.metrics_path).read().splitlines()[-1])
    assert rec['threshold'] == 0.6 and rec['verified'] is True
    h = (await client.get('/health')).json()
    assert h['stride'] == 4 and h['threshold'] == 0.6 and h['mode'] == 'hold'
    r = await client.post('/v1/chat/completions', json=body(GOOD, 'hold', stream=False))
    assert r.json()['verification']['threshold'] == 0.7             # 비스트리밍은 문장 끝 임계값


@pytest.mark.asyncio
@pytest.mark.parametrize('mode', ['pass', 'hold'])
async def test_tool_calls_and_usage_pass_through(gw, mode):
    client, _ = gw
    b = body(GOOD, mode); b['mock_tool_call'] = 'lookup'; b['stream_options'] = {'include_usage': True}
    r = await client.post('/v1/chat/completions', json=b)
    chunks = [json.loads(l[5:]) for l in r.text.splitlines() if l.startswith('data:') and not l.endswith('[DONE]')]
    text = ''.join(c['choices'][0]['delta'].get('content') or '' for c in chunks if c['choices'])
    tools = [c['choices'][0]['delta']['tool_calls'][0] for c in chunks if c['choices'] and c['choices'][0]['delta'].get('tool_calls')]
    finish = [c['choices'][0]['finish_reason'] for c in chunks if c['choices'] and c['choices'][0]['finish_reason']]
    assert text == GOOD
    assert tools[0]['function']['name'] == 'lookup' and tools[1]['function']['arguments'] == '{"q": 1}'
    assert finish == ['tool_calls']
    assert chunks[-1]['usage']['total_tokens'] == 10 and chunks[-1]['choices'] == []   # usage는 맨 끝


@pytest.mark.asyncio
@pytest.mark.parametrize('mode', ['pass', 'hold'])
async def test_fail_open_on_judge_error(tmp_path, mode):
    client, s = _gw_with(tmp_path, LexicalJudge(threshold=0.5, fail=lambda t: 'rockets' in t))
    text, ev, finish = await run(client, body(BAD_2ND, mode, 'cut'))
    assert text == BAD_2ND and finish == 'stop'                      # 본문은 끝까지, 끊지 않음
    vs = [e for k, e in ev if k == 'v']
    un = [v for v in vs if v.get('kind') == 'unverified']
    assert len(un) == 1 and un[0]['verified'] is False and un[0]['reason'] == 'judge_error'
    assert [v['sentence_idx'] for v in vs if v.get('kind') == 'sentence'] == [0]   # 장애 뒤로는 판정 안 함
    rec = json.loads(open(s.metrics_path).read().splitlines()[-1])
    assert rec['verified'] is False and rec['unverified_reason'] == 'judge_error'


@pytest.mark.asyncio
async def test_fail_open_on_timeout(tmp_path):
    client, _ = _gw_with(tmp_path, LexicalJudge(threshold=0.5, delay_ms=300), judge_timeout_ms=50)
    text, ev, _ = await run(client, body(BAD_2ND, 'hold'))
    assert text == BAD_2ND
    assert [e['reason'] for k, e in ev if k == 'v' and e.get('kind') == 'unverified'] == ['judge_timeout']


@pytest.mark.asyncio
async def test_non_english_unverified(gw):
    client, _ = gw
    ko = '이 회사는 1998년 서울에서 설립되었다. 2023년 매출은 20억 달러였다.'
    text, ev, _ = await run(client, body(ko, 'hold'))
    assert text == ko
    vs = [e for k, e in ev if k == 'v']
    assert len(vs) == 1 and vs[0]['reason'] == 'non_english'
    r = await client.post('/v1/chat/completions', json=body(ko, 'hold', stream=False))
    v = r.json()['verification']
    assert v['verified'] is False and v['reason'] == 'non_english' and v['sentences'] == []


# 실제 배포 가중치로 고정 예제 점수 일치(A11). 가중치는 저장소에 없으므로 JUDGE_PARITY_DIR을 지정했을 때만 실행
@pytest.mark.skipif(not os.environ.get('JUDGE_PARITY_DIR'), reason='JUDGE_PARITY_DIR 미지정 (가중치 필요)')
def test_parity_fixture():
    from hallugate.judge import load_judge
    d = os.environ['JUDGE_PARITY_DIR']
    fx = json.load(open(os.path.join(d, 'parity_fixture.json')))
    j = load_judge(Settings(judge_model_dir=d, judge_device='cpu'))
    diffs = [abs(j.judge(j.prepare(it['reference']), it['text']).score - it['score_cpu_fp32']) for it in fx['items']]
    assert max(diffs) <= 0.2, max(diffs)


# ---------------- 컬렉션·RAG (SPEC §2.5·2.6, A9) ----------------
class HashEmb:
    """테스트용 임베더: 단어 해시 묶음 벡터 (같은 단어가 많을수록 가깝다)."""
    def encode(self, texts, **kw):
        import numpy as np
        out = np.zeros((len(texts), 64), dtype=np.float32)
        for i, t in enumerate(texts):
            for w in t.lower().replace('.', ' ').replace('?', ' ').split():
                out[i, sum(map(ord, w)) % 64] += 1
        return out / np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-9)


DOC_A = 'The company was founded in 1998 in Seoul. It reported revenue of 2 billion dollars in 2023.'
DOC_B = 'Penguins live mostly in the southern hemisphere and cannot fly but swim very well.'


def _store(tmp_path):
    from hallugate.store import SQLiteStore
    return SQLiteStore(str(tmp_path / 'c.sqlite3'), HashEmb(), chunk_words=12, overlap=2)


def test_store_add_search_delete(tmp_path):
    st = _store(tmp_path)
    a = st.add('kb', DOC_A, title='company', doc_id='a')
    st.add('kb', DOC_B, title='penguin', doc_id='b')
    assert a['n_chunks'] >= 2 and [d['doc_id'] for d in st.list('kb')] == ['a', 'b']
    hit = st.search('kb', 'When was the company founded in Seoul?', 2)
    assert hit[0].doc_id == 'a' and hit[0].score >= hit[1].score
    st.add('kb', 'Replaced text about owls.', doc_id='b')                 # 같은 id → 교체
    assert len(st.list('kb')) == 2 and st.list('kb')[1]['n_chunks'] == 1
    assert st.delete('kb', 'a') and not st.delete('kb', 'a')
    assert st.collections() == [{'collection': 'kb', 'n_docs': 1}] and not st.exists('nope')
    with pytest.raises(ValueError):
        st.add('kb', '   ')


class CapJudge(LexicalJudge):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.refs = []

    def prepare(self, reference):
        self.refs.append(reference)
        return super().prepare(reference)


def _rag_gw(tmp_path, judge=None, **kw):
    sent = []

    async def hook(req):
        sent.append(json.loads(req.content))
    up = httpx.AsyncClient(transport=httpx.ASGITransport(app=mock_upstream.app), base_url='http://mock',
                           event_hooks={'request': [hook]})
    s = Settings(upstream_base_url='http://mock/v1', metrics_path=str(tmp_path / 'm.jsonl'), stride_tokenizer='words',
                 collection_top_k=2, **kw)
    st = _store(tmp_path)
    app = create_app(s, judge=judge or CapJudge(threshold=0.5), http_client=up, store=st)
    return httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url='http://gw'), s, st, sent


def rag_body(resp, mode='hold', stream=True, coll='kb', question='When was the company founded and what was its revenue?'):
    return {'model': 'mock', 'stream': stream, 'messages': [{'role': 'system', 'content': 'Be brief.'},
                                                             {'role': 'user', 'content': question}],
            'verification': {'collection': coll, 'mode': mode}, 'mock_response': resp,
            'mock_ttft_ms': 0, 'mock_token_ms': 0}


@pytest.mark.asyncio
async def test_collection_api_auth_and_crud(tmp_path):
    client, *_ = _rag_gw(tmp_path, admin_token='secret')
    url = '/v1/collections/kb/documents'
    assert (await client.post(url, json={'text': DOC_A})).status_code == 401
    H = {'Authorization': 'Bearer secret'}
    r = await client.post(url, json={'documents': [{'text': DOC_A, 'doc_id': 'a'}, {'text': DOC_B, 'title': 'p'}]}, headers=H)
    assert r.status_code == 200 and [d['n_chunks'] > 0 for d in r.json()['data']] == [True, True]
    assert (await client.post(url, json={'text': ''}, headers=H)).status_code == 400
    assert len((await client.get(url, headers=H)).json()['data']) == 2
    assert (await client.get('/v1/collections', headers=H)).json()['data'] == [{'collection': 'kb', 'n_docs': 2}]
    assert (await client.delete(url + '/a', headers=H)).json()['deleted'] is True
    assert (await client.delete(url + '/a', headers=H)).status_code == 404
    h = (await client.get('/health')).json()
    assert h['collections'] and h['collections_auth']


@pytest.mark.asyncio
async def test_collections_disabled(tmp_path):
    up = httpx.AsyncClient(transport=httpx.ASGITransport(app=mock_upstream.app), base_url='http://mock')
    s = Settings(upstream_base_url='http://mock/v1', metrics_path='', stride_tokenizer='words', collection_db='')
    client = httpx.AsyncClient(transport=httpx.ASGITransport(app=create_app(s, judge=LexicalJudge(), http_client=up)),
                               base_url='http://gw')
    assert (await client.get('/v1/collections')).status_code == 503
    assert (await client.post('/v1/chat/completions', json=rag_body(GOOD))).status_code == 503


@pytest.mark.asyncio
@pytest.mark.parametrize('mode', ['pass', 'hold'])
async def test_rag_same_evidence_for_llm_and_judge(tmp_path, mode):
    # A9: LLM 프롬프트에 넣은 근거 = 판정 근거, 본문 보존, 사용한 근거를 본문보다 먼저 알림
    j = CapJudge(threshold=0.5)
    client, s, st, sent = _rag_gw(tmp_path, judge=j)
    st.add('kb', DOC_A, title='company', doc_id='a'); st.add('kb', DOC_B, title='penguin', doc_id='b')
    text, ev, finish = await run(client, rag_body(BAD_2ND, mode))
    assert text == BAD_2ND and finish == 'stop'
    vs = [e for k, e in ev if k == 'v']
    assert ev[0][0] == 'v' and vs[0]['kind'] == 'evidence' and vs[0]['collection'] == 'kb'
    evid = [e['text'] for e in vs[0]['evidence']]
    assert len(evid) == 2 and vs[0]['evidence'][0]['doc_id'] == 'a'
    msgs = sent[-1]['messages']
    assert msgs[0]['role'] == 'system' and all(t in msgs[0]['content'] for t in evid)
    assert msgs[1:] == rag_body(GOOD)['messages']                     # 서비스 메시지는 그대로
    assert 'verification' not in sent[-1]
    assert j.refs[-1] == '\n\n'.join(evid)                            # 판정 근거 = 프롬프트 근거
    sents = [v for v in vs if v.get('kind') == 'sentence']
    assert [v['hallucinated'] for v in sents] == [False, True, False]
    rec = json.loads(open(s.metrics_path).read().splitlines()[-1])
    assert rec['collection'] == 'kb' and rec['n_evidence'] == 2


@pytest.mark.asyncio
async def test_rag_non_stream_and_errors(tmp_path):
    client, _, st, sent = _rag_gw(tmp_path)
    st.add('kb', DOC_A, doc_id='a')
    r = await client.post('/v1/chat/completions', json=rag_body(BAD_2ND, 'hold', stream=False))
    v = r.json()['verification']
    assert v['collection'] == 'kb' and v['evidence'][0]['doc_id'] == 'a' and r.headers['x-verification-evidence'] == '2'
    assert [x['hallucinated'] for x in v['sentences']] == [False, True, False]
    assert (await client.post('/v1/chat/completions', json=rag_body(GOOD, coll='missing'))).status_code == 404
    # 요청에 근거를 직접 담으면 그것이 우선, 컬렉션 검색·프롬프트 삽입 안 함
    b = rag_body(GOOD); b['verification']['context'] = REF
    n = len(sent)
    await run(client, b)
    assert sent[n]['messages'] == b['messages']


# ---------------- 조합 판정기 (SPEC §4 ComboJudge, A13) ----------------
def _combo_dir(tmp_path, thresholds=None):
    words = 'the company was founded in seoul revenue billion dollars reported it'.split()
    member_cfg = {'input': {'retrieval': None}, 'max_len': 64, 'default': {'kind': 'end'}, 'threshold': 99.0}
    for name in ['f', 'i']:
        d = tmp_path / name; d.mkdir()
        _tiny_bert(d, words)
        (d / 'judge_config.json').write_text(json.dumps(member_cfg))
    cfg = {'type': 'combo', 'version': 'test', 'members': {'F': 'f', 'I': str(tmp_path / 'i')},   # 상대·절대 경로 모두
           'standardize': {'F': {'mean': -1.0, 'std': 2.0}, 'I': {'mean': 0.5, 'std': 4.0}},
           'default': {'kind': '8', 'threshold': 'svcB'},
           'thresholds': thresholds if thresholds is not None else {'8': {'svcB': 0.25, 'svc10_orig': 0.4}}, 'threshold': 0.3}
    (tmp_path / 'judge_config.json').write_text(json.dumps(cfg))
    return tmp_path


def test_combo_judge_standardized_mean(tmp_path):
    from hallugate.judge import ComboJudge, load_judge
    d = _combo_dir(tmp_path)
    j = load_judge(Settings(judge_model_dir=str(d), judge_device='cpu'))
    assert isinstance(j, ComboJudge) and j.mode == 'combo' and j.default_stride == 8
    assert (j.threshold_for(8), j.threshold_for(8, 'svc10_orig'), j.threshold_for(0), j.threshold_for(4)) == (0.25, 0.4, 0.3, 0.3)
    jf = load_judge(Settings(judge_model_dir=str(d / 'f'), judge_device='cpu'))
    ji = load_judge(Settings(judge_model_dir=str(d / 'i'), judge_device='cpu'))
    prior, text = 'The company was founded in 1998 in Seoul. ', 'It reported revenue.'
    sf = jf.judge(jf.prepare(REF), text, prior).score
    si = ji.judge(ji.prepare(REF), text, prior).score
    want = ((sf + 1.0) / 2.0 + (si - 0.5) / 4.0) / 2                 # 노트북 combine과 같은 식
    v = j.judge(j.prepare(REF), text, prior)
    assert abs(v.score - want) < 1e-5 and v.judge_ms > 0
    assert v.hallucinated == (v.score >= 0.3)                         # threshold 미지정이면 조합 기본 임계값
    assert j.judge(j.prepare(REF), text, prior, threshold=v.score - 1).hallucinated
    # 하위 판정기의 임계값(99)·간격(end)은 쓰지 않는다. 환경 변수 임계값은 조합에 적용
    j2 = load_judge(Settings(judge_model_dir=str(d), judge_device='cpu', judge_threshold=7.0))
    assert j2.threshold_for(8) == 7.0 and j2.threshold_for(0) == 7.0


def test_combo_judge_missing_member_fails(tmp_path):
    from hallugate.judge import load_judge
    d = _combo_dir(tmp_path)
    cfg = json.loads((d / 'judge_config.json').read_text()); cfg['members']['I'] = 'nope'
    (d / 'judge_config.json').write_text(json.dumps(cfg))
    with pytest.raises(FileNotFoundError):
        load_judge(Settings(judge_model_dir=str(d), judge_device='cpu'))


class _Shift(LexicalJudge):
    """조합 게이트웨이 테스트용: 대체 판정기 점수에 상수를 더한다 (두 하위 판정기 점수를 다르게)."""
    def __init__(self, add, **kw):
        super().__init__(**kw); self.add = add

    def judge(self, ref, text, prior='', threshold=None):
        v = super().judge(ref, text, prior, threshold)
        v.score += self.add
        return v


@pytest.mark.asyncio
@pytest.mark.parametrize('mode', ['pass', 'hold'])
async def test_combo_judge_in_gateway(tmp_path, mode):
    from hallugate.judge import ComboJudge
    j = ComboJudge([('F', _Shift(0.0), 0.0, 1.0), ('I', _Shift(1.0), 1.0, 2.0)], thresholds={}, threshold=0.25)
    up = httpx.AsyncClient(transport=httpx.ASGITransport(app=mock_upstream.app), base_url='http://mock')
    s = Settings(upstream_base_url='http://mock/v1', metrics_path='', stride_tokenizer='words', collection_db='')
    client = httpx.AsyncClient(transport=httpx.ASGITransport(app=create_app(s, judge=j, http_client=up)), base_url='http://gw')
    text, ev, finish = await run(client, body(BAD_2ND, mode))
    assert text == BAD_2ND and finish == 'stop'                       # 본문 보존
    vs = [e for k, e in ev if k == 'v']
    lex = LexicalJudge(); r = lex.prepare(REF)
    for v in vs:                                                      # 조합 점수 = (s/1 + (s+1−1)/2)/2 = 0.75·s
        assert abs(v['score'] - 0.75 * lex.judge(r, v['text']).score) < 1e-6
    assert [v['hallucinated'] for v in vs] == [False, True, False]
    h = (await client.get('/health')).json()
    assert h['judge'] == 'ComboJudge' and h['input_mode'] == 'combo' and h['threshold'] == 0.25


@pytest.mark.asyncio
async def test_combo_member_failure_fails_open(tmp_path):
    from hallugate.judge import ComboJudge
    bad = LexicalJudge(fail=lambda t: 'rockets' in t)
    j = ComboJudge([('F', LexicalJudge(), 0.0, 1.0), ('I', bad, 0.0, 1.0)], thresholds={}, threshold=0.5)
    up = httpx.AsyncClient(transport=httpx.ASGITransport(app=mock_upstream.app), base_url='http://mock')
    s = Settings(upstream_base_url='http://mock/v1', metrics_path='', stride_tokenizer='words', collection_db='')
    client = httpx.AsyncClient(transport=httpx.ASGITransport(app=create_app(s, judge=j, http_client=up)), base_url='http://gw')
    text, ev, _ = await run(client, body(BAD_2ND, 'hold'))
    assert text == BAD_2ND                                            # 하위 판정기 하나가 실패해도 본문은 그대로
    assert any(e.get('verified') is False and e.get('reason') == 'judge_error' for k, e in ev if k == 'v')


def test_combo_config_v1_matches_analysis():
    # 저장소의 배포 조합 설정이 실험 결과 파일(I3 표준화 상수, K2 서비스 임계값 B)과 같은지
    root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    if not os.path.exists(os.path.join(root, 'configs', 'judge_combo_v1.json')):
        pytest.skip('저장소 설정·분석 파일이 없는 환경(S3 노트북)')
    cfg = json.load(open(os.path.join(root, 'configs', 'judge_combo_v1.json')))
    i3 = json.load(open(os.path.join(root, 'docs', 'analysis', 'i3', 'i3_summary.json')))['standardize']
    k2 = json.load(open(os.path.join(root, 'docs', 'analysis', 'k2', 'k2b', 'k2_summary.json')))['thr']['old']
    assert cfg['type'] == 'combo' and set(cfg['members']) == set(cfg['standardize']) == {'F', 'I'}
    assert cfg['standardize']['F'] == i3['Fb'] and cfg['standardize']['I'] == i3['I2']
    assert cfg['thresholds']['8'] == {'svcB': k2['B'], 'svc10_orig': k2['A']} and cfg['threshold'] == k2['B']
    assert cfg['default'] == {'kind': '8', 'threshold': 'svcB'}


# ---------------- 응답 끝 2단 검증 (SPEC §5.3 post_check, A14) ----------------
import asyncio                                                    # noqa: E402
import re as _re                                                  # noqa: E402
from types import SimpleNamespace                                 # noqa: E402


class FakeClaude:
    """anthropic.AsyncAnthropic 대역: messages.create(**params) → 응답 객체. fn(params, k)가 텍스트를 만든다."""
    def __init__(self, fn, delay=0.0):
        self.calls, self.fn, self.delay = [], fn, delay
        self.messages = self

    async def create(self, **p):
        k = len(self.calls); self.calls.append(p)
        if self.delay:
            await asyncio.sleep(self.delay)
        return SimpleNamespace(content=[SimpleNamespace(type='text', text=self.fn(p, k))], stop_reason='end_turn',
                               usage=SimpleNamespace(input_tokens=100, output_tokens=20))


def rocket_labels(p, k):
    # 번호 붙은 문장 중 'rockets'가 든 문장만 H
    lines = _re.findall(r'^\[(\d+)\] (.*)$', p['messages'][0]['content'], _re.M)
    return json.dumps({'sentences': [{'i': int(i), 'label': 'H' if 'rockets' in t else 'S',
                                      **({'phrase': 'flew rockets', 'reason': 'not in reference'} if 'rockets' in t else {})}
                                     for i, t in lines]})


def _pc_client(tmp_path, checker=None, **kw):
    up = httpx.AsyncClient(transport=httpx.ASGITransport(app=mock_upstream.app), base_url='http://mock')
    s = Settings(upstream_base_url='http://mock/v1', metrics_path=str(tmp_path / 'm.jsonl'), stride_tokenizer='words',
                 collection_db='', anthropic_api_key='', **kw)
    app = create_app(s, judge=LexicalJudge(threshold=0.5), http_client=up, post_checker=checker)
    return httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url='http://gw'), s


async def run_chunks(client, b):
    r = await client.post('/v1/chat/completions', json=b)
    assert r.status_code == 200, r.text
    out = []
    for line in r.text.splitlines():
        if line.startswith('data:'):
            out.append('DONE' if line.endswith('[DONE]') else json.loads(line[5:]))
    return r, out


def _pc_body(post_check=True, mode='hold', action='flag', stream=True):
    b = body(BAD_2ND, mode, action, stream)
    if post_check is not None:
        b['verification']['post_check'] = post_check
    return b


def _post(out):
    return [c['verification'] for c in out if c != 'DONE' and (c.get('verification') or {}).get('kind') == 'post_check']


@pytest.mark.asyncio
@pytest.mark.parametrize('mode', ['pass', 'hold'])
async def test_post_check_stream(tmp_path, mode):
    from hallugate.postcheck import ClaudePostChecker
    fc = FakeClaude(rocket_labels)
    client, s = _pc_client(tmp_path, ClaudePostChecker(client=fc, model='claude-sonnet-5-5', votes=3, threshold=0.5))
    r, out = await run_chunks(client, _pc_body(mode=mode))
    assert r.headers['x-verification-post-check'] == 'on'
    text = ''.join(ch['delta'].get('content') or '' for c in out if c != 'DONE' for ch in c['choices'])
    assert text == BAD_2ND                                            # 본문 보존
    # 순서: ... 종료 청크 → post_check 청크 → [DONE]
    i_fin = next(i for i, c in enumerate(out) if c != 'DONE' and c['choices'] and c['choices'][0].get('finish_reason'))
    assert (out[i_fin + 1].get('verification') or {}).get('kind') == 'post_check' and out[i_fin + 2] == 'DONE'
    pc = _post(out)
    assert len(pc) == 1
    pc = pc[0]
    assert pc['verified'] and pc['flagged'] and pc['score'] == 1.0 and pc['votes'] == pc['valid_votes'] == 3
    assert pc['model'] == 'claude-sonnet-5-5' and pc['threshold'] == 0.5 and pc['check_ms'] >= 0
    flagged = [x for x in pc['sentences'] if x['flagged']]
    assert [x['sentence_idx'] for x in flagged] == [1] and flagged[0]['h_votes'] == 3 and flagged[0]['phrase'] == 'flew rockets'
    assert len(fc.calls) == 3 and all(c['model'] == 'claude-sonnet-5-5' for c in fc.calls)
    assert REF in fc.calls[0]['messages'][0]['content'] and '[2] Its chief executive flew rockets' in fc.calls[0]['messages'][0]['content']
    rec = json.loads(open(s.metrics_path).read().splitlines()[-1])
    assert rec['post_check']['flagged'] is True


@pytest.mark.asyncio
async def test_post_check_off_by_default(tmp_path):
    from hallugate.postcheck import ClaudePostChecker
    fc = FakeClaude(rocket_labels)
    client, _ = _pc_client(tmp_path, ClaudePostChecker(client=fc))
    r, out = await run_chunks(client, _pc_body(post_check=None))
    assert _post(out) == [] and fc.calls == [] and r.headers['x-verification-post-check'] == 'off'
    # 서버 기본값을 켜면 요청에 없어도 한다
    client, _ = _pc_client(tmp_path, ClaudePostChecker(client=fc), post_check_default=True)
    _, out = await run_chunks(client, _pc_body(post_check=None))
    assert len(_post(out)) == 1 and len(fc.calls) == 3


@pytest.mark.asyncio
async def test_post_check_votes_and_parse_failure(tmp_path):
    from hallugate.postcheck import ClaudePostChecker
    # 판정 3번: H, 파싱 실패, 모두 S → 응답 점수 1/3 < 0.5 → 표시 안 함. 문장은 유효 판정 2개 중 1개 H = 0.5 → 표시
    def fn(p, k):
        if k == 1:
            return 'not json'
        return rocket_labels(p, k) if k == 0 else rocket_labels(p, k).replace('"H"', '"S"')
    client, _ = _pc_client(tmp_path, ClaudePostChecker(client=FakeClaude(fn), votes=3, threshold=0.5))
    _, out = await run_chunks(client, _pc_body())
    pc = _post(out)[0]
    assert pc['verified'] and pc['valid_votes'] == 2 and abs(pc['score'] - 1 / 3) < 1e-9 and not pc['flagged']
    assert [x['sentence_idx'] for x in pc['sentences'] if x['flagged']] == [1]


@pytest.mark.asyncio
@pytest.mark.parametrize('case', ['error', 'all_bad', 'timeout', 'unavailable', 'cut'])
async def test_post_check_fail_open(tmp_path, case):
    from hallugate.postcheck import ClaudePostChecker
    def boom(p, k):
        raise RuntimeError('api down')
    checker = {'error': ClaudePostChecker(client=FakeClaude(boom)),
               'all_bad': ClaudePostChecker(client=FakeClaude(lambda p, k: 'nope')),
               'timeout': ClaudePostChecker(client=FakeClaude(rocket_labels, delay=0.5)),
               'unavailable': None, 'cut': ClaudePostChecker(client=FakeClaude(rocket_labels))}[case]
    client, _ = _pc_client(tmp_path, checker, post_check_timeout_ms=100.0)
    _, out = await run_chunks(client, _pc_body(action='cut' if case == 'cut' else 'flag'))
    text = ''.join(ch['delta'].get('content') or '' for c in out if c != 'DONE' for ch in c['choices'])
    assert text == ('The company was founded in 1998 in Seoul. ' if case == 'cut' else BAD_2ND)
    pc = _post(out)
    want = {'error': 'error', 'all_bad': 'error', 'timeout': 'timeout', 'unavailable': 'unavailable', 'cut': 'cut'}[case]
    assert len(pc) == 1 and pc[0]['verified'] is False and pc[0]['reason'] == want, pc
    assert out[-1] == 'DONE'


@pytest.mark.asyncio
async def test_post_check_non_stream_and_validation(tmp_path):
    from hallugate.postcheck import ClaudePostChecker
    client, _ = _pc_client(tmp_path, ClaudePostChecker(client=FakeClaude(rocket_labels)))
    r = await client.post('/v1/chat/completions', json=_pc_body(stream=False))
    d = r.json()
    assert d['choices'][0]['message']['content'] == BAD_2ND
    assert d['verification']['post_check']['flagged'] is True
    bad = _pc_body(); bad['verification']['post_check'] = 'yes'
    assert (await client.post('/v1/chat/completions', json=bad)).status_code == 400
    h = (await client.get('/health')).json()['post_check']
    assert h == {'available': True, 'default': False, 'model': 'claude-sonnet-5-5', 'votes': 3, 'threshold': 0.5}


def test_post_check_prompt_matches_j2():
    # 2단 검증 지시문 = J2 노트북 sonnet_H 후보 지시문 (J2 결과를 그대로 옮겨 쓰기 위해)
    from hallugate.postcheck import SYSTEM
    root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    path = os.path.join(root, 'experiments', 'builders', 'build_j2.py')
    if not os.path.exists(path):
        pytest.skip('실험 builder가 없는 환경(S3 노트북)')
    src = open(path, encoding='utf-8').read()
    i = src.index("SYS_SENT = '''") + len("SYS_SENT = '''")
    assert src[i:src.index("'''", i)] == SYSTEM


# ---------------- 데모 화면 (SPEC §2.8) ----------------
@pytest.mark.asyncio
async def test_demo_page(tmp_path):
    up = httpx.AsyncClient(transport=httpx.ASGITransport(app=mock_upstream.app), base_url='http://mock')
    s = Settings(upstream_base_url='http://mock/v1', metrics_path='', collection_db='')
    client = httpx.AsyncClient(transport=httpx.ASGITransport(app=create_app(s, judge=LexicalJudge(), http_client=up)),
                               base_url='http://gw')
    r = await client.get('/demo')
    assert r.status_code == 200 and r.headers['content-type'].startswith('text/html')
    assert '/v1/chat/completions' in r.text and 'post_check' in r.text and 'mock_response' in r.text
    # 근거를 LLM에도 보내기: <context> 태그 + 시스템 지시, 운영 모드 전용 예제 (SPEC §2.8)
    assert 'id="ragmsg"' in r.text and '<context>' in r.text and 'LIVE_PRESETS' in r.text
    assert 'http://' not in r.text.replace('http://www.w3.org', '') and 'https://' not in r.text   # 외부 리소스 없음
    s_off = Settings(upstream_base_url='http://mock/v1', metrics_path='', collection_db='', demo=False)
    off = httpx.AsyncClient(transport=httpx.ASGITransport(app=create_app(s_off, judge=LexicalJudge(), http_client=up)),
                            base_url='http://gw')
    assert (await off.get('/demo')).status_code == 404


# ---------------- 운영 모드: 인증·사용자 분리 (SPEC §10.1·10.2, H1·H2) ----------------
def test_keystore_create_verify_revoke(tmp_path):
    import sqlite3
    from hallugate.keys import KeyStore
    ks = KeyStore(str(tmp_path / 'k.sqlite3'))
    kid, secret = ks.create('alice')
    assert secret.startswith(f'hg_{kid}_') and len(secret) > 30
    t = ks.verify(secret)
    assert t is not None and t.key_id == kid and t.name == 'alice'
    assert ks.verify(None) is None and ks.verify('') is None and ks.verify('hg_nope_x') is None
    assert ks.verify(secret[:-1] + ('a' if secret[-1] != 'a' else 'b')) is None      # 비밀 한 글자만 달라도 거부
    assert ks.verify(f'hg_{kid}') is None and ks.verify('Bearer ' + secret) is None
    rows = ks.list()
    assert [r['key_id'] for r in rows] == [kid] and 'hash' not in rows[0] and rows[0]['revoked'] is None
    dump = '\n'.join(sqlite3.connect(str(tmp_path / 'k.sqlite3')).iterdump())
    assert secret.split('_', 2)[2] not in dump                           # 비밀 원문은 저장하지 않음 (해시만)
    assert ks.revoke(kid) and not ks.revoke('nope')
    assert ks.verify(secret) is None and ks.list()[0]['revoked'] is not None


def _hosted_gw(tmp_path, **kw):
    from hallugate.keys import KeyStore
    sent = []

    async def hook(req):
        sent.append({**json.loads(req.content), '_host': req.url.host, '_auth': req.headers.get('authorization')})
    up = httpx.AsyncClient(transport=httpx.ASGITransport(app=mock_upstream.app), base_url='http://mock',
                           event_hooks={'request': [hook]})
    limiter, checker = kw.pop('limiter', None), kw.pop('post_checker', None)
    kw.setdefault('upstream_allowlist', 'openai=http://up-openai/v1,anthropic=http://up-anth/v1')
    s = Settings(upstream_base_url='http://mock/v1', metrics_path=str(tmp_path / 'm.jsonl'), stride_tokenizer='words',
                 collection_top_k=2, hosted=True, keys_db=str(tmp_path / 'k.sqlite3'), admin_token='adm', **kw)
    ks, st = KeyStore(s.keys_db), _store(tmp_path)
    app = create_app(s, judge=CapJudge(threshold=0.5), http_client=up, store=st, keys=ks, limiter=limiter,
                     post_checker=checker)
    return httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url='http://gw'), s, st, ks, sent


def _K(secret):
    return {'x-hallugate-key': secret}


@pytest.mark.asyncio
async def test_hosted_auth_required(tmp_path):
    client, s, st, ks, sent = _hosted_gw(tmp_path)
    kid, key = ks.create('alice')
    b = body(GOOD, 'hold')
    for h in [{}, _K('hg_bad_bad'), {'Authorization': f'Bearer {key}'}, {'Authorization': 'Bearer adm'}]:
        r = await client.post('/v1/chat/completions', json=b, headers=h)
        assert r.status_code == 401 and r.json()['error']['type'] == 'authentication_error', h
        assert (await client.get('/v1/collections', headers=h)).status_code == 401
        assert (await client.post('/v1/collections/kb/documents', json={'text': DOC_A}, headers=h)).status_code == 401
    assert sent == []                                                  # 인증 실패는 업스트림을 부르지 않음
    text, ev, _ = await run(client.__class__(transport=client._transport, base_url='http://gw', headers=_K(key)), b)
    assert text == GOOD
    ks.revoke(kid)
    assert (await client.post('/v1/chat/completions', json=b, headers=_K(key))).status_code == 401
    h = (await client.get('/health')).json()                           # 운영 모드 health는 내부 값 숨김
    assert h['ok'] and h['hosted'] and 'threshold' not in h and 'judge' not in h


@pytest.mark.asyncio
async def test_hosted_tenant_isolation(tmp_path):
    client, s, st, ks, sent = _hosted_gw(tmp_path)
    _, ka = ks.create('alice'); _, kb = ks.create('bob')
    A, B = _K(ka), _K(kb)
    url = '/v1/collections/kb/documents'
    assert (await client.post(url, json={'text': DOC_A, 'doc_id': 'a'}, headers=A)).status_code == 200
    # B에게는 A의 컬렉션이 없는 것과 같다
    assert (await client.get('/v1/collections', headers=B)).json()['data'] == []
    assert (await client.get(url, headers=B)).status_code == 404
    assert (await client.delete(url + '/a', headers=B)).status_code == 404
    rb = rag_body(GOOD)
    assert (await client.post('/v1/chat/completions', json=rb, headers=B)).status_code == 404
    rb['verification']['collection'] = f'{ks.list()[0]["key_id"]}/kb'                # 내부 이름을 흉내 내도 못 봄
    assert (await client.post('/v1/chat/completions', json=rb, headers=B)).status_code == 404
    # B가 같은 이름으로 만들면 별개 컬렉션
    assert (await client.post(url, json={'text': DOC_B, 'doc_id': 'b'}, headers=B)).status_code == 200
    assert (await client.get('/v1/collections', headers=A)).json()['data'] == [{'collection': 'kb', 'n_docs': 1}]
    assert [d['doc_id'] for d in (await client.get(url, headers=A)).json()['data']] == ['a']
    assert [d['doc_id'] for d in (await client.get(url, headers=B)).json()['data']] == ['b']
    # 채팅 근거도 자기 컬렉션에서만
    cA = client.__class__(transport=client._transport, base_url='http://gw', headers=A)
    _, ev, _ = await run(cA, rag_body(GOOD))
    evi = next(e for k, e in ev if k == 'v' and e.get('kind') == 'evidence')
    assert evi['collection'] == 'kb' and {x['doc_id'] for x in evi['evidence']} == {'a'}
    assert 'penguin' not in sent[-1]['messages'][0]['content'].lower()
    # A가 지워도 B의 문서는 그대로
    assert (await client.delete(url + '/a', headers=A)).json()['deleted'] is True
    assert [d['doc_id'] for d in (await client.get(url, headers=B)).json()['data']] == ['b']


@pytest.mark.asyncio
async def test_hosted_document_limits(tmp_path):
    client, s, st, ks, _ = _hosted_gw(tmp_path, max_docs=2, max_doc_chars=50)
    _, k = ks.create('alice')
    url = '/v1/collections/kb/documents'
    assert (await client.post(url, json={'text': 'x' * 51}, headers=_K(k))).status_code == 413
    assert (await client.post(url, json={'documents': [{'text': 'one doc'}, {'text': 'two doc'}, {'text': 'three'}]},
                              headers=_K(k))).status_code == 413
    assert (await client.post(url, json={'documents': [{'text': 'one doc', 'doc_id': '1'}, {'text': 'two', 'doc_id': '2'}]},
                              headers=_K(k))).status_code == 200
    assert (await client.post('/v1/collections/other/documents', json={'text': 'three'}, headers=_K(k))).status_code == 413
    assert (await client.post(url, json={'text': 'replaced', 'doc_id': '1'}, headers=_K(k))).status_code == 200   # 교체는 개수 안 늘림


def test_keys_cli(tmp_path, capsys):
    import importlib.util
    root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    spec = importlib.util.spec_from_file_location('keys_cli', os.path.join(root, 'scripts', 'keys.py'))
    cli = importlib.util.module_from_spec(spec); spec.loader.exec_module(cli)
    db, cdb = str(tmp_path / 'k.sqlite3'), str(tmp_path / 'c.sqlite3')
    cli.main(['--db', db, 'create', '--name', 'alice'])
    out = capsys.readouterr().out
    key = next(w for w in out.split() if w.startswith('hg_'))
    from hallugate.keys import KeyStore
    kid = KeyStore(db).verify(key).key_id
    st = _store(tmp_path); st.add(f'{kid}/kb', DOC_A); st.add('other/kb', DOC_B); st.close()
    cli.main(['--db', db, 'list']); assert kid in capsys.readouterr().out
    cli.main(['--db', db, '--collections', cdb, 'revoke', kid, '--purge'])
    assert KeyStore(db).verify(key) is None
    from hallugate.store import SQLiteStore
    assert [c['collection'] for c in SQLiteStore(cdb, None).collections()] == ['other/kb']     # 그 사용자 것만 지움
    from hallugate.metrics import MetricsStore
    MetricsStore(db).add({'key_id': kid, 'status': 200}); MetricsStore(db).add({'key_id': 'zz', 'status': 200})
    cli.main(['--db', db, '--collections', cdb, 'revoke', kid, '--purge'])
    assert [r['key_id'] for r in MetricsStore(db).rows(0)] == ['zz']                         # 기록도 그 사용자 것만 지움


# ---------------- 운영 모드: 업스트림 허용 목록·사용량 제한 (SPEC §10.3·10.4, H3·H4) ----------------
def _hc(client, key, **h):
    return client.__class__(transport=client._transport, base_url='http://gw', headers={**_K(key), **h})


@pytest.mark.asyncio
async def test_hosted_upstream_allowlist(tmp_path):
    client, s, st, ks, sent = _hosted_gw(tmp_path, upstream_api_key='SERVER-KEY')
    _, k = ks.create('alice')
    c = _hc(client, k, Authorization='Bearer user-llm-key')
    r = await c.post('/v1/chat/completions', json=body(GOOD, 'hold'))
    assert r.status_code == 200 and r.headers['x-upstream'] == 'openai'
    assert sent[-1]['_host'] == 'up-openai' and sent[-1]['_auth'] == 'Bearer user-llm-key'   # 운영자 키 아님
    r = await _hc(client, k, Authorization='Bearer a', **{'x-upstream': 'anthropic'}).post('/v1/chat/completions', json=body(GOOD, 'hold'))
    assert r.status_code == 200 and r.headers['x-upstream'] == 'anthropic' and sent[-1]['_host'] == 'up-anth'
    n = len(sent)
    for bad in ['http://evil.example/v1', 'OPENAI ', 'mock']:
        r = await _hc(client, k, **{'x-upstream': bad}).post('/v1/chat/completions', json=body(GOOD, 'hold'))
        assert r.status_code == 400, bad
    assert len(sent) == n                                              # 임의 주소는 부르지 않음
    await _hc(client, k).post('/v1/chat/completions', json=body(GOOD, 'hold'))
    assert sent[-1]['_auth'] is None                                    # 사용자 키가 없어도 운영자 키를 넣지 않음


def test_limiter_window_and_concurrency():
    from hallugate.limits import Limiter
    now = [1000.0]
    L = Limiter(now=lambda: now[0])
    assert L.hit('a', 2) is None and L.hit('a', 2) is None
    ra = L.hit('a', 2)
    assert ra is not None and 59 <= ra <= 60 and L.hit('b', 2) is None     # 다른 키는 따로
    now[0] += 30; assert L.hit('a', 2) is not None                      # 거절은 세지 않음, 창 안이면 계속 거절
    now[0] += 31; assert L.hit('a', 2) is None                          # 60초 지나면 다시 허용
    assert L.acquire('a', 1) and not L.acquire('a', 1) and L.acquire('b', 1)
    L.release('a'); assert L.acquire('a', 1)


@pytest.mark.asyncio
async def test_hosted_rate_limit(tmp_path):
    from hallugate.limits import Limiter
    now = [0.0]
    client, s, st, ks, sent = _hosted_gw(tmp_path, rate_rpm=2, limiter=Limiter(now=lambda: now[0]))
    _, ka = ks.create('alice'); _, kb = ks.create('bob'); _, kc = ks.create('carol', limits={'rpm': 1})
    a = _hc(client, ka)
    assert [(await a.post('/v1/chat/completions', json=body(GOOD, 'hold'))).status_code for _ in range(2)] == [200, 200]
    r = await a.post('/v1/chat/completions', json=body(GOOD, 'hold'))
    assert r.status_code == 429 and r.json()['error']['type'] == 'rate_limit_error' and int(r.headers['retry-after']) >= 1
    assert len(sent) == 2                                              # 거절된 요청은 업스트림을 부르지 않음
    assert (await _hc(client, kb).post('/v1/chat/completions', json=body(GOOD, 'hold'))).status_code == 200
    c = _hc(client, kc)                                                # 키별 한도가 기본값보다 우선
    assert [(await c.post('/v1/chat/completions', json=body(GOOD, 'hold'))).status_code for _ in range(2)] == [200, 429]
    now[0] += 61
    assert (await a.post('/v1/chat/completions', json=body(GOOD, 'hold'))).status_code == 200


@pytest.mark.asyncio
async def test_hosted_concurrency_and_size_limits(tmp_path):
    client, s, st, ks, sent = _hosted_gw(tmp_path, max_concurrent=1, max_ref_chars=200)
    _, k = ks.create('alice')
    c = _hc(client, k)
    slow = body(GOOD, 'hold'); slow['mock_token_ms'] = 20
    r1, r2 = await asyncio.gather(c.post('/v1/chat/completions', json=slow), c.post('/v1/chat/completions', json=slow))
    assert sorted([r1.status_code, r2.status_code]) == [200, 429]
    assert (await c.post('/v1/chat/completions', json=body(GOOD, 'hold'))).status_code == 200      # 끝나면 다시 허용
    assert (await c.post('/v1/chat/completions', json=rag_body(GOOD, coll='missing'))).status_code == 404
    assert (await c.post('/v1/chat/completions', json=body(GOOD, 'hold', stream=False))).status_code == 200   # 오류 뒤에도 풀림
    big = body(GOOD, 'hold'); big['verification']['context'] = 'word ' * 50
    n = len(sent)
    r = await c.post('/v1/chat/completions', json=big)
    assert r.status_code == 413 and len(sent) == n


@pytest.mark.asyncio
async def test_hosted_post_check_daily_quota(tmp_path):
    from hallugate.keys import KeyStore
    from hallugate.postcheck import ClaudePostChecker
    fc = FakeClaude(rocket_labels)
    client, s, st, ks, sent = _hosted_gw(tmp_path, post_check_daily=1, post_checker=ClaudePostChecker(client=fc, votes=3))
    _, k = ks.create('alice')
    c = _hc(client, k)
    _, out = await run_chunks(c, _pc_body())
    assert _post(out)[0]['verified'] is True and len(fc.calls) == 3
    _, out = await run_chunks(c, _pc_body())
    text = ''.join(ch['delta'].get('content') or '' for x in out if x != 'DONE' for ch in x['choices'])
    assert text == BAD_2ND and _post(out)[0] == {'kind': 'post_check', 'verified': False, 'reason': 'quota'}
    assert len(fc.calls) == 3                                          # 한도를 넘으면 Claude를 부르지 않음
    kid = ks.list()[0]['key_id']
    assert KeyStore(s.keys_db).post_checks_today(kid) == 1             # 컨테이너가 다시 떠도 유지


# ---------------- 운영 모드: 원문 없는 기록·관리자 통계 (SPEC §10.5·10.6, H5·H7) ----------------
def _rows(path):
    import sqlite3
    return [json.loads(r[0]) for r in sqlite3.connect(path).execute('SELECT data FROM requests ORDER BY ts')]


@pytest.mark.asyncio
async def test_hosted_records_have_no_raw_text(tmp_path, capsys):
    from hallugate.postcheck import ClaudePostChecker
    client, s, st, ks, sent = _hosted_gw(tmp_path, post_checker=ClaudePostChecker(client=FakeClaude(rocket_labels)))
    _, k = ks.create('alice')
    c = _hc(client, k, Authorization='Bearer user-llm-key-123')
    QUESTION = 'Which secret question zebra?'
    b = _pc_body(); b['messages'][0]['content'] = QUESTION
    _, out = await run_chunks(c, b)                                        # 스트리밍 + 2단
    assert _post(out)[0]['flagged']
    await c.post('/v1/chat/completions', json=body(BAD_2ND, 'hold', stream=False))          # 비스트리밍
    await c.post('/v1/collections/kb/documents', json={'text': DOC_A, 'doc_id': 'a'})
    await run(c, rag_body(BAD_2ND))                                        # 컬렉션
    await client.post('/v1/chat/completions', json=b)                      # 401 (키 없음)
    off = body(GOOD, 'off'); await c.post('/v1/chat/completions', json=off)   # 검증 끔 → 그대로 통과도 기록
    rows = _rows(s.keys_db)
    assert [r['status'] for r in rows] == [200, 200, 200, 401, 200]
    assert rows[0]['stream'] and rows[0]['post_check']['flagged'] and rows[0]['n_flagged'] == 1 and rows[0]['n_sentences'] == 3
    assert rows[2]['collection_used'] and rows[2]['n_evidence'] >= 1 and rows[3]['key_id'] == ''
    dump = json.dumps(rows) + capsys.readouterr().out
    for secret in ['rockets', 'founded in 1998', 'zebra', 'Penguins', 'user-llm-key-123', k, k.split('_', 2)[2],
                   'not in reference', 'flew rockets', 'Be brief']:
        assert secret not in dump, secret                                 # 원문·키가 기록에 없음
    assert dump.count('"key_id"') >= 2 and not os.path.exists(s.metrics_path)   # 운영 모드는 JSONL을 쓰지 않음


def test_metrics_store_retention_and_purge(tmp_path):
    import time as _t
    from hallugate.metrics import MetricsStore
    m = MetricsStore(str(tmp_path / 'k.sqlite3'))
    m.add({'key_id': 'a', 'status': 200}, ts=_t.time() - 40 * 86400)
    m.add({'key_id': 'a', 'status': 200}); m.add({'key_id': 'b', 'status': 200})
    assert m.prune(30) == 1 and len(m.rows(0)) == 2
    assert m.purge('a') == 1 and [r['key_id'] for r in m.rows(0)] == ['b']


@pytest.mark.asyncio
async def test_admin_stats(tmp_path):
    from hallugate.postcheck import ClaudePostChecker
    client, s, st, ks, sent = _hosted_gw(tmp_path, rate_rpm=3, post_checker=ClaudePostChecker(client=FakeClaude(rocket_labels)))
    _, ka = ks.create('alice'); kid = ks.list()[0]['key_id']
    a = _hc(client, ka)
    await run_chunks(a, _pc_body())
    await a.post('/v1/chat/completions', json=body(GOOD, 'hold'))
    await a.post('/v1/chat/completions', json=body(GOOD, 'hold'))
    assert (await a.post('/v1/chat/completions', json=body(GOOD, 'hold'))).status_code == 429
    await client.post('/v1/chat/completions', json=body(GOOD, 'hold'))     # 401
    assert (await client.get('/admin/stats')).status_code == 401
    assert (await client.get('/admin/stats', headers={'Authorization': 'Bearer nope'})).status_code == 401
    assert (await client.get('/admin/stats?window=abc', headers={'Authorization': 'Bearer adm'})).status_code == 400
    r = await client.get('/admin/stats?window=24h', headers={'Authorization': 'Bearer adm'})
    st_ = r.json()
    assert st_['window'] == '24h' and st_['requests']['total'] == 5
    assert st_['requests']['by_status'] == {'200': 3, '401': 1, '429': 1} and st_['rate_limited'] == 1
    assert st_['requests']['by_key'][kid] == 4 and st_['requests']['by_upstream']['openai'] == 4
    for k_ in ['ttft_ms', 'total_ms']:
        assert st_['latency'][k_]['p50'] is not None and st_['latency'][k_]['p95'] >= st_['latency'][k_]['p50']
    assert st_['latency']['judge_ms_p95'] is not None
    assert st_['unverified']['rate'] == 0.0 and st_['post_check']['runs'] == 1 and st_['post_check']['flagged'] == 1
    assert st_['post_check']['est_cost_usd'] > 0 and st_['cold_starts'] == []
    rows = _rows(s.keys_db)
    assert rows[0]['cold'] is True and all(not x.get('cold') for x in rows[1:])    # 컨테이너 시작 직후 첫 요청만 콜드
    m = client._transport.app if hasattr(client._transport, 'app') else None
    from hallugate.metrics import MetricsStore
    MetricsStore(s.keys_db).add_start(1234.5)
    r = await client.get('/admin/stats?window=30m', headers={'Authorization': 'Bearer adm'})
    cs = r.json()['cold_starts']
    assert len(cs) == 1 and cs[0]['warmup_ms'] == 1234.5


# ---------------- 운영 모드: 관리자 키 API (SPEC §10.1) ----------------
@pytest.mark.asyncio
async def test_admin_keys_api(tmp_path):
    client, s, st, ks, sent = _hosted_gw(tmp_path)
    A = {'Authorization': 'Bearer adm'}
    assert (await client.post('/admin/keys', json={'name': 'x'})).status_code == 401
    assert (await client.post('/admin/keys', json={'name': 'x'}, headers={'Authorization': 'Bearer no'})).status_code == 401
    r = (await client.post('/admin/keys', json={'name': 'alice', 'limits': {'rpm': 5}}, headers=A)).json()
    kid, key = r['key_id'], r['key']
    assert key.startswith(f'hg_{kid}_') and ks.verify(key).limits == {'rpm': 5}
    lst = (await client.get('/admin/keys', headers=A)).json()['data']
    assert [x['key_id'] for x in lst] == [kid] and 'hash' not in lst[0] and key not in json.dumps(lst)
    assert (await client.post('/admin/keys', json={'limits': 'bad'}, headers=A)).status_code == 400
    # 발급한 키로 바로 사용 → 폐기(+purge) → 401, 문서·기록 삭제
    c = _hc(client, key)
    await c.post('/v1/collections/kb/documents', json={'text': DOC_A})
    assert (await c.post('/v1/chat/completions', json=body(GOOD, 'hold'))).status_code == 200
    r = (await client.post(f'/admin/keys/{kid}/revoke', json={'purge': True}, headers=A)).json()
    assert r['revoked'] and r['purged_docs'] == 1 and r['purged_records'] >= 1
    assert (await c.post('/v1/chat/completions', json=body(GOOD, 'hold'))).status_code == 401
    assert (await client.post('/admin/keys/nope/revoke', json={}, headers=A)).json()['revoked'] is False


def test_keys_cli_remote(tmp_path, capsys, monkeypatch):
    # scripts/keys.py --url: 관리자 API 호출 (ASGI 앱에 바로 연결)
    import importlib.util
    root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    spec = importlib.util.spec_from_file_location('keys_cli2', os.path.join(root, 'scripts', 'keys.py'))
    cli = importlib.util.module_from_spec(spec); spec.loader.exec_module(cli)
    client, s, st, ks, sent = _hosted_gw(tmp_path)
    monkeypatch.setenv('ADMIN_TOKEN', 'adm')

    class Bridge(httpx.BaseTransport):                                  # 동기 httpx → 비동기 ASGI 앱
        def handle_request(self, request):
            async def go():
                r = await client._transport.handle_async_request(request)
                body_ = b''.join([c async for c in r.stream])
                return httpx.Response(r.status_code, headers=r.headers, content=body_)
            return asyncio.new_event_loop().run_until_complete(go())
    monkeypatch.setattr(cli, 'TRANSPORT', Bridge())
    cli.main(['--url', 'http://gw', 'create', '--name', 'bob', '--limits', '{"rpm": 3}'])
    key = next(w for w in capsys.readouterr().out.split() if w.startswith('hg_'))
    t = ks.verify(key); assert t.name == 'bob' and t.limits == {'rpm': 3}
    cli.main(['--url', 'http://gw', 'list']); assert t.key_id in capsys.readouterr().out
    cli.main(['--url', 'http://gw', 'revoke', t.key_id, '--purge']); assert ks.verify(key) is None


# ---------------- 운영 모드: 사용자 LLM 사용량·예산 (SPEC §10.10, H9) ----------------
def test_prices_lookup():
    from hallugate.prices import cost_usd, price_for
    assert price_for('gpt-4o-mini-2024-07-18') == price_for('gpt-4o-mini')          # 날짜 붙은 이름은 접두어로
    assert price_for('gpt-4o-2024-08-06') != price_for('gpt-4o-mini')               # 가장 긴 접두어
    assert price_for('unknown-model') is None and cost_usd('unknown-model', 10, 10) is None
    pin, pout = price_for('gpt-4o-mini')
    assert abs(cost_usd('gpt-4o-mini', 1_000_000, 2_000_000) - (pin + 2 * pout)) < 1e-9


@pytest.mark.asyncio
async def test_hosted_usage_collected_without_changing_output(tmp_path):
    client, s, st, ks, sent = _hosted_gw(tmp_path)
    _, k = ks.create('alice')
    c = _hc(client, k)
    b = body(GOOD, 'hold'); b['model'] = 'gpt-4o-mini'
    _, out = await run_chunks(c, b)                                     # 사용자는 usage를 요청하지 않음
    assert sent[-1]['stream_options'] == {'include_usage': True}       # 업스트림에는 요청
    assert not any(x != 'DONE' and x.get('usage') for x in out)        # 사용자에게는 보내지 않음
    b2 = dict(b, stream_options={'include_usage': True})
    _, out2 = await run_chunks(c, b2)
    assert any(x != 'DONE' and x.get('usage') for x in out2)           # 요청하면 그대로 보냄
    await c.post('/v1/chat/completions', json=body(GOOD, 'hold', stream=False) | {'model': 'gpt-4o-mini'})
    r, out3 = await run_chunks(c, body(GOOD, 'off') | {'model': 'gpt-4o-mini'})      # 검증 끔
    assert not any(x != 'DONE' and x.get('usage') for x in out3)
    rows = _rows(s.keys_db)
    assert [(x['tokens_in'], x['tokens_out']) for x in rows] == [(3, 7)] * 4
    assert all(x['model'] == 'gpt-4o-mini' and x['llm_cost_usd'] > 0 for x in rows)
    u = (await c.get('/v1/usage?window=30d')).json()
    assert u['requests'] == 4 and u['tokens_in'] == 12 and u['tokens_out'] == 28
    assert u['by_model']['gpt-4o-mini']['requests'] == 4 and u['est_cost_usd'] > 0
    assert (await client.get('/v1/usage')).status_code == 401
    st_ = (await client.get('/admin/stats', headers={'Authorization': 'Bearer adm'})).json()
    assert st_['llm']['tokens_in'] == 12 and st_['llm']['by_model']['gpt-4o-mini']['tokens_out'] == 28


@pytest.mark.asyncio
@pytest.mark.parametrize('action', ['warn', 'block'])
async def test_hosted_budget(tmp_path, action):
    client, s, st, ks, sent = _hosted_gw(tmp_path)
    _, k = ks.create('alice', limits={'monthly_tokens': 15, 'budget_action': action})
    c = _hc(client, k)
    r1 = await c.post('/v1/chat/completions', json=body(GOOD, 'hold'))           # 요청마다 10토큰
    assert r1.headers['x-budget-used'] == '0.000' and 'x-budget-warning' not in r1.headers
    r2 = await c.post('/v1/chat/completions', json=body(GOOD, 'hold'))
    assert r2.headers['x-budget-used'] == '0.667' and 'x-budget-warning' not in r2.headers
    n = len(sent)
    r3 = await c.post('/v1/chat/completions', json=body(GOOD, 'hold'))
    if action == 'warn':
        assert r3.status_code == 200 and r3.headers['x-budget-warning'] == 'exceeded' and len(sent) == n + 1
    else:
        assert r3.status_code == 429 and r3.json()['error']['type'] == 'insufficient_quota' and len(sent) == n
    u = (await c.get('/v1/usage')).json()['budget']
    assert u['monthly_tokens'] == 15 and u['tokens_used'] == (30 if action == 'warn' else 20) and u['action'] == action
    # 예산 없는 키는 헤더만 없음
    _, k2 = ks.create('bob')
    r = await _hc(client, k2).post('/v1/chat/completions', json=body(GOOD, 'hold'))
    assert r.status_code == 200 and 'x-budget-used' not in r.headers


@pytest.mark.asyncio
async def test_admin_key_limits_accept_budget(tmp_path):
    client, *_ = _hosted_gw(tmp_path)
    A = {'Authorization': 'Bearer adm'}
    ok = await client.post('/admin/keys', json={'limits': {'monthly_usd': 5.0, 'budget_action': 'block', 'rpm': 10}}, headers=A)
    assert ok.status_code == 200
    bad = await client.post('/admin/keys', json={'limits': {'budget_action': 'nope'}}, headers=A)
    assert bad.status_code == 400


# ---------------- 근거 다이어트 (SPEC §2.9, A15) ----------------
FILLER = [f'Unrelated note {i}: the weather station recorded rain and wind on day {i} near the old harbor bridge.'
          for i in range(12)]
LONG_REF = ' '.join(FILLER[:6]) + ' ' + DOC_A + ' ' + ' '.join(FILLER[6:])


def test_trimmer_selects_relevant_in_order():
    from hallugate.trim import Trimmer
    t = Trimmer(HashEmb(), chunk_words=12, overlap=2)
    kept, total = t.select(LONG_REF, 'When was the company founded and what was its revenue?', 2)
    assert total > 2 and len(kept) == 2
    assert any('founded in 1998' in c for c in kept) or any('revenue' in c for c in kept)
    pos = [LONG_REF.find(c[:30]) for c in kept]
    assert pos == sorted(pos)                                          # 원문 순서
    kept2, total2 = t.select('short text only', 'q', 4)
    assert kept2 == ['short text only'] and total2 == 1


def _trim_body(resp, src='field', trim=True, stream=True):
    q = 'When was the company founded and what was its revenue?'
    if src == 'field':
        msgs = [{'role': 'user', 'content': q}]
        ver = {'context': LONG_REF, 'mode': 'hold', 'trim': trim}
    else:
        msgs = [{'role': 'system', 'content': 'Be brief.'}, {'role': 'user', 'content': f'<context>{LONG_REF}</context>\n{q}'}]
        ver = {'mode': 'hold', 'trim': trim}
    return {'model': 'mock', 'stream': stream, 'messages': msgs, 'verification': ver,
            'mock_response': resp, 'mock_ttft_ms': 0, 'mock_token_ms': 0}


@pytest.mark.asyncio
@pytest.mark.parametrize('src', ['field', 'tag'])
async def test_trim_sends_only_selected_chunks(tmp_path, src):
    j = CapJudge(threshold=0.5)
    client, s, st, sent = _rag_gw(tmp_path, judge=j, collection_chunk_words=12, collection_overlap=2)
    r = await client.post('/v1/chat/completions', json=_trim_body(GOOD, src, {'top_k': 2}))
    assert r.status_code == 200 and r.headers['x-verification-trim'].startswith('2/')
    chunks = [json.loads(l[5:]) for l in r.text.splitlines() if l.startswith('data:') and not l.endswith('[DONE]')]
    text = ''.join(ch['delta'].get('content') or '' for c in chunks for ch in c['choices'])
    assert text == GOOD                                                # 본문 보존
    info = next(c['verification'] for c in chunks if (c.get('verification') or {}).get('kind') == 'trim')
    assert info['chunks_kept'] == 2 and info['chunks_total'] > 2 and 0 < info['chars_after'] < info['chars_before']
    assert 0 < info['saved_ratio'] < 1 and info['top_k'] == 2
    msgs = sent[-1]['messages']
    sent_text = json.dumps(msgs)
    assert 'Unrelated note 0:' not in sent_text or 'Unrelated note 11:' not in sent_text   # 다 보내지 않음
    ref = j.refs[-1]
    assert len(ref) == info['chars_after']                             # 판정 근거 = 고른 조각
    if src == 'field':
        assert msgs[0]['role'] == 'system' and all(p_ in msgs[0]['content'] for p_ in ref.split('\n\n'))
        assert msgs[1:] == _trim_body(GOOD, src)['messages']
    else:
        assert msgs[0] == {'role': 'system', 'content': 'Be brief.'}
        assert msgs[1]['content'] == f'<context>{ref}</context>\nWhen was the company founded and what was its revenue?'


@pytest.mark.asyncio
async def test_trim_noop_cases_and_validation(tmp_path):
    j = CapJudge(threshold=0.5)
    client, s, st, sent = _rag_gw(tmp_path, judge=j, collection_chunk_words=12, collection_overlap=2)
    b = _trim_body(GOOD, 'tag', trim=False)                             # 끄면 그대로
    await client.post('/v1/chat/completions', json=b)
    assert sent[-1]['messages'] == b['messages'] and j.refs[-1] == LONG_REF
    b = _trim_body(GOOD, 'tag', {'top_k': 32})                          # 조각 수 이하면 그대로
    r = await client.post('/v1/chat/completions', json=b)
    assert sent[-1]['messages'] == b['messages'] and r.headers['x-verification-trim'].split('/')[0] == r.headers['x-verification-trim'].split('/')[1]
    nb = _trim_body(GOOD, 'field', stream=False)                       # 비스트리밍
    v = (await client.post('/v1/chat/completions', json=nb)).json()['verification']
    assert v['trim']['top_k'] == s.trim_top_k == 8 and Settings().trim_top_k == 8   # 기본 조각 수 = TRIM_TOP_K (T1c 채택값)
    assert v['trim']['chunks_kept'] == min(8, v['trim']['chunks_total'])
    st.add('kb', DOC_A, doc_id='a')                                     # 컬렉션 요청에는 적용 안 함
    rb = rag_body(GOOD); rb['verification']['trim'] = True
    r = await client.post('/v1/chat/completions', json=rb)
    assert r.status_code == 200 and 'x-verification-trim' not in r.headers
    for bad in ['yes', {'top_k': 0}, {'top_k': 99}, {'k': 2}]:
        b = _trim_body(GOOD); b['verification']['trim'] = bad
        assert (await client.post('/v1/chat/completions', json=b)).status_code == 400, bad


@pytest.mark.asyncio
async def test_hosted_trim_recorded_without_text(tmp_path):
    client, s, st, ks, sent = _hosted_gw(tmp_path, collection_chunk_words=12, collection_overlap=2)
    _, k = ks.create('alice')
    await _hc(client, k).post('/v1/chat/completions', json=_trim_body(GOOD, 'field', {'top_k': 2}))
    row = _rows(s.keys_db)[-1]
    assert row['trim_chunks_kept'] == 2 and row['trim_chars_after'] < row['trim_chars_before'] == len(LONG_REF)
    assert 'harbor' not in json.dumps(row)


# ---------------- 쓸 때만 켜기 (SPEC §10.7) ----------------
class _FakeWarm:
    """scripts/warm.py의 Modal 연결을 대신한다: 최소 컨테이너 수와 Dict를 메모리에 둔다."""
    def __init__(self):
        self.min, self.state, self.health = 0, {}, []

    def set_min(self, n):
        self.min = n

    def get(self, k):
        return self.state.get(k)

    def put(self, k, v):
        self.state[k] = v

    def pop(self, k):
        self.state.pop(k, None)

    def ping(self, url):
        self.health.append(url)
        return True


def _warm_cli():
    import importlib.util
    root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    spec = importlib.util.spec_from_file_location('warm_cli', os.path.join(root, 'scripts', 'warm.py'))
    cli = importlib.util.module_from_spec(spec); spec.loader.exec_module(cli)
    return cli


def test_warm_up_down_status(capsys):
    cli = _warm_cli(); b = _FakeWarm(); cli.BACKEND = b; cli.NOW = lambda: 1000.0
    cli.main(['up'])
    assert b.min == 1 and b.state['until'] == 1000.0 + 2 * 3600            # 기본 2시간
    cli.main(['up', '--hours', '0.5'])
    assert b.min == 1 and b.state['until'] == 1000.0 + 1800                 # 다시 켜면 끌 시각만 새로
    capsys.readouterr(); cli.main(['status']); assert '30' in capsys.readouterr().out
    cli.main(['down'])
    assert b.min == 0 and 'until' not in b.state
    for bad in ['0', '-1', '13']:                                           # 0 < h ≤ 12
        with pytest.raises(SystemExit):
            cli.main(['up', '--hours', bad])
    assert b.min == 0
    cli.main(['up', '--wait', '--url', 'https://gw.example'])
    assert b.health == ['https://gw.example']


def test_warm_auto_cooldown():
    cli = _warm_cli(); b = _FakeWarm()
    b.min, b.state = 1, {'until': 5000.0}
    assert cli.cooldown(b, now=4999.0) is False and b.min == 1              # 아직
    assert cli.cooldown(b, now=5001.0) is True and b.min == 0 and 'until' not in b.state
    assert cli.cooldown(b, now=9999.0) is False                              # 켠 적 없으면 그대로


# ---------------- 운영자·사용자 화면 (SPEC §10.11) ----------------
@pytest.mark.asyncio
async def test_verification_trim_totals(tmp_path):
    client, s, st, ks, sent = _hosted_gw(tmp_path, collection_chunk_words=12, collection_overlap=2)
    _, ka = ks.create('alice'); _, kb = ks.create('bob')
    a, b = _hc(client, ka), _hc(client, kb)
    await a.post('/v1/chat/completions', json=body(GOOD, 'hold'))                  # 2문장, 환각 0
    await a.post('/v1/chat/completions', json=body(BAD_2ND, 'hold'))               # 3문장, 환각 1
    await a.post('/v1/chat/completions', json=body(GOOD, 'off'))                   # 검증 끔 → 집계 제외
    await a.post('/v1/chat/completions', json=_trim_body(GOOD, 'field', {'top_k': 2}))
    await b.post('/v1/chat/completions', json=body(BAD_2ND, 'hold'))
    A = {'Authorization': 'Bearer adm'}
    v = (await client.get('/admin/stats?window=1h', headers=A)).json()['verification']
    assert v['responses'] == 4 and v['responses_flagged'] == 2 and v['fully_verified'] == 4
    assert v['sentences_flagged'] == 2 and v['sentences_judged'] >= 10
    assert v['flag_rate'] == pytest.approx(v['sentences_flagged'] / v['sentences_judged'])
    assert v['unverified_by_reason'] == {}
    tr = (await client.get('/admin/stats?window=1h', headers=A)).json()['trim']
    assert tr['requests'] == 1 and tr['chars_after'] < tr['chars_before'] and 0 < tr['saved_ratio'] < 1
    u = (await a.get('/v1/usage?window=1h')).json()                                # 사용자 본인 것만
    assert u['verification']['responses'] == 3 and u['verification']['responses_flagged'] == 1
    assert u['trim']['requests'] == 1
    ub = (await b.get('/v1/usage?window=1h')).json()
    assert ub['verification']['responses'] == 1 and ub['trim'] == {'requests': 0, 'chars_before': 0, 'chars_after': 0,
                                                                     'saved_ratio': None}


@pytest.mark.asyncio
@pytest.mark.parametrize('path,needs', [('/admin', ['/admin/stats', '/admin/keys', 'revoke']),
                                        ('/console', ['/v1/usage', 'x-hallugate-key']),
                                        ('/start', ['x-hallugate-key', 'verification', 'base_url'])])
async def test_hosted_pages(tmp_path, path, needs):
    client, s, st, ks, sent = _hosted_gw(tmp_path)
    r = await client.get(path)
    assert r.status_code == 200 and r.headers['content-type'].startswith('text/html')
    for n in needs:
        assert n in r.text
    assert 'https://' not in r.text and 'http://' not in r.text.replace('http://www.w3.org', '')   # 외부 리소스 없음
    up = httpx.AsyncClient(transport=httpx.ASGITransport(app=mock_upstream.app), base_url='http://mock')
    plain = httpx.AsyncClient(transport=httpx.ASGITransport(app=create_app(
        Settings(upstream_base_url='http://mock/v1', metrics_path='', collection_db=''), judge=LexicalJudge(), http_client=up)),
        base_url='http://gw')
    assert (await plain.get(path)).status_code == 404                              # 운영 모드 전용

In [ ]:
%%writefile gateway/pytest.ini
[pytest]
asyncio_mode = strict

In [ ]:
%%writefile gateway/hallugate/static/demo.html
<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>hallugate 데모</title>
<style>
:root {
  --bg: #f6f7f9; --panel: #ffffff; --ink: #1d2330; --muted: #677085; --line: #e2e5eb;
  --accent: #3157d5; --ok-bg: #e3f5ea; --ok-ink: #17613a; --bad-bg: #fde2e1; --bad-ink: #a4231c;
  --warn: #d9822b; --post: #7b4ad1; --pending: #eef0f4; --code: #f1f3f7;
}
@media (prefers-color-scheme: dark) {
  :root {
    --bg: #12151b; --panel: #1b1f27; --ink: #e6e9ef; --muted: #98a1b3; --line: #2c3240;
    --accent: #7d9bff; --ok-bg: #173826; --ok-ink: #8fe0ad; --bad-bg: #4a1d1b; --bad-ink: #ffb3ad;
    --warn: #f0a35a; --post: #b392f0; --pending: #252a35; --code: #232835;
  }
}
* { box-sizing: border-box; }
body { margin: 0; background: var(--bg); color: var(--ink);
  font: 14px/1.55 -apple-system, BlinkMacSystemFont, "Apple SD Gothic Neo", "Malgun Gothic", "Segoe UI", sans-serif; }
header { display: flex; align-items: baseline; gap: 14px; flex-wrap: wrap; padding: 16px 24px; border-bottom: 1px solid var(--line); background: var(--panel); }
header h1 { margin: 0; font-size: 18px; }
header .sub { color: var(--muted); font-size: 13px; }
#health { margin-left: auto; font-size: 12px; color: var(--muted); }
#health b { color: var(--ink); font-weight: 600; }
main { display: grid; grid-template-columns: minmax(300px, 400px) 1fr; gap: 16px; padding: 16px 24px; max-width: 1400px; margin: 0 auto; }
@media (max-width: 860px) { main { grid-template-columns: 1fr; padding: 12px 16px; } header { padding: 12px 16px; } }
.card { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 14px 16px; }
.card + .card { margin-top: 12px; }
h2 { font-size: 13px; margin: 0 0 10px; color: var(--muted); font-weight: 600; letter-spacing: .02em; }
label { display: block; font-size: 12px; color: var(--muted); margin: 10px 0 4px; }
textarea, input[type=text], input[type=number], input[type=password], select {
  width: 100%; font: inherit; color: var(--ink); background: var(--bg); border: 1px solid var(--line); border-radius: 6px; padding: 7px 9px; }
textarea { resize: vertical; min-height: 64px; }
.row { display: flex; gap: 8px; }
.row > * { flex: 1; min-width: 0; }
.chips { display: flex; gap: 6px; flex-wrap: wrap; }
.chip { border: 1px solid var(--line); background: var(--bg); color: var(--ink); border-radius: 999px; padding: 4px 10px; font: inherit; font-size: 12px; cursor: pointer; }
.chip:hover { border-color: var(--accent); }
.check { display: flex; align-items: center; gap: 6px; font-size: 13px; color: var(--ink); margin-top: 10px; }
.check input { width: auto; }
.btns { display: flex; gap: 8px; margin-top: 14px; }
button.primary { flex: 1; background: var(--accent); color: #fff; border: 0; border-radius: 6px; padding: 9px; font: inherit; font-weight: 600; cursor: pointer; }
button.primary:disabled { opacity: .5; cursor: default; }
button.ghost { background: none; border: 1px solid var(--line); color: var(--ink); border-radius: 6px; padding: 9px 14px; font: inherit; cursor: pointer; }
.hint { font-size: 12px; color: var(--muted); margin-top: 6px; }
#out { font-size: 16px; line-height: 1.9; min-height: 120px; white-space: pre-wrap; word-break: break-word; }
#out .empty { color: var(--muted); font-size: 14px; }
.seg { padding: 1px 0; transition: background-color .25s; }
.pending { background: var(--pending); }
.ok { background: var(--ok-bg); }
.bad { background: var(--bad-bg); color: var(--bad-ink); }
.pre { text-decoration: underline wavy var(--warn); text-underline-offset: 4px; }
.seg.post { box-shadow: inset 0 -3px 0 var(--post); }
.cursor { display: inline-block; width: 7px; height: 1.1em; background: var(--accent); vertical-align: -2px; animation: blink 1s steps(1) infinite; margin-left: 1px; }
@keyframes blink { 50% { opacity: 0; } }
.legend { display: flex; gap: 14px; flex-wrap: wrap; font-size: 12px; color: var(--muted); margin-top: 12px; }
.legend span { display: inline-flex; align-items: center; gap: 5px; }
.sw { width: 14px; height: 10px; border-radius: 3px; display: inline-block; }
.stats { display: grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap: 8px; margin-top: 12px; }
.stat { background: var(--bg); border-radius: 8px; padding: 8px 10px; }
.stat .k { font-size: 11px; color: var(--muted); }
.stat .v { font-size: 17px; font-weight: 600; font-variant-numeric: tabular-nums; }
.banner { border-radius: 8px; padding: 9px 12px; margin-top: 12px; font-size: 13px; display: none; }
.banner.show { display: block; }
.banner.warn { background: var(--pending); border-left: 3px solid var(--warn); }
.banner.post { background: var(--pending); border-left: 3px solid var(--post); }
.banner.cut { background: var(--bad-bg); color: var(--bad-ink); border-left: 3px solid var(--bad-ink); }
.side { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-top: 12px; }
@media (max-width: 1100px) { .side { grid-template-columns: 1fr; } }
.log { max-height: 280px; overflow: auto; font: 12px/1.5 ui-monospace, SFMono-Regular, Menlo, monospace; }
.log div { padding: 3px 0; border-bottom: 1px dashed var(--line); }
.log .t { color: var(--muted); display: inline-block; min-width: 58px; }
.tag { display: inline-block; font-size: 11px; border-radius: 4px; padding: 0 5px; margin-right: 4px; }
.tag.ok { background: var(--ok-bg); color: var(--ok-ink); }
.tag.bad { background: var(--bad-bg); color: var(--bad-ink); }
.tag.post { background: var(--pending); color: var(--post); }
.tag.gray { background: var(--pending); color: var(--muted); }
.evi { font-size: 12px; max-height: 280px; overflow: auto; }
.evi div { padding: 6px 0; border-bottom: 1px dashed var(--line); }
.evi .m { color: var(--muted); }
details summary { cursor: pointer; color: var(--muted); font-size: 12px; margin-top: 10px; }
#raw { max-height: 240px; overflow: auto; background: var(--code); border-radius: 6px; padding: 8px; font: 11px/1.45 ui-monospace, Menlo, monospace; white-space: pre-wrap; word-break: break-all; }
</style>
</head>
<body>
<header>
  <h1>hallugate</h1>
  <span class="sub">LLM 응답을 스트리밍 중에 문장 단위로 근거와 대조하는 검증 게이트웨이</span>
  <span id="health">게이트웨이 확인 중…</span>
</header>

<main>
  <section>
    <div class="card">
      <h2>예제 시나리오</h2>
      <div class="chips" id="presets"></div>
      <div class="hint" id="presethint">가짜 LLM(<code>mock_upstream.py</code>)이 아래 "가짜 LLM 응답"을 토큰 단위로 흘려보냅니다.</div>
    </div>

    <div class="card" id="keycard" hidden>
      <h2>운영 모드 키</h2>
      <label for="hgkey">게이트웨이 키 (<code>x-hallugate-key</code>)</label>
      <input id="hgkey" type="password" autocomplete="off" placeholder="hg_...">
      <div class="row">
        <div><label for="upstream">LLM 제공자 (<code>x-upstream</code>)</label>
          <select id="upstream"><option value="openai">OpenAI</option><option value="anthropic">Anthropic (Claude)</option></select></div>
        <div><label for="llmkey">LLM API 키</label><input id="llmkey" type="password" autocomplete="off" placeholder="sk-..."></div>
      </div>
      <div class="hint">키는 이 브라우저 탭에만 보관하고 게이트웨이는 저장·기록하지 않습니다. LLM 키는 고른 제공자에게만 전달됩니다.</div>
    </div>

    <div class="card">
      <h2>요청</h2>
      <label for="source">근거</label>
      <select id="source"><option value="">직접 입력한 근거 문서</option></select>
      <label for="ref">근거 문서</label>
      <textarea id="ref" rows="6"></textarea>
      <label for="q">질문</label>
      <input id="q" type="text">
      <label class="check"><input id="ragmsg" type="checkbox" checked> 근거를 LLM에도 보내기 (질문 메시지에 <code>&lt;context&gt;</code> 태그로 넣음 — 실제 RAG처럼 LLM이 근거를 보고 답함)</label>
      <label for="mock">가짜 LLM 응답 <span class="hint">(실제 LLM에 연결했다면 비워 두세요)</span></label>
      <textarea id="mock" rows="5"></textarea>
      <div class="row">
        <div><label for="model">모델</label><input id="model" type="text" value="mock"></div>
        <div><label for="speed">토큰 간격(ms)</label><input id="speed" type="number" min="0" max="500" value="60"></div>
      </div>
    </div>

    <div class="card">
      <h2>검증 옵션</h2>
      <div class="row">
        <div><label for="mode">전달 방식</label>
          <select id="mode">
            <option value="hold">hold — 판정 후 전달</option>
            <option value="pass">pass — 즉시 전달</option>
            <option value="off">off — 검증 안 함</option>
          </select></div>
        <div><label for="action">환각 판정 시</label>
          <select id="action">
            <option value="flag">flag — 표시만</option>
            <option value="cut">cut — 응답 끊기</option>
          </select></div>
      </div>
      <label for="stride">문장 앞부분 재판정 간격(토큰, 0 = 문장 끝만)</label>
      <input id="stride" type="number" min="0" max="64" value="8">
      <label class="check"><input id="post" type="checkbox"> 응답 끝 2단 검증 (Claude, 실험 기능)</label>
      <div class="btns">
        <button class="primary" id="send">보내기</button>
        <button class="ghost" id="stop" disabled>멈추기</button>
      </div>
    </div>
  </section>

  <section>
    <div class="card">
      <h2>응답</h2>
      <div id="out"><span class="empty">시나리오를 고르고 보내기를 누르세요.</span></div>
      <div class="legend">
        <span><i class="sw" style="background:var(--pending)"></i>판정 전</span>
        <span><i class="sw" style="background:var(--ok-bg)"></i>근거와 일치</span>
        <span><i class="sw" style="background:var(--bad-bg)"></i>환각 판정</span>
        <span><i class="sw" style="background:none;border-bottom:2px dotted var(--warn);border-radius:0"></i>문장 도중 경보</span>
        <span><i class="sw" style="background:none;box-shadow:inset 0 -3px 0 var(--post);border-radius:0"></i>2단 검증 환각</span>
      </div>
      <div class="banner cut" id="b-cut"></div>
      <div class="banner warn" id="b-unv"></div>
      <div class="banner post" id="b-post"></div>
      <div class="stats">
        <div class="stat"><div class="k">첫 글자까지</div><div class="v" id="s-first">–</div></div>
        <div class="stat"><div class="k">전체 시간</div><div class="v" id="s-total">–</div></div>
        <div class="stat"><div class="k">판정 수 (환각)</div><div class="v" id="s-n">–</div></div>
        <div class="stat"><div class="k">판정 시간 최대</div><div class="v" id="s-judge">–</div></div>
      </div>
    </div>

    <div class="side">
      <div class="card"><h2>판정 기록</h2><div class="log" id="log"></div></div>
      <div class="card"><h2>사용한 근거 / 2단 검증</h2><div class="evi" id="evi"><span class="hint">근거 컬렉션을 쓰거나 2단 검증을 켜면 여기에 나옵니다.</span></div></div>
    </div>
    <details class="card"><summary>받은 청크 원문 (SSE)</summary><div id="raw"></div></details>
  </section>
</main>

<script>
const REF = 'Hanbit Robotics was founded in 2011 in Daejeon by Dr. Mina Seo. The company builds warehouse robots and employs about 340 people. In 2023 it reported revenue of 52 million dollars, up 18 percent from the previous year. Its main customers are logistics firms in Korea and Japan.';
const PRESETS = [
  { name: '근거와 일치', q: 'Tell me about Hanbit Robotics.',
    a: 'Hanbit Robotics was founded in 2011 in Daejeon by Dr. Mina Seo. It builds warehouse robots and employs about 340 people. In 2023 it reported revenue of 52 million dollars, up 18 percent from the previous year.' },
  { name: '근거에 없는 내용', q: 'What is new at Hanbit Robotics?',
    a: 'Hanbit Robotics was founded in 2011 in Daejeon by Dr. Mina Seo. The company recently opened a research lab in Berlin and plans to list its shares on the Nasdaq next spring. Its main customers are logistics firms in Korea and Japan.' },
  { name: '근거와 모순', q: 'How did Hanbit Robotics do in 2023?',
    a: 'Hanbit Robotics builds warehouse robots and employs about 340 people. In 2023 it reported revenue of 52 million dollars, down 18 percent after losing its biggest customer in Germany. Its main customers are logistics firms in Korea and Japan.' },
];
// 운영 모드(실제 LLM)용: 근거와 질문만 채우고 답은 LLM이 만든다 (SPEC §2.8)
const LIVE_PRESETS = [
  { name: '문서 안 질문', q: 'How did Hanbit Robotics do in 2023?' },
  { name: '문서 밖까지 묻기 (환각 유도)', q: 'Write a detailed company profile of Hanbit Robotics: the founder\'s background, its flagship robot models, its biggest customers by name, and its plans for the next two years.' },
  { name: '수치 계산', q: 'What was Hanbit Robotics\' revenue in 2022, and about how much revenue does it make per employee?' },
];
const RAG_SYSTEM = "Answer the user's question in English based on the document inside the context tags.";
const $ = id => document.getElementById(id);
let ctrl = null, S = null, HOSTED = false;

function fmtMs(x) { return x == null ? '–' : (x >= 1000 ? (x / 1000).toFixed(2) + 's' : Math.round(x) + 'ms'); }
function esc(s) { return s.replace(/[&<>"]/g, c => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;' }[c])); }

function keyHeaders() {
  if (!HOSTED) return {};
  const k = $('hgkey').value.trim(), l = $('llmkey').value.trim(), u = $('upstream').value;
  try { sessionStorage.setItem('hgkey', k); sessionStorage.setItem('llmkey', l); sessionStorage.setItem('upstream', u); } catch (e) {}
  return { 'x-upstream': u, ...(k ? { 'x-hallugate-key': k } : {}), ...(l ? { 'Authorization': 'Bearer ' + l } : {}) };
}

function loadPreset(p) { $('ref').value = REF; $('q').value = p.q; $('mock').value = HOSTED ? '' : (p.a || ''); $('source').value = ''; }
function showPresets(list) {
  $('presets').innerHTML = '';
  list.forEach(p => { const b = document.createElement('button'); b.className = 'chip'; b.textContent = p.name; b.onclick = () => loadPreset(p); $('presets').appendChild(b); });
}

async function init() {
  showPresets(PRESETS);
  loadPreset(PRESETS[1]);
  try {
    const h = await (await fetch('/health')).json();
    if (h.hosted) {
      HOSTED = true; $('keycard').hidden = false;
      showPresets(LIVE_PRESETS); loadPreset(LIVE_PRESETS[1]);
      $('presethint').textContent = '예제는 근거와 질문만 채웁니다. 실제 LLM이 답을 만들므로 환각이 매번 나오지는 않습니다. "문서 밖까지 묻기"가 환각을 가장 잘 끌어냅니다.';
      try {
        $('hgkey').value = sessionStorage.getItem('hgkey') || ''; $('llmkey').value = sessionStorage.getItem('llmkey') || '';
        $('upstream').value = sessionStorage.getItem('upstream') || 'openai';
      } catch (e) {}
      $('upstream').onchange = () => { if (!$('model').value || /^(gpt|claude)/.test($('model').value)) $('model').value = $('upstream').value === 'anthropic' ? 'claude-sonnet-5-5' : 'gpt-4o-mini'; };
      $('mock').value = ''; $('model').value = '';
      $('upstream').onchange();
      $('health').innerHTML = '<b>운영 모드</b> · 영어 응답만 검증합니다';
      $('post').disabled = false;
      return;
    }
    const pc = h.post_check || {};
    $('health').innerHTML = `판정기 <b>${esc(h.judge)}</b>${h.judge === 'LexicalJudge' ? ' (시연용 대체 판정기, 판별력 없음)' : ''} · 임계값 <b>${Number(h.threshold).toFixed(3)}</b> · 2단 검증 <b>${pc.available ? esc(pc.model) : '사용 불가'}</b>`;
    $('stride').value = h.stride ?? 8; $('mode').value = h.mode || 'hold'; $('action').value = h.action || 'flag';
    $('post').checked = !!pc.default; $('post').disabled = !pc.available;
    if (h.collections) {
      const r = await fetch('/v1/collections');
      if (r.ok) (await r.json()).data.forEach(c => { const o = document.createElement('option'); o.value = c.collection; o.textContent = `컬렉션: ${c.collection} (문서 ${c.n_docs})`; $('source').appendChild(o); });
    }
  } catch (e) { $('health').textContent = '게이트웨이 상태를 읽지 못했습니다'; }
  $('source').onchange = () => { $('ref').disabled = !!$('source').value; };
}

function reset() {
  S = { text: '', sent: {}, pre: [], post: null, evidence: null, unv: null, cut: null, t0: performance.now(), tFirst: null, tEnd: null,
        nJ: 0, nBad: 0, maxJ: 0, done: false };
  $('log').innerHTML = ''; $('raw').textContent = '';
  $('evi').innerHTML = '<span class="hint">근거 컬렉션을 쓰거나 2단 검증을 켜면 여기에 나옵니다.</span>';
  ['b-cut', 'b-unv', 'b-post'].forEach(id => $(id).classList.remove('show'));
  render(); stats();
}

function stats() {
  $('s-first').textContent = fmtMs(S.tFirst);
  $('s-total').textContent = fmtMs(S.tEnd ?? (S.t0 && !S.done ? performance.now() - S.t0 : null));
  $('s-n').textContent = S.nJ ? `${S.nJ} (${S.nBad})` : '–';
  $('s-judge').textContent = S.nJ ? fmtMs(S.maxJ) : '–';
}

function postRanges() {
  if (!S.post || !S.post.verified) return [];
  const rows = S.post.sentences || [];
  return rows.filter(r => r.flagged).map(r => {
    const s = S.sent[r.sentence_idx]; const nxt = rows.find(x => x.start > r.start);
    return [r.start, s ? s.upto : (nxt ? nxt.start : S.text.length), r];
  });
}

function render() {
  const out = $('out'), T = S.text;
  if (!T) { out.innerHTML = S.done ? '<span class="empty">(본문 없음)</span>' : (S.t0 ? '<span class="cursor"></span>' : '<span class="empty">시나리오를 고르고 보내기를 누르세요.</span>'); return; }
  const sents = Object.values(S.sent), pr = postRanges();
  const cuts = new Set([0, T.length]);
  sents.forEach(v => { cuts.add(v.start); cuts.add(Math.min(v.upto, T.length)); });
  S.pre.forEach(v => { cuts.add(v.start); cuts.add(Math.min(v.upto, T.length)); });
  pr.forEach(([a, b]) => { cuts.add(a); cuts.add(Math.min(b, T.length)); });
  const pts = [...cuts].filter(x => x >= 0 && x <= T.length).sort((a, b) => a - b);
  let html = '';
  for (let i = 0; i + 1 < pts.length; i++) {
    const a = pts[i], b = pts[i + 1], mid = (a + b) / 2;
    const sv = sents.find(v => v.start <= a && mid < v.upto);
    const pv = S.pre.find(v => v.hallucinated && v.start <= a && mid < v.upto);
    const po = pr.find(([x, y]) => x <= a && mid < y);
    const cls = ['seg', sv ? (sv.hallucinated ? 'bad' : 'ok') : (S.mode === 'off' ? '' : 'pending')];
    if (pv && !(sv && !sv.hallucinated)) cls.push('pre');
    if (po) cls.push('post');
    const tip = [];
    if (sv) tip.push(`문장 ${sv.sentence_idx + 1}: 점수 ${sv.score.toFixed(3)} (${sv.hallucinated ? '환각' : '일치'}), 판정 ${sv.judge_ms.toFixed(1)}ms`);
    if (pv) tip.push(`문장 도중 경보: 점수 ${pv.score.toFixed(3)}`);
    if (po) tip.push(`2단 검증: ${po[2].h_votes}/${S.post.valid_votes} 판정이 환각${po[2].reason ? ' — ' + po[2].reason : ''}`);
    html += `<span class="${cls.join(' ')}"${tip.length ? ` title="${esc(tip.join('\n'))}"` : ''}>${esc(T.slice(a, b))}</span>`;
  }
  out.innerHTML = html + (S.done ? '' : '<span class="cursor"></span>');
}

function log(kind, html) {
  const d = document.createElement('div');
  d.innerHTML = `<span class="t">${fmtMs(performance.now() - S.t0)}</span>${html}`;
  $('log').appendChild(d); $('log').scrollTop = 1e9;
}

function onVerification(v) {
  if (v.kind === 'evidence' || v.kind === 'post_check') { const h = $('evi').querySelector('.hint'); if (h) h.remove(); }
  if (v.kind === 'evidence') {
    S.evidence = v;
    $('evi').innerHTML = `<div class="m">컬렉션 <b>${esc(v.collection)}</b>에서 조각 ${v.evidence.length}개 (${fmtMs(v.retrieve_ms)})</div>` +
      v.evidence.map((e, i) => `<div><span class="m">[${i + 1}] ${esc(e.title || e.doc_id)} · 유사도 ${e.score.toFixed(3)}</span><br>${esc(e.text)}</div>`).join('');
    return;
  }
  if (v.kind === 'post_check') {
    S.post = v; const b = $('b-post');
    if (v.verified) {
      b.innerHTML = `<b>2단 검증 (${esc(v.model)})</b>: ${v.valid_votes}/${v.votes} 판정 유효, 응답 점수 ${v.score.toFixed(2)} (임계값 ${v.threshold}) → <b>${v.flagged ? '환각 있음' : '환각 없음'}</b> · ${fmtMs(v.check_ms)}`;
      log('post', `<span class="tag post">2단</span>${v.flagged ? '환각 있음' : '환각 없음'} · 점수 ${v.score.toFixed(2)}`);
      const rows = (v.sentences || []).filter(r => r.h_votes);
      $('evi').insertAdjacentHTML('beforeend', rows.length ? rows.map(r => `<div><span class="tag post">2단</span>문장 ${r.sentence_idx + 1}: ${r.h_votes}표 환각${r.phrase ? ` — “${esc(r.phrase)}”` : ''}${r.reason ? `<br><span class="m">${esc(r.reason)}</span>` : ''}</div>`).join('') : '<div class="m">2단 검증: 환각으로 본 문장 없음</div>');
    } else {
      const why = { unavailable: 'ANTHROPIC_API_KEY가 없어 사용할 수 없음', timeout: '시간 초과', error: '검증기 오류', cut: '응답이 끊겨 건너뜀', empty: '판정할 문장 없음' }[v.reason] || v.reason;
      b.innerHTML = `<b>2단 검증 안 됨</b>: ${esc(why)}`;
      log('post', `<span class="tag gray">2단</span>${esc(why)}`);
    }
    b.classList.add('show'); render(); return;
  }
  if (v.kind === 'unverified') {
    S.unv = v; const why = { judge_error: '판정기 오류', judge_timeout: '판정 시간 초과', non_english: '영어가 아닌 응답' }[v.reason] || v.reason;
    $('b-unv').innerHTML = `<b>검증 중단</b> (${esc(why)}): 이후 본문은 검증 없이 그대로 전달됩니다.`; $('b-unv').classList.add('show');
    log('unv', `<span class="tag gray">중단</span>${esc(why)}`); return;
  }
  if (v.cut) {
    S.cut = v; $('b-cut').innerHTML = `<b>응답 차단</b>: 문장 ${v.sentence_idx + 1}에서 환각이 판정되어 응답을 끊었습니다 (cut).`; $('b-cut').classList.add('show');
    log('cut', `<span class="tag bad">차단</span>문장 ${v.sentence_idx + 1}`); return;
  }
  if (v.kind === 'sentence' || v.kind === 'prefix') {
    S.nJ++; if (v.hallucinated) S.nBad++; S.maxJ = Math.max(S.maxJ, v.judge_ms || 0);
    if (v.kind === 'sentence') S.sent[v.sentence_idx] = v; else S.pre.push(v);
    log(v.kind, `<span class="tag ${v.hallucinated ? 'bad' : 'ok'}">${v.kind === 'sentence' ? '문장' : '앞부분'} ${v.sentence_idx + 1}</span>점수 ${v.score.toFixed(3)} · ${v.judge_ms.toFixed(1)}ms · “${esc(v.text.length > 48 ? v.text.slice(0, 48) + '…' : v.text)}”`);
    render(); stats();
  }
}

async function send() {
  reset(); S.mode = $('mode').value;
  const ver = { mode: $('mode').value, action: $('action').value, stride: Math.max(0, parseInt($('stride').value || '0', 10)), post_check: $('post').checked };
  if ($('source').value) ver.collection = $('source').value; else ver.context = $('ref').value;
  let messages = [{ role: 'user', content: $('q').value }];
  if (!$('source').value && $('ragmsg').checked) {      // 근거를 LLM에도: <context> 태그 → 게이트웨이가 같은 근거로 판정 (SPEC §2.1·2.8)
    delete ver.context;
    messages = [{ role: 'system', content: RAG_SYSTEM }, { role: 'user', content: '<context>\n' + $('ref').value + '\n</context>\n\n' + $('q').value }];
  }
  const body = { model: $('model').value || 'mock', stream: true, messages, verification: ver };
  if ($('mock').value.trim()) Object.assign(body, { mock_response: $('mock').value, mock_token_ms: parseInt($('speed').value || '60', 10), mock_ttft_ms: 300 });
  ctrl = new AbortController(); $('send').disabled = true; $('stop').disabled = false;
  const tick = setInterval(stats, 100);
  try {
    const r = await fetch('/v1/chat/completions', { method: 'POST', headers: { 'Content-Type': 'application/json', ...keyHeaders() }, body: JSON.stringify(body), signal: ctrl.signal });
    if (!r.ok) {
      const t = await r.text(); let msg = t.slice(0, 200);
      try { msg = JSON.parse(t).error.message; } catch (e) {}
      const why = { 401: '키가 없거나 올바르지 않습니다', 413: '근거·문서가 한도보다 깁니다', 429: `사용량 한도를 넘었습니다${r.headers.get('retry-after') ? ` (${r.headers.get('retry-after')}초 뒤 다시 시도)` : ''}` }[r.status];
      log('err', `<span class="tag bad">오류 ${r.status}</span>${why ? esc(why) + ' — ' : ''}${esc(msg)}`); return;
    }
    const bw = r.headers.get('x-budget-warning'), bu = r.headers.get('x-budget-used');
    if (bw) log('budget', `<span class="tag ${bw === 'exceeded' ? 'bad' : 'gray'}">예산</span>이번 달 예산 ${Math.round(parseFloat(bu) * 100)}% 사용${bw === 'exceeded' ? ' (초과)' : ''}`);
    const rd = r.body.getReader(), dec = new TextDecoder(); let buf = '';
    while (true) {
      const { value, done } = await rd.read(); if (done) break;
      buf += dec.decode(value, { stream: true });
      let k;
      while ((k = buf.indexOf('\n\n')) >= 0) {
        const block = buf.slice(0, k); buf = buf.slice(k + 2);
        for (const line of block.split('\n')) {
          if (!line.startsWith('data:')) continue;
          const data = line.slice(5).trim();
          $('raw').textContent += data + '\n';
          if (data === '[DONE]') continue;
          const c = JSON.parse(data);
          if (c.verification) onVerification(c.verification);
          for (const ch of c.choices || []) {
            const t = ch.delta && ch.delta.content;
            if (t) { if (S.tFirst == null) S.tFirst = performance.now() - S.t0; S.text += t; render(); }
          }
        }
      }
    }
  } catch (e) {
    if (e.name !== 'AbortError') log('err', `<span class="tag bad">오류</span>${esc(String(e))}`);
  } finally {
    clearInterval(tick); S.done = true; S.tEnd = performance.now() - S.t0;
    render(); stats(); $('send').disabled = false; $('stop').disabled = true;
  }
}

$('send').onclick = send;
$('stop').onclick = () => ctrl && ctrl.abort();
init();
</script>
</body>
</html>

In [ ]:
%%writefile gateway/hallugate/static/admin.html
<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>hallugate 운영</title>
<style>
:root {
  --bg: #f6f7f9; --panel: #ffffff; --ink: #1d2330; --muted: #677085; --line: #e2e5eb;
  --accent: #3157d5; --ok-bg: #e3f5ea; --ok-ink: #17613a; --bad-bg: #fde2e1; --bad-ink: #a4231c;
  --warn-bg: #fdf0e0; --warn-ink: #8a4b0f; --soft: #eef0f4; --code: #f1f3f7;
}
@media (prefers-color-scheme: dark) {
  :root {
    --bg: #12151b; --panel: #1b1f27; --ink: #e6e9ef; --muted: #98a1b3; --line: #2c3240;
    --accent: #7d9bff; --ok-bg: #173826; --ok-ink: #8fe0ad; --bad-bg: #4a1d1b; --bad-ink: #ffb3ad;
    --warn-bg: #43301a; --warn-ink: #f3c38b; --soft: #252a35; --code: #232835;
  }
}
* { box-sizing: border-box; }
body { margin: 0; background: var(--bg); color: var(--ink);
  font: 14px/1.55 -apple-system, BlinkMacSystemFont, "Apple SD Gothic Neo", "Malgun Gothic", "Segoe UI", sans-serif; }
header { display: flex; align-items: center; gap: 14px; flex-wrap: wrap; padding: 14px 24px; border-bottom: 1px solid var(--line); background: var(--panel); }
header h1 { margin: 0; font-size: 18px; }
header nav { display: flex; gap: 12px; font-size: 13px; }
header nav a { color: var(--muted); text-decoration: none; }
header nav a:hover, header nav a.on { color: var(--accent); }
.auth { margin-left: auto; display: flex; gap: 8px; align-items: center; }
.auth input { width: 220px; }
main { padding: 16px 24px 40px; max-width: 1300px; margin: 0 auto; }
@media (max-width: 760px) { main { padding: 12px 16px 32px; } header { padding: 12px 16px; } .auth { margin-left: 0; width: 100%; } .auth input { flex: 1; width: auto; } }
input, select, button { font: inherit; }
input[type=text], input[type=password], input[type=number], select {
  color: var(--ink); background: var(--bg); border: 1px solid var(--line); border-radius: 6px; padding: 6px 9px; }
button { border-radius: 6px; padding: 6px 12px; cursor: pointer; border: 1px solid var(--line); background: var(--panel); color: var(--ink); }
button.primary { background: var(--accent); color: #fff; border-color: var(--accent); font-weight: 600; }
button.danger { color: var(--bad-ink); }
button:disabled { opacity: .5; cursor: default; }
.bar { display: flex; align-items: center; gap: 10px; flex-wrap: wrap; margin-bottom: 14px; }
.bar .seg { display: inline-flex; border: 1px solid var(--line); border-radius: 6px; overflow: hidden; }
.bar .seg button { border: 0; border-radius: 0; padding: 5px 12px; background: var(--panel); }
.bar .seg button.on { background: var(--accent); color: #fff; }
.muted { color: var(--muted); font-size: 12px; }
.msg { border-radius: 8px; padding: 10px 12px; margin-bottom: 14px; display: none; }
.msg.show { display: block; }
.msg.err { background: var(--bad-bg); color: var(--bad-ink); }
.msg.ok { background: var(--ok-bg); color: var(--ok-ink); }
.kpis { display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 10px; margin-bottom: 14px; }
.kpi { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 10px 12px; }
.kpi .k { font-size: 12px; color: var(--muted); }
.kpi .v { font-size: 22px; font-weight: 650; font-variant-numeric: tabular-nums; margin-top: 2px; }
.kpi .s { font-size: 11px; color: var(--muted); margin-top: 2px; }
.kpi.good .v { color: var(--ok-ink); }
.kpi.warn .v { color: var(--warn-ink); }
.kpi.bad .v { color: var(--bad-ink); }
.grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 12px; }
.card { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 12px 14px; min-width: 0; }
.card h2 { font-size: 13px; margin: 0 0 8px; color: var(--muted); font-weight: 600; }
.wide { margin-top: 12px; }
table { width: 100%; border-collapse: collapse; font-size: 13px; }
th, td { text-align: left; padding: 6px 6px; border-bottom: 1px solid var(--line); vertical-align: top; }
th { font-weight: 600; color: var(--muted); font-size: 12px; }
td.n, th.n { text-align: right; font-variant-numeric: tabular-nums; white-space: nowrap; }
.scroll { overflow-x: auto; }
.empty { color: var(--muted); font-size: 12px; padding: 6px 0; }
.pill { display: inline-block; font-size: 11px; border-radius: 4px; padding: 0 6px; }
.pill.ok { background: var(--ok-bg); color: var(--ok-ink); }
.pill.off { background: var(--soft); color: var(--muted); }
form.newkey { display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 8px; align-items: end; margin-top: 10px; }
form.newkey label { font-size: 11px; color: var(--muted); display: block; margin-bottom: 3px; }
form.newkey input, form.newkey select { width: 100%; }
.newsecret { margin-top: 10px; background: var(--warn-bg); color: var(--warn-ink); border-radius: 8px; padding: 10px 12px; display: none; }
.newsecret code { display: block; margin: 6px 0; padding: 6px 8px; background: var(--code); color: var(--ink); border-radius: 6px; word-break: break-all; font-size: 12px; }
</style>
</head>
<body>
<header>
  <h1>hallugate 운영</h1>
  <nav><a class="on" href="/admin">운영</a><a href="/demo">데모</a><a href="/console">사용자 화면</a><a href="/start">시작 안내</a></nav>
  <div class="auth">
    <input id="tok" type="password" autocomplete="off" placeholder="관리자 토큰 (ADMIN_TOKEN)">
    <button class="primary" id="load">불러오기</button>
  </div>
</header>

<main>
  <div class="msg" id="msg"></div>

  <div class="bar">
    <span class="seg" id="win">
      <button data-w="1h">1시간</button><button data-w="24h" class="on">24시간</button><button data-w="7d">7일</button><button data-w="30d">30일</button>
    </span>
    <span class="muted" id="when">관리자 토큰을 넣고 불러오기를 누르세요. 토큰은 이 탭에만 보관합니다.</span>
  </div>

  <div class="kpis" id="kpis"></div>

  <div class="grid">
    <div class="card"><h2>요청 · 상태 코드</h2><div id="t-status"></div></div>
    <div class="card"><h2>지연</h2><div id="t-lat"></div></div>
    <div class="card"><h2>검증 중단 이유</h2><div id="t-unv"></div></div>
    <div class="card"><h2>LLM 사용량 (모델별, 추정 비용)</h2><div class="scroll" id="t-llm"></div></div>
    <div class="card"><h2>사용자별 요청</h2><div class="scroll" id="t-bykey"></div></div>
    <div class="card"><h2>2단 검증 · 근거 다이어트</h2><div id="t-extra"></div></div>
  </div>

  <div class="card wide"><h2>콜드 스타트 (컨테이너 시작)</h2><div class="scroll" id="t-cold"></div></div>

  <div class="card wide">
    <h2>API 키</h2>
    <div class="scroll" id="t-keys"></div>
    <form class="newkey" id="newkey">
      <div><label for="nk-name">이름</label><input id="nk-name" type="text" placeholder="alice" required></div>
      <div><label for="nk-rpm">분당 요청 (기본 30)</label><input id="nk-rpm" type="number" min="0"></div>
      <div><label for="nk-conc">동시 요청 (기본 4)</label><input id="nk-conc" type="number" min="0"></div>
      <div><label for="nk-pc">하루 2단 검증 (기본 50)</label><input id="nk-pc" type="number" min="0"></div>
      <div><label for="nk-usd">월 예산 USD (선택)</label><input id="nk-usd" type="number" min="0" step="0.01"></div>
      <div><label for="nk-act">예산 초과 시</label><select id="nk-act"><option value="warn">경고만</option><option value="block">차단</option></select></div>
      <div><button class="primary" type="submit">키 발급</button></div>
    </form>
    <div class="newsecret" id="newsecret">
      새 키입니다. <b>지금 한 번만 보입니다.</b> 사용자에게 안전한 경로로 전달하세요.
      <code id="ns-key"></code>
      <button type="button" id="ns-copy">복사</button>
    </div>
  </div>
</main>

<script>
const $ = id => document.getElementById(id);
let WIN = '24h', NAMES = {};

const fmtMs = x => x == null ? '–' : (x >= 1000 ? (x / 1000).toFixed(2) + 's' : Math.round(x) + 'ms');
const fmtPct = x => x == null ? '–' : (x * 100).toFixed(x < 0.1 ? 1 : 0) + '%';
const fmtUsd = x => x == null ? '–' : '$' + (x < 1 ? x.toFixed(4) : x.toFixed(2));
const fmtN = x => x == null ? '–' : Number(x).toLocaleString('ko-KR');
const fmtTs = t => t ? new Date(t * 1000).toLocaleString('ko-KR', { month: '2-digit', day: '2-digit', hour: '2-digit', minute: '2-digit' }) : '–';

function msg(text, kind) { const m = $('msg'); m.textContent = text; m.className = 'msg show ' + kind; if (!text) m.className = 'msg'; }
function hdr() { return { 'Authorization': 'Bearer ' + $('tok').value.trim() }; }

function el(tag, attrs, ...kids) {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs || {})) { if (k === 'class') e.className = v; else if (k.startsWith('on')) e[k] = v; else e.setAttribute(k, v); }
  for (const c of kids) e.append(c instanceof Node ? c : document.createTextNode(c == null ? '' : String(c)));
  return e;
}
// rows: [[셀, ...]], numeric: 오른쪽 정렬할 열 번호
function table(head, rows, numeric = []) {
  if (!rows.length) return el('div', { class: 'empty' }, '이 기간에 기록이 없습니다.');
  const n = i => numeric.includes(i) ? 'n' : '';
  return el('table', {}, el('thead', {}, el('tr', {}, ...head.map((h, i) => el('th', { class: n(i) }, h)))),
    el('tbody', {}, ...rows.map(r => el('tr', {}, ...r.map((c, i) => el('td', { class: n(i) }, c))))));
}
function put(id, node) { const t = $(id); t.replaceChildren(node); }
function kpi(k, v, s, cls) { return el('div', { class: 'kpi ' + (cls || '') }, el('div', { class: 'k' }, k), el('div', { class: 'v' }, v), el('div', { class: 's' }, s || '')); }

async function api(path, opts = {}) {
  const r = await fetch(path, { ...opts, headers: { ...hdr(), ...(opts.body ? { 'content-type': 'application/json' } : {}) } });
  if (r.status === 401) throw new Error('관리자 토큰이 없거나 틀렸습니다.');
  if (!r.ok) throw new Error(`${path} → ${r.status}`);
  return r.json();
}

function render(st) {
  const req = st.requests || {}, lat = st.latency || {}, v = st.verification || {}, llm = st.llm || {}, pc = st.post_check || {}, tr = st.trim || {};
  const total = req.total || 0, byS = req.by_status || {};
  const err5 = Object.entries(byS).filter(([k]) => k.startsWith('5')).reduce((a, [, n]) => a + n, 0);
  const jp = lat.judge_ms_p95;
  const unvRate = v.responses ? (v.responses - v.fully_verified) / v.responses : null;
  $('kpis').replaceChildren(
    kpi('요청', fmtN(total), `성공 ${fmtN(byS['200'] || 0)} · 429 ${fmtN(st.rate_limited || 0)}`),
    kpi('서버 오류 (5xx)', fmtN(err5), total ? '요청의 ' + fmtPct(err5 / total) : '', err5 ? 'bad' : 'good'),
    kpi('첫 글자 p95', fmtMs(lat.ttft_ms && lat.ttft_ms.p95), 'p50 ' + fmtMs(lat.ttft_ms && lat.ttft_ms.p50)),
    kpi('판정 시간 p95', fmtMs(jp), '목표 68ms 이하', jp == null ? '' : (jp <= 68 ? 'good' : 'warn')),
    kpi('환각 판정 문장', fmtPct(v.flag_rate), `${fmtN(v.sentences_flagged)} / ${fmtN(v.sentences_judged)} 문장 · 응답 ${fmtN(v.responses_flagged)}개`),
    kpi('검증 중단', fmtPct(unvRate), `검증 켠 응답 ${fmtN(v.responses)}개 중`, unvRate ? 'warn' : ''),
    kpi('LLM 추정 비용', fmtUsd(llm.est_cost_usd), `입력 ${fmtN(llm.tokens_in)} · 출력 ${fmtN(llm.tokens_out)} 토큰`),
    kpi('2단 검증 비용', fmtUsd(pc.est_cost_usd), `실행 ${fmtN(pc.runs)} · 환각 ${fmtN(pc.flagged)}`),
  );
  put('t-status', table(['상태 코드', '요청'], Object.entries(byS).map(([k, n]) => [k, fmtN(n)]), [1]));
  put('t-lat', table(['항목', 'p50', 'p95'], [
    ['첫 글자까지', fmtMs(lat.ttft_ms && lat.ttft_ms.p50), fmtMs(lat.ttft_ms && lat.ttft_ms.p95)],
    ['전체 시간', fmtMs(lat.total_ms && lat.total_ms.p50), fmtMs(lat.total_ms && lat.total_ms.p95)],
    ['판정 1회', '–', fmtMs(jp)]], [1, 2]));
  const reasons = { non_english: '영어 아님', judge_timeout: '판정 시간 초과', judge_error: '판정기 오류', unknown: '알 수 없음' };
  put('t-unv', table(['이유', '응답'], Object.entries(v.unverified_by_reason || {}).map(([k, n]) => [reasons[k] || k, fmtN(n)]), [1]));
  put('t-llm', table(['모델', '요청', '입력', '출력', '비용'], Object.entries(llm.by_model || {}).map(([m, d]) =>
    [m, fmtN(d.requests), fmtN(d.tokens_in), fmtN(d.tokens_out), fmtUsd(d.est_cost_usd)]), [1, 2, 3, 4]));
  put('t-bykey', table(['키', '이름', '요청'], Object.entries(req.by_key || {}).sort((a, b) => b[1] - a[1]).map(([k, n]) =>
    [k, NAMES[k] || '–', fmtN(n)]), [2]));
  put('t-extra', table(['항목', '값'], [
    ['2단 검증 요청 / 실행', `${fmtN(pc.requested)} / ${fmtN(pc.runs)}`],
    ['2단 검증 하루 한도 초과', fmtN(pc.quota)],
    ['근거 다이어트 사용 응답', fmtN(tr.requests)],
    ['근거 글자 수 (전 → 후)', `${fmtN(tr.chars_before)} → ${fmtN(tr.chars_after)}`],
    ['근거 절감', fmtPct(tr.saved_ratio)]], [1]));
  put('t-cold', table(['시작 시각', '판정기 예열', '시작 직후 첫 요청 첫 글자'], (st.cold_starts || []).slice().reverse().map(c =>
    [fmtTs(c.ts), fmtMs(c.warmup_ms), fmtMs(c.first_ttft_ms)]), [1, 2]));
}

function renderKeys(list) {
  NAMES = Object.fromEntries(list.map(k => [k.key_id, k.name]));
  put('t-keys', table(['키 ID', '이름', '상태', '만든 시각', '한도', ''], list.map(k => {
    const lim = k.limits && Object.keys(k.limits).length ? Object.entries(k.limits).map(([a, b]) => `${a}=${b}`).join(' ') : '기본';
    const st = k.revoked ? el('span', { class: 'pill off' }, '폐기 ' + fmtTs(k.revoked)) : el('span', { class: 'pill ok' }, '사용 중');
    const act = k.revoked ? '' : el('button', { class: 'danger', onclick: () => revoke(k) }, '폐기');
    return [k.key_id, k.name || '–', st, fmtTs(k.created), lim, act];
  })));
}

async function load() {
  msg('');
  try { sessionStorage.setItem('admintok', $('tok').value.trim()); } catch (e) {}
  $('load').disabled = true;
  try {
    const keys = await api('/admin/keys');
    renderKeys(keys.data || []);
    render(await api('/admin/stats?window=' + WIN));
    $('when').textContent = `최근 ${WIN} · ${new Date().toLocaleTimeString('ko-KR')} 기준 · 원문 없이 숫자만 기록합니다.`;
  } catch (e) { msg(e.message, 'err'); }
  finally { $('load').disabled = false; }
}

async function revoke(k) {
  if (!confirm(`키 ${k.key_id} (${k.name || '이름 없음'})를 폐기할까요? 이 키로 오는 요청은 바로 401이 됩니다.`)) return;
  const purge = confirm('이 사용자의 문서(컬렉션)와 측정 기록도 모두 지울까요?\n확인 = 지움 (되돌릴 수 없음), 취소 = 키만 폐기');
  try {
    const r = await api(`/admin/keys/${encodeURIComponent(k.key_id)}/revoke`, { method: 'POST', body: JSON.stringify({ purge }) });
    msg(`폐기했습니다: ${r.key_id}` + (purge ? ` (문서 ${r.purged_docs}개·기록 ${r.purged_records}개 삭제)` : ''), 'ok');
    load();
  } catch (e) { msg(e.message, 'err'); }
}

$('newkey').onsubmit = async ev => {
  ev.preventDefault();
  const limits = {}, num = (id, k, f = parseInt) => { const v = $(id).value.trim(); if (v !== '') limits[k] = f(v, 10); };
  num('nk-rpm', 'rpm'); num('nk-conc', 'concurrent'); num('nk-pc', 'post_check_daily'); num('nk-usd', 'monthly_usd', parseFloat);
  if ('monthly_usd' in limits) limits.budget_action = $('nk-act').value;
  try {
    const r = await api('/admin/keys', { method: 'POST', body: JSON.stringify({ name: $('nk-name').value.trim(), limits }) });
    $('ns-key').textContent = r.key; $('newsecret').style.display = 'block';
    $('newkey').reset(); load();
  } catch (e) { msg(e.message, 'err'); }
};
$('ns-copy').onclick = async () => { try { await navigator.clipboard.writeText($('ns-key').textContent); $('ns-copy').textContent = '복사됨'; } catch (e) {} };

document.querySelectorAll('#win button').forEach(b => b.onclick = () => {
  document.querySelectorAll('#win button').forEach(x => x.classList.toggle('on', x === b));
  WIN = b.dataset.w; if ($('tok').value.trim()) load();
});
$('load').onclick = load;
$('tok').onkeydown = e => { if (e.key === 'Enter') load(); };
try { const t = sessionStorage.getItem('admintok'); if (t) { $('tok').value = t; load(); } } catch (e) {}
</script>
</body>
</html>

In [ ]:
%%writefile gateway/hallugate/static/console.html
<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>hallugate 내 사용 현황</title>
<style>
:root {
  --bg: #f6f7f9; --panel: #ffffff; --ink: #1d2330; --muted: #677085; --line: #e2e5eb;
  --accent: #3157d5; --ok-bg: #e3f5ea; --ok-ink: #17613a; --bad-bg: #fde2e1; --bad-ink: #a4231c;
  --warn-bg: #fdf0e0; --warn-ink: #8a4b0f; --soft: #eef0f4; --code: #f1f3f7;
}
@media (prefers-color-scheme: dark) {
  :root {
    --bg: #12151b; --panel: #1b1f27; --ink: #e6e9ef; --muted: #98a1b3; --line: #2c3240;
    --accent: #7d9bff; --ok-bg: #173826; --ok-ink: #8fe0ad; --bad-bg: #4a1d1b; --bad-ink: #ffb3ad;
    --warn-bg: #43301a; --warn-ink: #f3c38b; --soft: #252a35; --code: #232835;
  }
}
* { box-sizing: border-box; }
body { margin: 0; background: var(--bg); color: var(--ink);
  font: 14px/1.55 -apple-system, BlinkMacSystemFont, "Apple SD Gothic Neo", "Malgun Gothic", "Segoe UI", sans-serif; }
header { display: flex; align-items: center; gap: 14px; flex-wrap: wrap; padding: 14px 24px; border-bottom: 1px solid var(--line); background: var(--panel); }
header h1 { margin: 0; font-size: 18px; }
header nav { display: flex; gap: 12px; font-size: 13px; }
header nav a { color: var(--muted); text-decoration: none; }
header nav a:hover, header nav a.on { color: var(--accent); }
.auth { margin-left: auto; display: flex; gap: 8px; align-items: center; }
.auth input { width: 220px; }
main { padding: 16px 24px 40px; max-width: 1300px; margin: 0 auto; }
@media (max-width: 760px) { main { padding: 12px 16px 32px; } header { padding: 12px 16px; } .auth { margin-left: 0; width: 100%; } .auth input { flex: 1; width: auto; } }
input, select, button { font: inherit; }
input[type=text], input[type=password], input[type=number], select {
  color: var(--ink); background: var(--bg); border: 1px solid var(--line); border-radius: 6px; padding: 6px 9px; }
button { border-radius: 6px; padding: 6px 12px; cursor: pointer; border: 1px solid var(--line); background: var(--panel); color: var(--ink); }
button.primary { background: var(--accent); color: #fff; border-color: var(--accent); font-weight: 600; }
button.danger { color: var(--bad-ink); }
button:disabled { opacity: .5; cursor: default; }
.bar { display: flex; align-items: center; gap: 10px; flex-wrap: wrap; margin-bottom: 14px; }
.bar .seg { display: inline-flex; border: 1px solid var(--line); border-radius: 6px; overflow: hidden; }
.bar .seg button { border: 0; border-radius: 0; padding: 5px 12px; background: var(--panel); }
.bar .seg button.on { background: var(--accent); color: #fff; }
.muted { color: var(--muted); font-size: 12px; }
.msg { border-radius: 8px; padding: 10px 12px; margin-bottom: 14px; display: none; }
.msg.show { display: block; }
.msg.err { background: var(--bad-bg); color: var(--bad-ink); }
.msg.ok { background: var(--ok-bg); color: var(--ok-ink); }
.kpis { display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 10px; margin-bottom: 14px; }
.kpi { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 10px 12px; }
.kpi .k { font-size: 12px; color: var(--muted); }
.kpi .v { font-size: 22px; font-weight: 650; font-variant-numeric: tabular-nums; margin-top: 2px; }
.kpi .s { font-size: 11px; color: var(--muted); margin-top: 2px; }
.kpi.good .v { color: var(--ok-ink); }
.kpi.warn .v { color: var(--warn-ink); }
.kpi.bad .v { color: var(--bad-ink); }
.grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 12px; }
.card { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 12px 14px; min-width: 0; }
.card h2 { font-size: 13px; margin: 0 0 8px; color: var(--muted); font-weight: 600; }
.wide { margin-top: 12px; }
table { width: 100%; border-collapse: collapse; font-size: 13px; }
th, td { text-align: left; padding: 6px 6px; border-bottom: 1px solid var(--line); vertical-align: top; }
th { font-weight: 600; color: var(--muted); font-size: 12px; }
td.n, th.n { text-align: right; font-variant-numeric: tabular-nums; white-space: nowrap; }
.scroll { overflow-x: auto; }
.empty { color: var(--muted); font-size: 12px; padding: 6px 0; }
.pill { display: inline-block; font-size: 11px; border-radius: 4px; padding: 0 6px; }
.pill.ok { background: var(--ok-bg); color: var(--ok-ink); }
.pill.off { background: var(--soft); color: var(--muted); }
form.newkey { display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 8px; align-items: end; margin-top: 10px; }
form.newkey label { font-size: 11px; color: var(--muted); display: block; margin-bottom: 3px; }
form.newkey input, form.newkey select { width: 100%; }
.newsecret { margin-top: 10px; background: var(--warn-bg); color: var(--warn-ink); border-radius: 8px; padding: 10px 12px; display: none; }
.newsecret code { display: block; margin: 6px 0; padding: 6px 8px; background: var(--code); color: var(--ink); border-radius: 6px; word-break: break-all; font-size: 12px; }
.budget { margin-top: 6px; }
.budget .track { height: 10px; background: var(--soft); border-radius: 999px; overflow: hidden; }
.budget .fill { height: 100%; background: var(--accent); width: 0; }
.budget .fill.near { background: var(--warn-ink); }
.budget .fill.over { background: var(--bad-ink); }
.tips { margin: 0; padding-left: 18px; font-size: 13px; }
.tips li { margin: 4px 0; }
</style>
</head>
<body>
<header>
  <h1>hallugate</h1>
  <nav><a class="on" href="/console">내 사용 현황</a><a href="/start">시작 안내</a><a href="/demo">데모</a></nav>
  <div class="auth">
    <input id="key" type="password" autocomplete="off" placeholder="게이트웨이 키 (x-hallugate-key, hg_...)">
    <button class="primary" id="load">불러오기</button>
  </div>
</header>

<main>
  <div class="msg" id="msg"></div>
  <div class="bar">
    <span class="seg" id="win">
      <button data-w="24h">24시간</button><button data-w="7d">7일</button><button data-w="30d" class="on">30일</button>
    </span>
    <span class="muted" id="when">게이트웨이 키를 넣고 불러오기를 누르세요. 키는 이 탭에만 보관하고, 내 키의 기록만 보입니다.</span>
  </div>

  <div class="kpis" id="kpis"></div>

  <div class="grid">
    <div class="card"><h2>이번 달 예산</h2><div id="t-budget"></div></div>
    <div class="card"><h2>검증 결과</h2><div id="t-verify"></div></div>
    <div class="card"><h2>LLM 사용량 (모델별, 추정 비용)</h2><div class="scroll" id="t-llm"></div></div>
    <div class="card"><h2>참고</h2><ul class="tips" id="tips"></ul></div>
  </div>
  <p class="muted">추정 비용은 공개 단가표로 계산한 값이며 캐시 할인을 반영하지 않아 실제 청구액보다 클 수 있습니다. 질문·근거·응답 원문은 기록하지 않고 숫자만 남깁니다(30일 보관).</p>
</main>

<script>
const $ = id => document.getElementById(id);
let WIN = '30d';
const fmtPct = x => x == null ? '–' : (x * 100).toFixed(x < 0.1 ? 1 : 0) + '%';
const fmtUsd = x => x == null ? '–' : '$' + (x < 1 ? x.toFixed(4) : x.toFixed(2));
const fmtN = x => x == null ? '–' : Number(x).toLocaleString('ko-KR');

function msg(text, kind) { const m = $('msg'); m.textContent = text; m.className = text ? 'msg show ' + kind : 'msg'; }
function el(tag, attrs, ...kids) {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs || {})) { if (k === 'class') e.className = v; else e.setAttribute(k, v); }
  for (const c of kids) e.append(c instanceof Node ? c : document.createTextNode(c == null ? '' : String(c)));
  return e;
}
function table(head, rows, numeric = []) {
  if (!rows.length) return el('div', { class: 'empty' }, '이 기간에 기록이 없습니다.');
  const n = i => numeric.includes(i) ? 'n' : '';
  return el('table', {}, el('thead', {}, el('tr', {}, ...head.map((h, i) => el('th', { class: n(i) }, h)))),
    el('tbody', {}, ...rows.map(r => el('tr', {}, ...r.map((c, i) => el('td', { class: n(i) }, c))))));
}
function kpi(k, v, s, cls) { return el('div', { class: 'kpi ' + (cls || '') }, el('div', { class: 'k' }, k), el('div', { class: 'v' }, v), el('div', { class: 's' }, s || '')); }
const REASONS = { non_english: '영어가 아님', judge_timeout: '판정 시간 초과', judge_error: '판정기 오류', unknown: '알 수 없음' };

function render(u) {
  const v = u.verification || {}, tr = u.trim || {}, b = u.budget;
  const unv = v.responses ? (v.responses - v.fully_verified) / v.responses : null;
  $('kpis').replaceChildren(
    kpi('요청', fmtN(u.requests), '성공한 요청'),
    kpi('환각 판정 문장', fmtPct(v.flag_rate), `${fmtN(v.sentences_flagged)} / ${fmtN(v.sentences_judged)} 문장`),
    kpi('환각이 있던 응답', fmtN(v.responses_flagged), `검증한 응답 ${fmtN(v.responses)}개 중`),
    kpi('검증 중단', fmtPct(unv), '영어가 아니거나 판정 실패', unv ? 'warn' : ''),
    kpi('LLM 추정 비용', fmtUsd(u.est_cost_usd), `입력 ${fmtN(u.tokens_in)} · 출력 ${fmtN(u.tokens_out)} 토큰`),
    kpi('근거 다이어트 절감', fmtPct(tr.saved_ratio), tr.requests ? `${fmtN(tr.requests)}개 요청에서 근거 글자 수` : 'trim을 켠 요청 없음', tr.saved_ratio ? 'good' : ''),
  );
  if (!b) $('t-budget').replaceChildren(el('div', { class: 'empty' }, '예산이 설정되지 않았습니다. 필요하면 운영자에게 월 한도(토큰 또는 USD)를 요청하세요.'));
  else {
    const used = b.used || 0, cls = used >= 1 ? 'over' : used >= 0.8 ? 'near' : '';
    const fill = el('div', { class: 'fill ' + cls }); fill.style.width = Math.min(100, used * 100) + '%';
    $('t-budget').replaceChildren(
      el('div', { class: 'kpi ' + (used >= 1 ? 'bad' : used >= 0.8 ? 'warn' : 'good') }, el('div', { class: 'k' }, `${b.month} 사용률`), el('div', { class: 'v' }, fmtPct(used))),
      el('div', { class: 'budget' }, el('div', { class: 'track' }, fill)),
      table(['항목', '사용', '한도'], [
        ...(b.monthly_usd != null ? [['비용 (USD)', fmtUsd(b.cost_used_usd), fmtUsd(b.monthly_usd)]] : []),
        ...(b.monthly_tokens != null ? [['토큰', fmtN(b.tokens_used), fmtN(b.monthly_tokens)]] : []),
        ['넘으면', b.action === 'block' ? '요청 차단 (429)' : '경고 헤더만', '']], [1, 2]));
  }
  $('t-verify').replaceChildren(table(['항목', '값'], [
    ['검증한 응답', fmtN(v.responses)], ['끝까지 검증된 응답', fmtN(v.fully_verified)],
    ['판정한 문장', fmtN(v.sentences_judged)], ['환각 판정 문장', fmtN(v.sentences_flagged)],
    ...Object.entries(v.unverified_by_reason || {}).map(([k, n]) => ['검증 중단: ' + (REASONS[k] || k), fmtN(n)])], [1]));
  $('t-llm').replaceChildren(table(['모델', '요청', '입력', '출력', '비용'], Object.entries(u.by_model || {}).map(([m, d]) =>
    [m, fmtN(d.requests), fmtN(d.tokens_in), fmtN(d.tokens_out), fmtUsd(d.est_cost_usd)]), [1, 2, 3, 4]));
  const tips = [];
  if ((v.unverified_by_reason || {}).non_english) tips.push('판정기는 영어만 검증합니다. 영어가 아닌 문장이 나오면 그 응답은 검증을 멈추고 본문만 그대로 전달합니다.');
  if (!tr.requests && u.tokens_in > 0) tips.push('근거 문서가 길다면 "verification": {"trim": true}로 LLM 입력 토큰을 줄일 수 있습니다(실험에서 약 48% 절감, 답 품질 유지).');
  if (v.responses === 0 && u.requests > 0) tips.push('근거 문서를 보내지 않으면 검증하지 않습니다. verification.context 또는 <context> 태그로 근거를 넣으세요.');
  if (b && b.used >= 0.8) tips.push('이번 달 예산의 80% 이상을 썼습니다.');
  if (!tips.length) tips.push('특이 사항이 없습니다.');
  $('tips').replaceChildren(...tips.map(t => el('li', {}, t)));
}

async function load() {
  msg('');
  const k = $('key').value.trim();
  try { sessionStorage.setItem('hgkey', k); } catch (e) {}
  $('load').disabled = true;
  try {
    const r = await fetch('/v1/usage?window=' + WIN, { headers: { 'x-hallugate-key': k } });
    if (r.status === 401) throw new Error('게이트웨이 키가 없거나 틀렸습니다.');
    if (!r.ok) throw new Error('/v1/usage → ' + r.status);
    render(await r.json());
    $('when').textContent = `최근 ${WIN} · ${new Date().toLocaleTimeString('ko-KR')} 기준`;
  } catch (e) { msg(e.message, 'err'); }
  finally { $('load').disabled = false; }
}
document.querySelectorAll('#win button').forEach(b => b.onclick = () => {
  document.querySelectorAll('#win button').forEach(x => x.classList.toggle('on', x === b));
  WIN = b.dataset.w; if ($('key').value.trim()) load();
});
$('load').onclick = load;
$('key').onkeydown = e => { if (e.key === 'Enter') load(); };
try { const k = sessionStorage.getItem('hgkey'); if (k) { $('key').value = k; load(); } } catch (e) {}
</script>
</body>
</html>

In [ ]:
%%writefile gateway/hallugate/static/start.html
<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>hallugate 시작 안내</title>
<style>
:root {
  --bg: #f6f7f9; --panel: #ffffff; --ink: #1d2330; --muted: #677085; --line: #e2e5eb;
  --accent: #3157d5; --ok-bg: #e3f5ea; --ok-ink: #17613a; --bad-bg: #fde2e1; --bad-ink: #a4231c;
  --warn-bg: #fdf0e0; --warn-ink: #8a4b0f; --soft: #eef0f4; --code: #f1f3f7;
}
@media (prefers-color-scheme: dark) {
  :root {
    --bg: #12151b; --panel: #1b1f27; --ink: #e6e9ef; --muted: #98a1b3; --line: #2c3240;
    --accent: #7d9bff; --ok-bg: #173826; --ok-ink: #8fe0ad; --bad-bg: #4a1d1b; --bad-ink: #ffb3ad;
    --warn-bg: #43301a; --warn-ink: #f3c38b; --soft: #252a35; --code: #232835;
  }
}
* { box-sizing: border-box; }
body { margin: 0; background: var(--bg); color: var(--ink);
  font: 14px/1.55 -apple-system, BlinkMacSystemFont, "Apple SD Gothic Neo", "Malgun Gothic", "Segoe UI", sans-serif; }
header { display: flex; align-items: center; gap: 14px; flex-wrap: wrap; padding: 14px 24px; border-bottom: 1px solid var(--line); background: var(--panel); }
header h1 { margin: 0; font-size: 18px; }
header nav { display: flex; gap: 12px; font-size: 13px; }
header nav a { color: var(--muted); text-decoration: none; }
header nav a:hover, header nav a.on { color: var(--accent); }
.auth { margin-left: auto; display: flex; gap: 8px; align-items: center; }
.auth input { width: 220px; }
main { padding: 16px 24px 40px; max-width: 1300px; margin: 0 auto; }
@media (max-width: 760px) { main { padding: 12px 16px 32px; } header { padding: 12px 16px; } .auth { margin-left: 0; width: 100%; } .auth input { flex: 1; width: auto; } }
input, select, button { font: inherit; }
input[type=text], input[type=password], input[type=number], select {
  color: var(--ink); background: var(--bg); border: 1px solid var(--line); border-radius: 6px; padding: 6px 9px; }
button { border-radius: 6px; padding: 6px 12px; cursor: pointer; border: 1px solid var(--line); background: var(--panel); color: var(--ink); }
button.primary { background: var(--accent); color: #fff; border-color: var(--accent); font-weight: 600; }
button.danger { color: var(--bad-ink); }
button:disabled { opacity: .5; cursor: default; }
.bar { display: flex; align-items: center; gap: 10px; flex-wrap: wrap; margin-bottom: 14px; }
.bar .seg { display: inline-flex; border: 1px solid var(--line); border-radius: 6px; overflow: hidden; }
.bar .seg button { border: 0; border-radius: 0; padding: 5px 12px; background: var(--panel); }
.bar .seg button.on { background: var(--accent); color: #fff; }
.muted { color: var(--muted); font-size: 12px; }
.msg { border-radius: 8px; padding: 10px 12px; margin-bottom: 14px; display: none; }
.msg.show { display: block; }
.msg.err { background: var(--bad-bg); color: var(--bad-ink); }
.msg.ok { background: var(--ok-bg); color: var(--ok-ink); }
.kpis { display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 10px; margin-bottom: 14px; }
.kpi { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 10px 12px; }
.kpi .k { font-size: 12px; color: var(--muted); }
.kpi .v { font-size: 22px; font-weight: 650; font-variant-numeric: tabular-nums; margin-top: 2px; }
.kpi .s { font-size: 11px; color: var(--muted); margin-top: 2px; }
.kpi.good .v { color: var(--ok-ink); }
.kpi.warn .v { color: var(--warn-ink); }
.kpi.bad .v { color: var(--bad-ink); }
.grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 12px; }
.card { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 12px 14px; min-width: 0; }
.card h2 { font-size: 13px; margin: 0 0 8px; color: var(--muted); font-weight: 600; }
.wide { margin-top: 12px; }
table { width: 100%; border-collapse: collapse; font-size: 13px; }
th, td { text-align: left; padding: 6px 6px; border-bottom: 1px solid var(--line); vertical-align: top; }
th { font-weight: 600; color: var(--muted); font-size: 12px; }
td.n, th.n { text-align: right; font-variant-numeric: tabular-nums; white-space: nowrap; }
.scroll { overflow-x: auto; }
.empty { color: var(--muted); font-size: 12px; padding: 6px 0; }
.pill { display: inline-block; font-size: 11px; border-radius: 4px; padding: 0 6px; }
.pill.ok { background: var(--ok-bg); color: var(--ok-ink); }
.pill.off { background: var(--soft); color: var(--muted); }
form.newkey { display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 8px; align-items: end; margin-top: 10px; }
form.newkey label { font-size: 11px; color: var(--muted); display: block; margin-bottom: 3px; }
form.newkey input, form.newkey select { width: 100%; }
.newsecret { margin-top: 10px; background: var(--warn-bg); color: var(--warn-ink); border-radius: 8px; padding: 10px 12px; display: none; }
.newsecret code { display: block; margin: 6px 0; padding: 6px 8px; background: var(--code); color: var(--ink); border-radius: 6px; word-break: break-all; font-size: 12px; }
article { max-width: 860px; margin: 0 auto; }
article h2 { font-size: 17px; margin: 28px 0 8px; color: var(--ink); }
article h3 { font-size: 14px; margin: 18px 0 6px; }
article p, article li { font-size: 14px; }
pre { background: var(--code); border-radius: 8px; padding: 12px 14px; overflow-x: auto; font: 12.5px/1.55 ui-monospace, SFMono-Regular, Menlo, monospace; position: relative; }
code { font: 12.5px ui-monospace, SFMono-Regular, Menlo, monospace; background: var(--code); border-radius: 4px; padding: 0 4px; }
pre code { background: none; padding: 0; }
.copy { position: absolute; top: 8px; right: 8px; font-size: 11px; padding: 2px 8px; }
.note { background: var(--warn-bg); color: var(--warn-ink); border-radius: 8px; padding: 10px 12px; font-size: 13px; }
td code { white-space: nowrap; }
</style>
</head>
<body>
<header>
  <h1>hallugate</h1>
  <nav><a href="/console">내 사용 현황</a><a class="on" href="/start">시작 안내</a><a href="/demo">데모</a></nav>
</header>
<main><article>
  <p>hallugate는 LLM API 앞에 두는 <b>검증 게이트웨이</b>입니다. 지금 쓰는 OpenAI 호환 코드에서 <b>주소와 헤더만 바꾸면</b>, LLM 답이 흘러나오는 동안 문장마다 근거 문서와 맞춰 보고 근거와 어긋나거나 근거에 없는 문장(환각)을 표시합니다. 답 본문은 그대로 전달됩니다.</p>
  <div class="note">현재 초대제 베타입니다. <b>영어 RAG 응답만</b> 검증합니다(판정기가 영어로만 학습됨). 영어가 아니면 본문은 그대로 전달하고 검증 결과만 <code>verified: false</code>로 알립니다. 서버가 쉬고 있으면 첫 요청에 20초 정도 걸릴 수 있습니다.</div>

  <h2>1. 필요한 것</h2>
  <ul>
    <li><b>게이트웨이 키</b> <code>hg_...</code> — 운영자에게 받습니다. 헤더 <code>x-hallugate-key</code>로 보냅니다</li>
    <li><b>내 LLM API 키</b> — OpenAI 또는 Anthropic. 게이트웨이는 고른 업체에만 전달하고 저장·기록하지 않습니다</li>
    <li><b>업체 선택</b> — 헤더 <code>x-upstream: openai</code> 또는 <code>anthropic</code></li>
  </ul>

  <h2>2. 연결 (OpenAI Python SDK)</h2>
<pre><button class="copy">복사</button><code>from openai import OpenAI

client = OpenAI(
    base_url="<span class="origin"></span>/v1",
    api_key="&lt;내 OpenAI 또는 Anthropic 키&gt;",
    default_headers={"x-hallugate-key": "hg_...", "x-upstream": "openai"},   # Claude면 "anthropic"
)

document = "Hanbit Robotics was founded in 2011 in Daejeon ..."   # 검색한 근거 문서 (영어)
stream = client.chat.completions.create(
    model="gpt-4o-mini",                                            # Claude면 예: "claude-sonnet-5-5"
    stream=True,
    messages=[{"role": "user", "content": f"&lt;context&gt;{document}&lt;/context&gt;\n\nHow did the company do in 2023?"}],
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:            # 본문: 평소와 똑같이 옵니다
        print(chunk.choices[0].delta.content, end="")
    v = getattr(chunk, "verification", None) or (chunk.model_extra or {}).get("verification")
    if v and v.get("kind") == "sentence" and v.get("hallucinated"): # 판정: choices가 빈 별도 청크로 옵니다
        print(f"\n[환각 의심] {v['text']}")</code></pre>
  <p>근거는 메시지 안 <code>&lt;context&gt;…&lt;/context&gt;</code> 태그로 넣거나, 메시지에 넣지 않고 <code>extra_body={"verification": {"context": document}}</code>로 따로 줄 수 있습니다. 근거가 없으면 검증하지 않고 그대로 전달합니다.</p>

  <h3>curl</h3>
<pre><button class="copy">복사</button><code>curl <span class="origin"></span>/v1/chat/completions \
  -H "x-hallugate-key: hg_..." -H "x-upstream: openai" \
  -H "Authorization: Bearer $OPENAI_API_KEY" -H "content-type: application/json" \
  -d '{"model": "gpt-4o-mini", "stream": true,
       "messages": [{"role": "user", "content": "How did the company do in 2023?"}],
       "verification": {"context": "Hanbit Robotics was founded in 2011 ..."}}'</code></pre>

  <h2>3. 받는 것</h2>
  <ul>
    <li><b>본문 청크</b>: OpenAI 형식 그대로</li>
    <li><b>판정 청크</b>: <code>choices: []</code>이고 <code>verification</code> 필드가 있는 청크. 문장마다 <code>{"kind": "sentence", "text", "score", "hallucinated", "start", "upto"}</code>(문장이 끝나기 전 앞부분 경보는 <code>"kind": "prefix"</code>). 기존 OpenAI 클라이언트는 이 청크를 무시하므로 코드가 깨지지 않습니다</li>
    <li><b>검증 중단</b>: <code>{"kind": "unverified", "verified": false, "reason": "non_english" | "judge_timeout" | "judge_error"}</code> 한 번 — 이후 본문은 판정 없이 계속 전달됩니다</li>
  </ul>

  <h2>4. 옵션 (<code>verification</code>)</h2>
  <div class="scroll"><table>
    <thead><tr><th>필드</th><th>값</th><th>뜻</th></tr></thead>
    <tbody>
      <tr><td><code>context</code></td><td>문자열 또는 목록</td><td>근거 문서. 없으면 메시지의 <code>&lt;context&gt;</code> 태그를 씁니다</td></tr>
      <tr><td><code>mode</code></td><td><code>hold</code> | <code>pass</code> | <code>off</code></td><td><code>hold</code>: 문장을 판정한 뒤 내보냄(기본) · <code>pass</code>: 바로 내보내고 판정은 뒤따라옴 · <code>off</code>: 검증 안 함</td></tr>
      <tr><td><code>action</code></td><td><code>flag</code> | <code>cut</code></td><td><code>flag</code>: 표시만(기본) · <code>cut</code>: 환각 문장에서 응답을 끊음</td></tr>
      <tr><td><code>trim</code></td><td><code>true</code> 또는 <code>{"top_k": n}</code></td><td><b>LLM 비용 절감</b>: 긴 근거에서 질문과 관련된 조각 8개만 LLM에 보냅니다(실험에서 입력 토큰 약 48% 절감, 답 품질 유지). 이때 근거는 <code>context</code> 필드로 주세요</td></tr>
      <tr><td><code>collection</code></td><td>컬렉션 이름</td><td>미리 올린 문서에서 근거를 찾아 넣습니다(아래)</td></tr>
      <tr><td><code>post_check</code></td><td><code>true</code></td><td>실험 기능: 답이 끝난 뒤 Claude로 한 번 더 검증(하루 한도 있음)</td></tr>
    </tbody>
  </table></div>

  <h2>5. 문서 올려 두고 쓰기 (선택)</h2>
<pre><button class="copy">복사</button><code>curl <span class="origin"></span>/v1/collections/kb/documents -H "x-hallugate-key: hg_..." \
  -H "content-type: application/json" -d '{"doc_id": "about", "title": "About us", "text": "..."}'</code></pre>
  <p>그다음 채팅 요청에 <code>"verification": {"collection": "kb"}</code>를 주면 질문으로 관련 조각을 찾아 LLM에 넣고 같은 조각으로 검증합니다. 컬렉션은 내 키에만 보입니다.</p>

  <h2>6. 한도 · 사용량 · 개인정보</h2>
  <ul>
    <li>기본 한도: 분당 30요청, 동시 4개, 근거 5만 자, 하루 2단 검증 50회. 넘으면 429(<code>retry-after</code>) 또는 413</li>
    <li>내 토큰·추정 비용·검증 결과는 <a href="/console">내 사용 현황</a>에서 봅니다(API: <code>GET /v1/usage?window=30d</code>). 월 예산을 받았으면 응답 헤더 <code>x-budget-used</code>로 사용률이 옵니다</li>
    <li>질문·근거·응답 원문과 LLM 키는 기록하지 않습니다. 시간·토큰 수·판정 수 같은 숫자만 30일 보관합니다. 데이터 삭제는 운영자에게 요청하세요</li>
  </ul>
</article></main>
<script>
document.querySelectorAll('.origin').forEach(s => s.textContent = location.origin);
document.querySelectorAll('pre .copy').forEach(b => b.onclick = async () => {
  try { await navigator.clipboard.writeText(b.parentElement.querySelector('code').innerText); b.textContent = '복사됨'; } catch (e) {}
});
</script>
</body>
</html>

In [ ]:
# 코드 점검 — 모델 없이 도는 테스트 16개
!cd gateway && python -m pytest -q 2>&1 | tail -3

## 3. 재생 데이터 — RAGTruth dev에서 과제별 20개 (S1~S2-b와 같은 dev 표본)

In [ ]:
!cd gateway && python scripts/make_replay_set.py --ragtruth ../RAGTruth/dataset --per-task 20 --out replay_dev.jsonl

## 4. 가짜 LLM과 게이트웨이 실행 (GPU 판정기)

In [ ]:
import subprocess, time, httpx, os

def start(cmd, env_extra, log):
    env = {**os.environ, **env_extra}
    return subprocess.Popen(cmd, cwd='gateway', env=env, stdout=open(log, 'w'), stderr=subprocess.STDOUT)

def wait(url, proc, log, sec=180):
    for _ in range(sec):
        try:
            return httpx.get(url, timeout=2).json()
        except Exception:
            if proc.poll() is not None:
                break
            time.sleep(1)
    print(open(log).read()[-3000:]); raise RuntimeError(f'{url} 시작 실패')

mock = start(['python', 'mock_upstream.py', '--port', '9000'], {}, 'mock.log')
gw = start(['python', 'serve.py', '--port', '8000'],
           {'JUDGE_MODEL_DIR': '../student_vark', 'UPSTREAM_BASE_URL': 'http://127.0.0.1:9000/v1',
            'METRICS_PATH': 'metrics_gpu.jsonl', 'JUDGE_K': '8', 'JUDGE_MAX_LEN': '512'}, 'gw.log')
time.sleep(2)
print(wait('http://127.0.0.1:8000/health', gw, 'gw.log'))

## 5. 종단 간 측정 (60개 응답 × 3모드, 약 15분)

In [ ]:
!cd gateway && python scripts/bench.py --replay replay_dev.jsonl --gateway http://127.0.0.1:8000 \
    --metrics metrics_gpu.jsonl --modes off pass hold --out bench_gpu

## 6. CPU 판정기 속도 (GPU가 없는 서버에 배포할 경우)

GPU에서는 k와 관계없이 약 15ms였다. CPU에서도 그런지, 그리고 k가 다시 지연을 좌우하는지 확인한다.

In [ ]:
cpu_runs = {}
for k in [2, 8]:
    port = 8100 + k
    p = start(['python', 'serve.py', '--port', str(port)],
              {'JUDGE_MODEL_DIR': '../student_vark', 'UPSTREAM_BASE_URL': 'http://127.0.0.1:9000/v1',
               'METRICS_PATH': f'metrics_cpu_k{k}.jsonl', 'JUDGE_DEVICE': 'cpu', 'JUDGE_K': str(k),
               'CUDA_VISIBLE_DEVICES': ''}, f'gw_cpu{k}.log')
    wait(f'http://127.0.0.1:{port}/health', p, f'gw_cpu{k}.log')
    !cd gateway && python scripts/bench.py --replay replay_dev.jsonl --gateway http://127.0.0.1:{port} \
        --metrics metrics_cpu_k{k}.jsonl --modes pass --n 15 --out bench_cpu_k{k} | grep -E "judge_ms|AUROC|verdict_lag"
    p.terminate()

## 7. 사전 기준 판정

In [ ]:
import json, pandas as pd
S = json.load(open('gateway/bench_gpu/bench_summary.json'))
print(pd.DataFrame(S).round(3).to_string())

ok_text = all(S[m]['text_ok_rate'] == 1.0 for m in S)
auc = S['pass']['AUROC']
j95 = max(S['pass']['judge_ms_p95'], S['hold']['judge_ms_p95'])
lag95 = S['pass']['verdict_lag_ms_p95']
print(f"\n본문 보존 100%          : {'충족' if ok_text else '미달'}")
print(f"게이트웨이 내 AUROC     : {auc:.3f} → {'충족' if auc >= 0.80 else '미달'} (기준 0.80)")
print(f"판정 p95                : {j95:.1f} ms → {'충족' if j95 <= 30 else '미달'} (기준 30ms)")
print(f"pass 판정 전달 p95      : {lag95:.1f} ms → {'충족' if lag95 <= 50 else '미달'} (기준 50ms)")
print('\n=== 정책 트레이드오프 기준선 (S4 출발점) ===')
print(f"hold: 첫 글자 +{S['hold']['ttft_added_ms_p50']:.0f} ms, 글자당 평균 +{S['hold']['char_delay_ms_mean']:.0f} ms "
      f"(p95 +{S['hold']['char_delay_ms_p95']:.0f} ms), 누출 0")
print(f"pass: 첫 글자 +{S['pass']['ttft_added_ms_p50']:.0f} ms, 글자당 평균 +{S['pass']['char_delay_ms_mean']:.0f} ms, "
      f"잡아낸 환각 문장의 누출 p50 {S['pass']['leak_chars_p50_on_caught']} 글자")
print(f"임계값 0.0에서 재현율 {S['pass']['recall_at_threshold']:.2f}, 정밀도 {S['pass']['precision_at_threshold']:.2f} (S4에서 보정)")

In [ ]:
# 결과 파일 모으기
!cd gateway && zip -qr ../s3_results.zip bench_gpu bench_cpu_k2 bench_cpu_k8 metrics_gpu.jsonl
from google.colab import files
files.download('s3_results.zip')

## (선택) 실제 API로 데모

API 키가 있으면 가짜 LLM 대신 실제 모델 응답을 검증해 볼 수 있다.

In [ ]:
# gw.terminate()
# real = start(['python', 'serve.py', '--port', '8001'],
#              {'JUDGE_MODEL_DIR': '../student_vark', 'UPSTREAM_BASE_URL': 'https://api.openai.com/v1',
#               'UPSTREAM_API_KEY': 'sk-...', 'METRICS_PATH': 'metrics_real.jsonl'}, 'gw_real.log')
# wait('http://127.0.0.1:8001/health', real, 'gw_real.log')
# from openai import OpenAI
# ctx = json.loads(open('gateway/replay_dev.jsonl').readline())['reference']
# client = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='unused')
# stream = client.chat.completions.create(model='gpt-4o-mini', stream=True,
#     messages=[{'role': 'user', 'content': 'Summarize the context in 3 sentences.'}],
#     extra_body={'verification': {'context': ctx, 'mode': 'hold'}})
# for ch in stream:
#     if ch.choices and ch.choices[0].delta.content: print(ch.choices[0].delta.content, end='')
#     if getattr(ch, 'verification', None): print('\n  [판정]', ch.verification)